In [ ]:
# ============================================================
#  Section 1 — Cell 1: Install Required Libraries
#  Run this first. Colab restart may be needed after install.
# ============================================================

# Core geospatial & image processing
!pip install rasterio -q
!pip install segmentation-models -q
!pip install albumentations -q
!pip install earthengine-api -q

# Verify installations
import rasterio
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import os, glob, warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow  : {tf.__version__}")
print(f"Rasterio    : {rasterio.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ============================================================
#  Section 1 — Cell 2: Download Sen1Floods11 Dataset
#  Downloads S1 SAR images + hand-labeled flood masks
#  Source: gs://sen1floods11/v1.1
# ============================================================

import os

DATA_ROOT  = "/content/sen1floods11"
S1_DIR     = f"{DATA_ROOT}/S1Hand"
LABEL_DIR  = f"{DATA_ROOT}/LabelHand"

os.makedirs(S1_DIR,    exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)

print("Downloading Sentinel-1 SAR images...")
!gsutil -m cp -n -r \
  gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand \
  {DATA_ROOT}

print("\nDownloading hand-labeled flood masks...")
!gsutil -m cp -n -r \
  gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand \
  {DATA_ROOT}

print("\nDownloading train/val/test split CSVs...")
!gsutil -m cp -n -r \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_train_data.csv \
  {DATA_ROOT}
!gsutil -m cp -n -r \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_valid_data.csv \
  {DATA_ROOT}
!gsutil -m cp -n -r \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_test_data.csv \
  {DATA_ROOT}

print("\nDownload complete!")

In [ ]:
# ============================================================
#  Section 1 — Cell 3: Verify Dataset Structure
#  Confirms all files downloaded correctly and are paired
# ============================================================

import glob, os
import pandas as pd

DATA_ROOT  = "/content/sen1floods11"
S1_DIR     = f"{DATA_ROOT}/S1Hand"
LABEL_DIR  = f"{DATA_ROOT}/LabelHand"

# Count files
s1_files    = sorted(glob.glob(f"{S1_DIR}/*.tif"))
label_files = sorted(glob.glob(f"{LABEL_DIR}/*.tif"))

print("=" * 50)
print(f"  S1 SAR images  : {len(s1_files)}")
print(f"  Flood masks    : {len(label_files)}")
print("=" * 50)

# Check pairing (every S1 image must have a matching mask)
s1_names    = set([os.path.basename(f).replace('_S1Hand','') for f in s1_files])
label_names = set([os.path.basename(f).replace('_LabelHand','') for f in label_files])
paired      = s1_names & label_names
missing     = s1_names - label_names

print(f"  Paired files   : {len(paired)}")
print(f"  Missing masks  : {len(missing)}")

# Show sample file names
print("\nSample S1 files:")
for f in s1_files[:3]:
    print(f"  {os.path.basename(f)}")

print("\nSample mask files:")
for f in label_files[:3]:
    print(f"  {os.path.basename(f)}")

# Load and verify official train/val/test splits
train_df = pd.read_csv(f"{DATA_ROOT}/flood_train_data.csv", header=None)
val_df   = pd.read_csv(f"{DATA_ROOT}/flood_valid_data.csv", header=None)
test_df  = pd.read_csv(f"{DATA_ROOT}/flood_test_data.csv",  header=None)

print(f"\nOfficial splits:")
print(f"  Train : {len(train_df)} images")
print(f"  Val   : {len(val_df)} images")
print(f"  Test  : {len(test_df)} images")

In [ ]:
# ============================================================
#  Section 1 — Cell 4: Global Configuration
#  All hyperparameters in one place — easy to tune later
# ============================================================

import os

# ── Paths ────────────────────────────────────────────────
DATA_ROOT   = "/content/sen1floods11"
S1_DIR      = f"{DATA_ROOT}/S1Hand"
LABEL_DIR   = f"{DATA_ROOT}/LabelHand"
MODEL_DIR   = "/content/models"
OUTPUT_DIR  = "/content/outputs"

os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Image settings ───────────────────────────────────────
IMG_SIZE    = 512          # Input patch size
N_CHANNELS  = 2            # VV + VH
N_CLASSES   = 1            # Binary: flood / non-flood

# ── Training settings ────────────────────────────────────
BATCH_SIZE  = 8
EPOCHS      = 50
LR          = 1e-4
THRESHOLD   = 0.5          # Flood probability threshold

# ── Pixel resolution (Sentinel-1 GRD = 10m) ─────────────
PIXEL_RES_M = 10           # metres per pixel
PIXEL_AREA  = (PIXEL_RES_M ** 2) / 1e6  # km² per pixel

# ── Random seed (reproducibility) ───────────────────────
SEED        = 42
import numpy as np
import tensorflow as tf
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Global config set:")
print(f"  Image size  : {IMG_SIZE}×{IMG_SIZE}×{N_CHANNELS}")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  Epochs      : {EPOCHS}")
print(f"  Pixel area  : {PIXEL_AREA} km²")
print(f"  Model dir   : {MODEL_DIR}")
print(f"  Output dir  : {OUTPUT_DIR}")

section 2


In [ ]:
# ============================================================
#  Fix Cell 1: Diagnose NaN / -inf in SAR bands
#  Run this to understand what's in the raw data
# ============================================================

import rasterio
import numpy as np
import glob

S1_DIR    = "/content/sen1floods11/S1Hand"
LABEL_DIR = "/content/sen1floods11/LabelHand"

s1_files    = sorted(glob.glob(f"{S1_DIR}/*.tif"))
label_files = sorted(glob.glob(f"{LABEL_DIR}/*.tif"))

# Read raw values without any processing
with rasterio.open(s1_files[0]) as src:
    vv_raw = src.read(1)
    vh_raw = src.read(2)
    nodata = src.nodata
    dtype  = src.dtypes[0]
    print(f"Band dtype     : {dtype}")
    print(f"Nodata value   : {nodata}")
    print(f"VV raw min/max : {np.nanmin(vv_raw):.4f} / {np.nanmax(vv_raw):.4f}")
    print(f"VH raw min/max : {np.nanmin(vh_raw):.4f} / {np.nanmax(vh_raw):.4f}")

# Check what types of bad values exist
print(f"\nVV bad value breakdown:")
print(f"  NaN count  : {np.sum(np.isnan(vv_raw)):,}")
print(f"  -inf count : {np.sum(np.isneginf(vv_raw)):,}")
print(f"  +inf count : {np.sum(np.isposinf(vv_raw)):,}")
print(f"  Zero count : {np.sum(vv_raw == 0):,}")
print(f"  Valid count: {np.sum(np.isfinite(vv_raw)):,}")

# Check correlation between nodata mask and SAR NaNs
with rasterio.open(label_files[0]) as src:
    mask_raw = src.read(1)

sar_invalid  = ~np.isfinite(vv_raw)
mask_nodata  = (mask_raw == -1)
both_invalid = sar_invalid & mask_nodata

print(f"\nNodata alignment:")
print(f"  SAR invalid pixels   : {sar_invalid.sum():,}")
print(f"  Mask nodata pixels   : {mask_nodata.sum():,}")
print(f"  Both invalid (match) : {both_invalid.sum():,}")
print(f"\n  → Strategy: replace NaN/inf with 0, exclude mask==-1 from loss")

In [ ]:
# ============================================================
#  Fix Cell 2: Robust SAR Loader — handles NaN, -inf, nodata
#  This replaces the simple loader used in Cell 1 earlier.
#  Use load_s1_image() everywhere from now on.
# ============================================================

import rasterio
import numpy as np

def load_s1_image(s1_path):
    """
    Load Sentinel-1 SAR image.
    Returns:
        vv, vh  : float32 arrays, NaN/inf replaced with 0
        valid   : boolean mask — True where data is valid
    """
    with rasterio.open(s1_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)

    # Mark valid pixels BEFORE replacing bad values
    valid = np.isfinite(vv) & np.isfinite(vh)

    # Replace NaN / ±inf with 0 (SAR nodata fill)
    vv = np.where(np.isfinite(vv), vv, 0.0)
    vh = np.where(np.isfinite(vh), vh, 0.0)

    return vv, vh, valid


def load_label(label_path):
    """
    Load flood mask.
    Returns:
        mask  : int8 array  (-1=nodata, 0=non-flood, 1=flood)
        valid : boolean mask — True where label is known
    """
    with rasterio.open(label_path) as src:
        mask = src.read(1).astype(np.int8)
    valid = (mask >= 0)
    return mask, valid


def normalize_bands(vv, vh, valid=None):
    """
    Normalize VV and VH to [0, 1] using
    fixed SAR dB range: [-30, 0] for VV, [-35, -5] for VH.
    These ranges cover 99%+ of valid Sentinel-1 GRD values.
    """
    VV_MIN, VV_MAX = -30.0, 0.0
    VH_MIN, VH_MAX = -35.0, -5.0

    vv_norm = np.clip((vv - VV_MIN) / (VV_MAX - VV_MIN), 0.0, 1.0)
    vh_norm = np.clip((vh - VH_MIN) / (VH_MAX - VH_MIN), 0.0, 1.0)

    # Zero out nodata regions after normalization
    if valid is not None:
        vv_norm = np.where(valid, vv_norm, 0.0)
        vh_norm = np.where(valid, vh_norm, 0.0)

    return vv_norm, vh_norm


# ── Test the fixed loader ─────────────────────────────────
vv, vh, valid = load_s1_image(s1_files[0])
mask, mlabel  = load_label(label_files[0])
vv_n, vh_n    = normalize_bands(vv, vh, valid)

print("Fixed loader test:")
print(f"  VV raw   — min:{vv[valid].min():.2f}  max:{vv[valid].max():.2f}  mean:{vv[valid].mean():.2f}")
print(f"  VH raw   — min:{vh[valid].min():.2f}  max:{vh[valid].max():.2f}  mean:{vh[valid].mean():.2f}")
print(f"  VV norm  — min:{vv_n[valid].min():.3f} max:{vv_n[valid].max():.3f} mean:{vv_n[valid].mean():.3f}")
print(f"  VH norm  — min:{vh_n[valid].min():.3f} max:{vh_n[valid].max():.3f} mean:{vh_n[valid].mean():.3f}")
print(f"  Valid px : {valid.sum():,} / {valid.size:,} ({100*valid.mean():.1f}%)")
print(f"  NaN remaining: {np.sum(np.isnan(vv_n))} ← should be 0")
print(f"\n  Flood px  : {(mask==1).sum():,}")
print(f"  Nonflood  : {(mask==0).sum():,}")
print(f"  No-data   : {(mask==-1).sum():,}")

In [ ]:
# ============================================================
#  Fix Cell 3: Correct Visualization using fixed loader
#  Now shows real SAR values — no NaN artifacts
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

indices = [0, 80, 160]
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
fig.suptitle('Sen1Floods11 — SAR Bands & Flood Masks (Fixed)',
             fontsize=15, fontweight='bold')

for row, idx in enumerate(indices):
    vv, vh, valid = load_s1_image(s1_files[idx])
    mask, _       = load_label(label_files[idx])
    vv_n, vh_n    = normalize_bands(vv, vh, valid)
    name = os.path.basename(s1_files[idx]).replace('_S1Hand.tif', '')

    # VV normalized
    axes[row,0].imshow(vv_n, cmap='gray', vmin=0, vmax=1)
    axes[row,0].set_title(f'{name}\nVV (normalized)', fontsize=9)

    # VH normalized
    axes[row,1].imshow(vh_n, cmap='gray', vmin=0, vmax=1)
    axes[row,1].set_title('VH (normalized)', fontsize=9)

    # Flood mask
    mask_disp = np.where(mask == -1, np.nan,
                 np.where(mask == 1, 1.0, 0.0))
    axes[row,2].imshow(mask_disp, cmap='RdYlBu_r', vmin=0, vmax=1)
    axes[row,2].set_title('Flood mask', fontsize=9)

    # Overlay
    axes[row,3].imshow(vv_n, cmap='gray', vmin=0, vmax=1)
    overlay = np.zeros((*mask.shape, 4))
    overlay[mask == 1]  = [0.0, 0.4, 1.0, 0.55]
    overlay[mask == -1] = [1.0, 0.3, 0.0, 0.35]
    axes[row,3].imshow(overlay)
    axes[row,3].set_title('VV + flood overlay', fontsize=9)

    for ax in axes[row]:
        ax.axis('off')

legend = [
    mpatches.Patch(color=[0,0.4,1],   label='Flood'),
    mpatches.Patch(color=[1,0.3,0], label='No-data'),
    mpatches.Patch(color='gray',      label='Non-flood')
]
fig.legend(handles=legend, loc='lower center',
          ncol=3, fontsize=11, frameon=False)
plt.tight_layout()
plt.savefig('/content/outputs/fig1_band_visualization.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → /content/outputs/fig1_band_visualization.png")

In [ ]:
# ============================================================
#  Fix Cell 4: Class Distribution — valid pixels only
#  Excludes nodata (mask==-1) from imbalance calculation
#  Saves CLASS_WEIGHT_FLOOD for use in Section 5 training
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from tqdm import tqdm

total_flood, total_nonflood = 0, 0
flood_pct_per_image = []

print("Scanning all masks (valid pixels only)...")
for lf in tqdm(label_files):
    mask, valid = load_label(lf)
    f  = int(np.sum(mask == 1))
    nf = int(np.sum(mask == 0))
    total_flood    += f
    total_nonflood += nf
    valid_px = f + nf
    if valid_px > 0:
        flood_pct_per_image.append(100 * f / valid_px)

total_valid = total_flood + total_nonflood
ratio = total_nonflood / max(total_flood, 1)

# Compute class weight
CLASS_WEIGHT_FLOOD = round(ratio, 2)

print(f"\nValid pixel class distribution:")
print(f"  Flood pixels     : {total_flood:,}  ({100*total_flood/total_valid:.1f}%)")
print(f"  Non-flood pixels : {total_nonflood:,}  ({100*total_nonflood/total_valid:.1f}%)")
print(f"  Imbalance ratio  : 1 : {ratio:.1f}")
print(f"\n  CLASS_WEIGHT_FLOOD = {CLASS_WEIGHT_FLOOD}")
print(f"  → Model will penalize missing flood pixels {CLASS_WEIGHT_FLOOD}x harder")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Distribution — Valid Pixels Only',
             fontsize=13, fontweight='bold')

ax1.pie([total_flood, total_nonflood],
       labels=[f'Flood\n({100*total_flood/total_valid:.1f}%)',
               f'Non-flood\n({100*total_nonflood/total_valid:.1f}%)'],
       colors=['#378ADD', '#B4B2A9'],
       startangle=90, textprops={'fontsize':12})
ax1.set_title('Valid pixel distribution')

ax2.hist(flood_pct_per_image, bins=30,
        color='#378ADD', edgecolor='white', linewidth=0.5)
ax2.axvline(np.mean(flood_pct_per_image), color='#E24B4A',
            linestyle='--',
            label=f'Mean: {np.mean(flood_pct_per_image):.1f}%')
ax2.set_xlabel('Flood pixels per image (%)')
ax2.set_ylabel('Number of images')
ax2.set_title('Flood coverage per image')
ax2.legend(fontsize=11)
plt.tight_layout()
plt.savefig('/content/outputs/fig2_class_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSaved CLASS_WEIGHT_FLOOD = {CLASS_WEIGHT_FLOOD}")
print("Saved → /content/outputs/fig2_class_distribution.png")

In [ ]:
# ============================================================
#  Verify Cell 1: Find images with flood pixels
#  Some Sen1Floods11 images have zero flood — that is normal.
#  We find good examples for visualization and testing.
# ============================================================

import rasterio
import numpy as np
from tqdm import tqdm

flood_counts = []

print("Scanning training set for flood pixels...")
for i, lf in enumerate(tqdm(train_lbl)):
    with rasterio.open(lf) as src:
        mask = src.read(1)
    flood_counts.append((i, int(np.sum(mask == 1))))

# Sort by flood pixel count descending
flood_counts.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop 10 flood-rich training images:")
for rank, (idx, count) in enumerate(flood_counts[:10]):
    pct = 100 * count / (512*512)
    name = os.path.basename(train_s1[idx]).replace('_S1Hand.tif','')
    print(f"  [{rank+1}] idx={idx:3d}  flood={count:6,}px ({pct:.1f}%)  {name}")

# Images with zero flood
zero_flood = [i for i, c in flood_counts if c == 0]
print(f"\nImages with 0 flood pixels: {len(zero_flood)} / {len(train_lbl)}")
print(f"  → This is expected — not all scenes are flooded")

# Store best indices for later use
FLOOD_RICH_INDICES = [idx for idx, count in flood_counts if count > 5000]
print(f"\nImages with >5000 flood px : {len(FLOOD_RICH_INDICES)}")
print(f"Best test index for visuals: {flood_counts[0][0]}")

In [ ]:
# ============================================================
#  Section 3 — Cell 2: Single Image Preprocessor
#  Loads → cleans NaN → normalizes → stacks → returns tensor
# ============================================================

# ============================================================
#  Verify Cell 2: Augmentation on a flood-rich image
#  Uses best index from Verify Cell 1
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Use the most flood-rich image
best_idx = flood_counts[0][0]
img, lbl, wt = preprocess_image(train_s1[best_idx], train_lbl[best_idx])

print(f"Using index {best_idx}: {os.path.basename(train_s1[best_idx])}")
print(f"  Flood pixels : {(lbl==1).sum():,}")
print(f"  Non-flood px : {(lbl==0).sum():,}")
print(f"  Weight unique: {np.unique(wt)}")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Augmentation Verification — Flood-Rich Image',
             fontsize=13, fontweight='bold')

for col, title in enumerate(['Original', 'Aug #1', 'Aug #2', 'Aug #3']):
    if col == 0:
        i_show, l_show = img, lbl
    else:
        i_show, l_show, _ = augment_sample(img, lbl, wt)

    # Row 0: VV band
    axes[0, col].imshow(i_show[..., 0], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(
        f'{title}\nVV  flood_px={int((l_show==1).sum())}', fontsize=9)
    axes[0, col].axis('off')

    # Row 1: flood mask overlay on VV
    axes[1, col].imshow(i_show[..., 0], cmap='gray', vmin=0, vmax=1)
    overlay = np.zeros((*l_show.squeeze().shape, 4))
    overlay[l_show.squeeze() == 1] = [0, 0.5, 1, 0.6]
    axes[1, col].imshow(overlay)
    axes[1, col].set_title('Flood overlay', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('/content/outputs/fig4_augmentation_check.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → /content/outputs/fig4_augmentation_check.png")

In [ ]:
# ============================================================
#  Section 3 — Cell 3: Augmentation Pipeline
#  Geometric augmentations only (no color jitter on SAR!)
#  Applied only to training set — val/test untouched
# ============================================================



# ============================================================
#  Verify Cell 3: Confirm CLASS_WEIGHT_FLOOD flows through
#  the tf.data pipeline correctly in actual training batches
# ============================================================

import numpy as np

CLASS_WEIGHT_FLOOD = 8.43
found_flood_batch  = False

print("Scanning batches for flood pixels...")
for batch_idx, (imgs, lbls, wts) in enumerate(train_ds.take(15)):
    lbls_np = lbls.numpy()
    wts_np  = wts.numpy()

    flood_px   = int((lbls_np == 1).sum())
    weight_vals = set(np.unique(wts_np).round(2).tolist())

    print(f"  Batch {batch_idx+1:2d}: flood_px={flood_px:6,}  weight_vals={weight_vals}")

    if CLASS_WEIGHT_FLOOD in weight_vals or round(CLASS_WEIGHT_FLOOD,1) in \
       {round(v,1) for v in weight_vals}:
        found_flood_batch = True

print(f"\nFlood weight ({CLASS_WEIGHT_FLOOD}) found in batches: {found_flood_batch}")

if found_flood_batch:
    print("  PIPELINE OK — flood class weights flowing correctly")
    print("  Ready for Section 4 — U-Net model!")
else:
    print("  WARNING — no flood pixels found in first 15 batches")
    print("  Check: does train set have enough flood-rich images?")
    print(f"  Flood-rich images in train: {len(FLOOD_RICH_INDICES)}")

In [ ]:
# ============================================================
#  Section 3 — Cell 4: tf.data Pipeline
#  Fast prefetched pipeline for GPU training in Colab
# ============================================================

import tensorflow as tf
import numpy as np

BATCH_SIZE = 8
IMG_SIZE   = 512
AUTOTUNE   = tf.data.AUTOTUNE

def load_and_preprocess(s1_path, label_path, augment=False):
    """tf.py_function wrapper — loads + preprocesses one sample."""
    img, lbl, wt = preprocess_image(
        s1_path.numpy().decode(),
        label_path.numpy().decode()
    )
    if augment:
        img, lbl, wt = augment_sample(img, lbl, wt)
    return img.astype(np.float32), lbl.astype(np.float32), wt.astype(np.float32)


def make_dataset(s1_paths, label_paths, augment=False,
                  batch_size=BATCH_SIZE, shuffle=False):
    """Build a tf.data.Dataset from file path lists."""
    ds = tf.data.Dataset.from_tensor_slices(
        (s1_paths, label_paths))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(s1_paths), seed=42)

    def _load_train(s1, lbl):
        img, lab, wt = tf.py_function(
            lambda a, b: load_and_preprocess(a, b, augment=True),
            [s1, lbl],
            [tf.float32, tf.float32, tf.float32]
        )
        img.set_shape([IMG_SIZE, IMG_SIZE, 2])
        lab.set_shape([IMG_SIZE, IMG_SIZE, 1])
        wt.set_shape( [IMG_SIZE, IMG_SIZE, 1])
        return img, lab, wt

    def _load_val(s1, lbl):
        img, lab, wt = tf.py_function(
            lambda a, b: load_and_preprocess(a, b, augment=False),
            [s1, lbl],
            [tf.float32, tf.float32, tf.float32]
        )
        img.set_shape([IMG_SIZE, IMG_SIZE, 2])
        lab.set_shape([IMG_SIZE, IMG_SIZE, 1])
        wt.set_shape( [IMG_SIZE, IMG_SIZE, 1])
        return img, lab, wt

    map_fn = _load_train if augment else _load_val
    ds = ds.map(map_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds


# Build all three datasets
train_ds = make_dataset(train_s1, train_lbl,
                         augment=True, shuffle=True)
val_ds   = make_dataset(val_s1,   val_lbl,
                         augment=False, shuffle=False)
test_ds  = make_dataset(test_s1,  test_lbl,
                         augment=False, shuffle=False)

print("tf.data datasets ready:")
print(f"  train_ds : {len(train_s1)} images → {len(train_s1)//BATCH_SIZE} batches")
print(f"  val_ds   : {len(val_s1)} images → {len(val_s1)//BATCH_SIZE} batches")
print(f"  test_ds  : {len(test_s1)} images → {len(test_s1)//BATCH_SIZE} batches")

# Verify one batch shape
for imgs, lbls, wts in train_ds.take(1):
    print(f"\nSample batch shapes:")
    print(f"  images  : {imgs.shape}")
    print(f"  labels  : {lbls.shape}")
    print(f"  weights : {wts.shape}")
    print(f"  image range : [{imgs.numpy().min():.3f}, {imgs.numpy().max():.3f}]")
    print(f"  weight vals : {set(wts.numpy().flatten().tolist()[:20])}")

In [ ]:
# ============================================================
#  Section 3 — Cell 5: Visual Augmentation Check
#  Shows original vs augmented pairs — sanity check before training
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Augmentation Check — Original vs Augmented',
             fontsize=13, fontweight='bold')

img, lbl, wt = preprocess_image(train_s1[10], train_lbl[10])

for col, title in enumerate(['Original', 'Aug #1', 'Aug #2', 'Aug #3']):
    if col == 0:
        i, l = img, lbl
    else:
        i, l, _ = augment_sample(img, lbl, wt)

    axes[0, col].imshow(i[..., 0], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'{title}\nVV band', fontsize=9)
    axes[0, col].axis('off')

    mask_disp = np.where(l.squeeze() == 1, 1.0, 0.3)
    axes[1, col].imshow(mask_disp, cmap='Blues', vmin=0, vmax=1)
    axes[1, col].set_title(f'Flood mask', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('/content/outputs/fig4_augmentation_check.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → /content/outputs/fig4_augmentation_check.png")
print("\nSection 3 complete! Ready for Section 4 — U-Net model.")

In [ ]:
# ============================================================
#  Section 4 — Cell 1: Custom Loss Functions
#  Dice loss fixes class imbalance better than BCE alone.
#  Combined Dice+BCE gives stable training + good F1 score.
# ============================================================

import tensorflow as tf
import tensorflow.keras.backend as K

def dice_loss(y_true, y_pred, smooth=1e-6):
    """
    Dice loss — directly optimizes F1/Dice score.
    Works well with class imbalance.
    """
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    dice = (2.0 * intersection + smooth) / \
           (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)
    return 1.0 - dice


def weighted_dice_bce_loss(y_true, y_pred):
    """
    Combined weighted Dice + BCE loss.
    - BCE handles per-pixel classification
    - Dice handles overall overlap (fixes imbalance)
    - sample_weight (8.43x for flood) already applied via
      model.fit(..., sample_weight=wt) in Section 5
    """
    smooth   = 1e-6
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.reshape(y_pred, [-1])

    # BCE component
    bce = tf.keras.losses.binary_crossentropy(
        y_true_f, y_pred_f)

    # Dice component
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    dice_coef    = (2.0 * intersection + smooth) / \
                   (tf.reduce_sum(y_true_f) +
                    tf.reduce_sum(y_pred_f) + smooth)
    dice = 1.0 - dice_coef

    # 50/50 combination
    return 0.5 * bce + 0.5 * dice


def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """Dice coefficient metric (= F1 score for binary segmentation)."""
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(
        tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / \
           (tf.reduce_sum(y_true_f) +
            tf.reduce_sum(y_pred_f) + smooth)


def iou_score(y_true, y_pred, smooth=1e-6):
    """Intersection over Union metric."""
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(
        tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union        = tf.reduce_sum(y_true_f) + \
                   tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


print("Loss functions defined:")
print("  weighted_dice_bce_loss  ← training loss")
print("  dice_coefficient        ← F1 metric")
print("  iou_score               ← IoU metric")

# Quick sanity check
dummy_true = tf.constant([[[1.0],[0.0]],[[1.0],[1.0]]])
dummy_pred = tf.constant([[[0.9],[0.1]],[[0.8],[0.7]]])
print(f"\nSanity check (near-perfect pred):")
print(f"  dice_loss      : {dice_loss(dummy_true, dummy_pred):.4f}  ← close to 0")
print(f"  dice_coeff     : {dice_coefficient(dummy_true, dummy_pred):.4f}  ← close to 1")
print(f"  iou_score      : {iou_score(dummy_true, dummy_pred):.4f}  ← close to 1")

In [ ]:
# ============================================================
#  Section 4 — Cell 2: U-Net Building Blocks
#  conv_block    : 2x Conv → BN → ReLU  (encoder/decoder)
#  encoder_block : conv_block + MaxPool  (downsampling)
#  decoder_block : UpSample + skip conn + conv_block
# ============================================================

from tensorflow.keras import layers, Model
import tensorflow as tf


def conv_block(x, filters, dropout_rate=0.0):
    """
    Double Conv → BatchNorm → ReLU block.
    BatchNorm added vs previous version — stabilizes training.
    """
    x = layers.Conv2D(filters, 3, padding='same',
                       kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, 3, padding='same',
                       kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    return x


def encoder_block(x, filters, dropout_rate=0.0):
    """Conv block → skip connection + MaxPool."""
    skip = conv_block(x, filters, dropout_rate)
    pool = layers.MaxPooling2D(2)(skip)
    return skip, pool


def decoder_block(x, skip, filters):
    """
    Upsample → concat skip connection → conv block.
    Uses bilinear upsampling (smoother than transpose conv).
    """
    x = layers.UpSampling2D(2, interpolation='bilinear')(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x


print("U-Net building blocks defined:")
print("  conv_block    : Conv→BN→ReLU × 2  (+ optional Dropout)")
print("  encoder_block : conv_block + MaxPool2D")
print("  decoder_block : UpSample2D + Concat(skip) + conv_block")

In [ ]:
# ============================================================
#  Section 4 — Cell 3: Improved U-Net Model
#  Architecture: 64→128→256→512→1024 bottleneck
#  Key improvements:
#    - BatchNorm after every conv
#    - Dropout in bottleneck (0.3) + deep encoder (0.2)
#    - Bilinear upsampling (smoother than TransposeConv)
#    - he_normal initialization
# ============================================================

from tensorflow.keras import Input, Model
from tensorflow.keras import layers

def build_unet(input_shape=(512, 512, 2)):
    """
    Improved U-Net for Sentinel-1 flood segmentation.
    Input : (512, 512, 2)  — VV, VH bands
    Output: (512, 512, 1)  — flood probability map
    """
    inputs = Input(shape=input_shape, name='sar_input')

    # ── Encoder ──────────────────────────────────────────────
    s1, p1 = encoder_block(inputs, 64)          # 512→256
    s2, p2 = encoder_block(p1, 128)             # 256→128
    s3, p3 = encoder_block(p2, 256, dropout_rate=0.1)  # 128→64
    s4, p4 = encoder_block(p3, 512, dropout_rate=0.2)  # 64→32

    # ── Bottleneck ───────────────────────────────────────────
    b = conv_block(p4, 1024, dropout_rate=0.3)   # 32×32×1024

    # ── Decoder ──────────────────────────────────────────────
    d1 = decoder_block(b,  s4, 512)              # 32→64
    d2 = decoder_block(d1, s3, 256)              # 64→128
    d3 = decoder_block(d2, s2, 128)              # 128→256
    d4 = decoder_block(d3, s1, 64)               # 256→512

    # ── Output ───────────────────────────────────────────────
    outputs = layers.Conv2D(
        1, 1,
        activation='sigmoid',
        name='flood_probability'
    )(d4)                                          # (512,512,1)

    model = Model(inputs, outputs, name='improved_unet')
    return model


# Build model
model = build_unet(input_shape=(512, 512, 2))

# Compile with Dice+BCE loss + all metrics
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss      = weighted_dice_bce_loss,
    metrics   = [
        dice_coefficient,
        iou_score,
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.BinaryAccuracy(name='accuracy')
    ]
)

# Summary
total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy()
                         for w in model.trainable_weights])

print(f"Model: {model.name}")
print(f"  Input  shape : {model.input_shape}")
print(f"  Output shape : {model.output_shape}")
print(f"  Total params : {total_params:,}")
print(f"\nArchitecture summary:")
print("  Encoder : 64 → 128 → 256(d0.1) → 512(d0.2)")
print("  Bottleneck: 1024 (dropout=0.3)")
print("  Decoder : 512 → 256 → 128 → 64")
print("  Output  : sigmoid → flood probability")
print(f"\nLoss     : Dice + BCE (combined)")
print(f"Metrics  : dice_coeff, iou, precision, recall, accuracy")

In [ ]:
# ============================================================
#  Section 4 — Cell 4: Verify Forward Pass
#  Runs one real batch through the model before training
#  Confirms shapes, output range, and GPU is being used
# ============================================================

import numpy as np
import tensorflow as tf
import time

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU available: {len(gpus) > 0}  — {gpus}")

# Get one real batch
for imgs, lbls, wts in train_ds.take(1):
    batch_imgs = imgs
    batch_lbls = lbls
    batch_wts  = wts

# Forward pass timing
t0   = time.time()
preds = model.predict(batch_imgs, verbose=0)
t1   = time.time()

print(f"\nForward pass test:")
print(f"  Input  shape : {batch_imgs.shape}")
print(f"  Output shape : {preds.shape}")
print(f"  Pred range   : [{preds.min():.4f}, {preds.max():.4f}]")
print(f"  Pred mean    : {preds.mean():.4f}")
print(f"  Time (batch) : {t1-t0:.2f}s")

# Output must be in [0,1] — sigmoid activation
assert preds.min() >= 0.0, "Pred below 0!"
assert preds.max() <= 1.0, "Pred above 1!"
assert preds.shape == (len(batch_imgs), 512, 512, 1), "Wrong output shape!"

print(f"\nAll assertions passed!")
print(f"Model is ready for training in Section 5.")

# Visualize untrained predictions (should look random/uniform)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Untrained Model — Forward Pass (random predictions expected)',
             fontsize=11)
axes[0].imshow(batch_imgs[0,...,0], cmap='gray')
axes[0].set_title('Input VV band')
axes[1].imshow(batch_lbls[0,...,0], cmap='Blues')
axes[1].set_title('Ground truth mask')
axes[2].imshow(preds[0,...,0], cmap='hot', vmin=0, vmax=1)
axes[2].set_title('Untrained prediction\n(should look noisy)')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('/content/outputs/fig5_untrained_pred.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Section 4 complete! Ready for Section 5 — Training.")

In [ ]:
# ============================================================
#  Section 5 — Cell 1: Training Callbacks
#  EarlyStopping  : stops if val_dice not improving
#  ReduceLROnPlateau : halves LR when stuck
#  ModelCheckpoint   : saves best model by val_dice
#  CSVLogger         : logs every epoch to CSV
# ============================================================

import tensorflow as tf
import os

MODEL_DIR  = "/content/models"
OUTPUT_DIR = "/content/outputs"
os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

BEST_MODEL_PATH = f"{MODEL_DIR}/best_unet_flood.keras"
LOG_PATH        = f"{OUTPUT_DIR}/training_log.csv"

callbacks = [

    # 1. Save best model (monitored by val dice_coefficient)
    tf.keras.callbacks.ModelCheckpoint(
        filepath        = BEST_MODEL_PATH,
        monitor         = 'val_dice_coefficient',
        mode            = 'max',
        save_best_only  = True,
        save_weights_only = False,
        verbose         = 1
    ),

    # 2. Stop training when val_dice plateaus for 8 epochs
    tf.keras.callbacks.EarlyStopping(
        monitor              = 'val_dice_coefficient',
        mode                 = 'max',
        patience             = 8,
        restore_best_weights = True,
        verbose              = 1
    ),

    # 3. Halve LR when val_dice plateaus for 4 epochs
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor   = 'val_dice_coefficient',
        mode      = 'max',
        factor    = 0.5,
        patience  = 4,
        min_lr    = 1e-7,
        verbose   = 1
    ),

    # 4. Log all metrics to CSV every epoch
    tf.keras.callbacks.CSVLogger(
        LOG_PATH,
        append = False
    ),
]

print("Callbacks configured:")
print(f"  ModelCheckpoint → {BEST_MODEL_PATH}")
print(f"  EarlyStopping   → patience=8 epochs on val_dice")
print(f"  ReduceLROnPlateau → factor=0.5, patience=4")
print(f"  CSVLogger       → {LOG_PATH}")

In [ ]:
# ============================================================
#  Section 5 — Cell 2: Train Model
#  sample_weight=wt applies CLASS_WEIGHT_FLOOD=8.43
#  per-pixel during loss calculation
#  Expected: ~3-5 min per epoch on Colab T4 GPU
# ============================================================

import time

EPOCHS = 50

# Unpack dataset into (inputs, targets, sample_weights)
# tf.data returns (img, lbl, wt) — Keras needs this format
def unpack_dataset(ds):
    """Reformat (img, lbl, wt) → ((img), (lbl), (wt)) for Keras."""
    return ds.map(lambda img, lbl, wt: (img, lbl, wt))

train_ds_fit = unpack_dataset(train_ds)
val_ds_fit   = unpack_dataset(val_ds)

print("Starting training...")
print(f"  Train batches : {len(train_s1)//8}")
print(f"  Val batches   : {len(val_s1)//8}")
print(f"  Max epochs    : {EPOCHS}")
print(f"  Early stop    : patience=8 on val_dice")
print(f"  CLASS_WEIGHT  : flood={8.43}x non-flood\n")

t_start = time.time()

history = model.fit(
    train_ds_fit,
    validation_data = val_ds_fit,
    epochs          = EPOCHS,
    callbacks       = callbacks,
    verbose         = 1
)

t_total = time.time() - t_start

print(f"\nTraining complete!")
print(f"  Total time    : {t_total/60:.1f} minutes")
print(f"  Epochs run    : {len(history.history['loss'])}")
print(f"  Best val_dice : {max(history.history['val_dice_coefficient']):.4f}")
print(f"  Best val_iou  : {max(history.history['val_iou_score']):.4f}")
print(f"  Model saved   : {BEST_MODEL_PATH}")

In [ ]:
# ============================================================
#  Section 5 — Cell 3: Training Curves (Research Paper Ready)
#  Generates 2 publication-quality figures:
#  Fig A: Loss curves (train vs val)
#  Fig B: Metric curves (Dice, IoU, Precision, Recall)
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np

# Load training log CSV
log_df = pd.read_csv(LOG_PATH)
epochs = log_df['epoch'] + 1

# ── Figure 1: Loss Curves ────────────────────────────────
fig1, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, log_df['loss'],
        color='#D85A30', lw=2, label='Training loss')
ax.plot(epochs, log_df['val_loss'],
        color='#185FA5', lw=2, linestyle='--', label='Validation loss')

best_epoch = log_df['val_dice_coefficient'].idxmax() + 1
ax.axvline(best_epoch, color='#639922', linestyle=':', lw=1.5,
           label=f'Best model (epoch {best_epoch})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Dice + BCE Loss', fontsize=12)
ax.set_title('Training and Validation Loss', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, len(epochs))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig6_loss_curves.png',
            dpi=300, bbox_inches='tight')
plt.show()

# ── Figure 2: Metric Curves ──────────────────────────────
fig2, axes = plt.subplots(2, 2, figsize=(14, 9))
fig2.suptitle('Training Metrics — Improved U-Net',
              fontsize=14, fontweight='bold')

metrics = [
    ('dice_coefficient', 'val_dice_coefficient', 'Dice Coefficient (F1)'),
    ('iou_score',        'val_iou_score',        'IoU Score'),
    ('precision',        'val_precision',        'Precision'),
    ('recall',           'val_recall',           'Recall'),
]

colors = [('#D85A30', '#185FA5'),
          ('#D85A30', '#185FA5'),
          ('#D85A30', '#185FA5'),
          ('#D85A30', '#185FA5')]

for ax, (tr_key, val_key, title), (c_tr, c_val) in \
        zip(axes.flat, metrics, colors):
    if tr_key in log_df.columns:
        ax.plot(epochs, log_df[tr_key],
                color=c_tr, lw=2, label='Train')
        ax.plot(epochs, log_df[val_key],
                color=c_val, lw=2, linestyle='--', label='Val')
        ax.axvline(best_epoch, color='#639922', linestyle=':', lw=1.5, alpha=0.7)

        best_val = log_df[val_key].max()
        ax.set_title(f'{title}\nBest val: {best_val:.4f}', fontsize=11)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(title)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(1, len(epochs))
        ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig7_metric_curves.png',
            dpi=300, bbox_inches='tight')
plt.show()

# Print final summary table
best_row = log_df.loc[log_df['val_dice_coefficient'].idxmax()]
print("\nBest epoch results:")
print(f"  Epoch          : {int(best_row['epoch'])+1}")
print(f"  val_dice (F1)  : {best_row['val_dice_coefficient']:.4f}")
print(f"  val_iou        : {best_row['val_iou_score']:.4f}")
print(f"  val_precision  : {best_row['val_precision']:.4f}")
print(f"  val_recall     : {best_row['val_recall']:.4f}")
print(f"\nSaved:")
print(f"  fig6_loss_curves.png    ← paper Figure")
print(f"  fig7_metric_curves.png  ← paper Figure")

In [ ]:
# ============================================================
#  Section 5 — Cell 4: Save Training Summary
#  Saves all metrics to JSON — used in Section 6 & 8
#  for evaluation tables and paper figures
# ============================================================

import json, os
import numpy as np

OUTPUT_DIR = "/content/outputs"

# Best epoch metrics
best_idx  = log_df['val_dice_coefficient'].idxmax()
best_row  = log_df.loc[best_idx]

training_summary = {
    "model"          : "Improved U-Net",
    "dataset"        : "Sen1Floods11",
    "input_channels" : "VV + VH",
    "image_size"     : "512x512",
    "class_weight"   : 8.43,
    "loss_function"  : "Dice + BCE",
    "total_epochs"   : len(log_df),
    "best_epoch"     : int(best_idx) + 1,
    "train_metrics"  : {
        "loss"           : round(float(best_row['loss']),           4),
        "dice_coeff"     : round(float(best_row['dice_coefficient']),4),
        "iou"            : round(float(best_row['iou_score']),        4),
        "precision"      : round(float(best_row['precision']),        4),
        "recall"         : round(float(best_row['recall']),           4),
    },
    "val_metrics"    : {
        "loss"           : round(float(best_row['val_loss']),             4),
        "dice_coeff"     : round(float(best_row['val_dice_coefficient']),4),
        "iou"            : round(float(best_row['val_iou_score']),        4),
        "precision"      : round(float(best_row['val_precision']),        4),
        "recall"         : round(float(best_row['val_recall']),           4),
    }
}

summary_path = f"{OUTPUT_DIR}/training_summary.json"
with open(summary_path, 'w') as f:
    json.dump(training_summary, f, indent=2)

print("Training summary saved:")
print(json.dumps(training_summary, indent=2))
print(f"\nSaved → {summary_path}")
print("\nSection 5 complete! Start training now.")
print("Share val_dice and val_recall after training finishes.")

In [ ]:
# ============================================================
#  Section 6 — Cell 1: Full Test Set Evaluation
#  Loads best saved model, runs on all 90 test images
#  Collects pixel-level predictions for all metrics
# ============================================================

import tensorflow as tf
import numpy as np
import os
from tqdm import tqdm

OUTPUT_DIR      = "/content/outputs"
BEST_MODEL_PATH = "/content/models/best_unet_flood.keras"
THRESHOLD       = 0.5
PIXEL_AREA_KM2  = 0.0001   # 10m × 10m = 0.0001 km²

# Load best model
model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects = {
        'weighted_dice_bce_loss' : weighted_dice_bce_loss,
        'dice_coefficient'       : dice_coefficient,
        'iou_score'              : iou_score
    }
)
print(f"Model loaded: {BEST_MODEL_PATH}")

# Run inference on all test images
all_tp, all_fp, all_fn, all_tn = 0, 0, 0, 0
per_image_results = []

print(f"\nEvaluating on {len(test_s1)} test images...")
for i, (s1_path, lbl_path) in enumerate(
        tqdm(zip(test_s1, test_lbl), total=len(test_s1))):

    # Preprocess
    img, lbl, wt = preprocess_image(s1_path, lbl_path)
    mask, valid   = load_label(lbl_path)

    # Predict
    pred_prob = model.predict(img[np.newaxis], verbose=0)[0,...,0]
    pred_mask = (pred_prob > THRESHOLD).astype(np.uint8)

    # Only evaluate on valid (non-nodata) pixels
    valid_mask = (mask >= 0)
    y_true = (mask[valid_mask] == 1).astype(np.uint8)
    y_pred = pred_mask[valid_mask]

    # Accumulate confusion matrix
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))

    all_tp += tp; all_fp += fp
    all_fn += fn; all_tn += tn

    # Per-image metrics
    smooth = 1e-6
    prec  = tp / (tp + fp + smooth)
    rec   = tp / (tp + fn + smooth)
    f1    = 2 * prec * rec / (prec + rec + smooth)
    iou   = tp / (tp + fp + fn + smooth)
    flood_area_km2 = int(np.sum(pred_mask)) * PIXEL_AREA_KM2

    per_image_results.append({
        'name'          : os.path.basename(s1_path).replace('_S1Hand.tif',''),
        'precision'     : round(prec, 4),
        'recall'        : round(rec,  4),
        'f1'            : round(f1,   4),
        'iou'           : round(iou,  4),
        'flood_area_km2': round(flood_area_km2, 4),
        'pred_prob'     : pred_prob,
        'pred_mask'     : pred_mask,
        'true_mask'     : mask,
        'image'         : img,
    })

# Global metrics from accumulated confusion matrix
smooth   = 1e-6
g_prec   = all_tp / (all_tp + all_fp + smooth)
g_rec    = all_tp / (all_tp + all_fn + smooth)
g_f1     = 2 * g_prec * g_rec / (g_prec + g_rec + smooth)
g_iou    = all_tp / (all_tp + all_fp + all_fn + smooth)
g_acc    = (all_tp + all_tn) / (all_tp+all_fp+all_fn+all_tn+smooth)
g_spec   = all_tn / (all_tn + all_fp + smooth)

print(f"\nTest Set Results (90 images):")
print(f"  Precision   : {g_prec:.4f}")
print(f"  Recall      : {g_rec:.4f}")
print(f"  F1 / Dice   : {g_f1:.4f}")
print(f"  IoU         : {g_iou:.4f}")
print(f"  Accuracy    : {g_acc:.4f}")
print(f"  Specificity : {g_spec:.4f}")
print(f"\nConfusion Matrix (global):")
print(f"  TP: {all_tp:,}  FP: {all_fp:,}")
print(f"  FN: {all_fn:,}  TN: {all_tn:,}")

In [ ]:
# ============================================================
#  Section 6 — Cell 2: Confusion Matrix (Paper Figure 1)
#  Publication-quality normalized confusion matrix
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

cm = np.array([[all_tn, all_fp],
               [all_fn, all_tp]])

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Confusion Matrix — Improved U-Net (Test Set)',
             fontsize=14, fontweight='bold')

labels = ['Non-flood', 'Flood']

for ax, data, title, fmt in [
    (axes[0], cm,      'Raw counts',   'd'),
    (axes[1], cm_norm, 'Normalized', '.3f')
]:
    im = ax.imshow(data, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_yticklabels(labels, fontsize=12)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title(title, fontsize=12)
    for i in range(2):
        for j in range(2):
            val = f"{data[i,j]:{fmt}}"
            color = 'white' if data[i,j] > data.max()*0.6 else 'black'
            ax.text(j, i, val, ha='center', va='center',
                   fontsize=14, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/paper_fig1_confusion_matrix.png',
            dpi=300, bbox_inches='tight')
plt.show()
print("Saved → paper_fig1_confusion_matrix.png")

In [ ]:
# ============================================================
#  Section 6 — Cell 3 FIXED: Prediction Grid (Paper Figure 2)
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Pick 5 diverse samples with flood pixels
has_flood = [i for i, r in enumerate(per_image_results)
             if (r['true_mask'] == 1).sum() > 1000]

show_idx = (
    [has_flood[0], has_flood[1]] +
    [has_flood[len(has_flood) // 2]] +
    [has_flood[-2], has_flood[-1]]
)[:5]

fig, axes = plt.subplots(5, 4, figsize=(18, 22))
fig.suptitle(
    'Flood Detection Results — Improved U-Net\n'
    'Input SAR (VV)  |  Ground Truth  |  Prediction  |  Overlay',
    fontsize=14, fontweight='bold', y=1.01
)

col_titles = ['Input SAR (VV)', 'Ground Truth',
              'Prediction', 'Overlay']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=12, fontweight='bold', pad=8)

for row, idx in enumerate(show_idx):
    r    = per_image_results[idx]
    img  = r['image']
    pred = r['pred_mask']
    true = r['true_mask']
    prob = r['pred_prob']

    vv_disp   = img[..., 0]
    true_disp = np.where(true == -1, np.nan,
                np.where(true == 1,  1.0, 0.0))

    # Col 0: VV band
    axes[row, 0].imshow(vv_disp, cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_ylabel(
        f"{r['name'][:20]}\nF1={r['f1']:.3f}", fontsize=8)

    # Col 1: Ground truth
    axes[row, 1].imshow(true_disp, cmap='RdYlBu_r', vmin=0, vmax=1)

    # Col 2: Prediction probability heatmap
    axes[row, 2].imshow(prob, cmap='hot', vmin=0, vmax=1)

    # Col 3: Overlay — TP=blue FP=red FN=yellow
    axes[row, 3].imshow(vv_disp, cmap='gray', vmin=0, vmax=1)
    overlay = np.zeros((*true.shape, 4))
    overlay[(true == 1) & (pred == 1)] = [0,   0.5, 1,   0.6]
    overlay[(true == 0) & (pred == 1)] = [1,   0.2, 0,   0.5]
    overlay[(true == 1) & (pred == 0)] = [1,   0.9, 0,   0.5]
    axes[row, 3].imshow(overlay)

    for ax in axes[row]:
        ax.axis('off')

# ── FIXED legend (removed invalid y= argument) ───────────
legend = [
    mpatches.Patch(color=[0, 0.5, 1], label='True Positive (TP)'),
    mpatches.Patch(color=[1, 0.2, 0], label='False Positive (FP)'),
    mpatches.Patch(color=[1, 0.9, 0], label='False Negative (FN)'),
]
fig.legend(
    handles        = legend,
    loc            = 'lower center',
    bbox_to_anchor = (0.5, -0.02),   # ← fixed: was y=-0.01
    ncol           = 3,
    fontsize       = 12,
    frameon        = True
)

plt.tight_layout()
plt.savefig(
    f'{OUTPUT_DIR}/paper_fig2_prediction_grid.png',
    dpi            = 300,
    bbox_inches    = 'tight'
)
plt.show()
print("Saved → paper_fig2_prediction_grid.png")

In [ ]:
# ============================================================
#  Section 6 — Cell 4: Model Comparison Bar Chart (Paper Fig 3)
#  Compares Previous U-Net vs Improved U-Net on all metrics
#  This is the key "improvement" figure for the paper
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Previous model results (from your original project)
prev = {
    'Precision'  : 0.71,
    'Recall'     : 0.61,
    'F1 / Dice'  : 0.65,
    'Accuracy'   : 0.95,
}

# New model results from test set
new = {
    'Precision'  : round(g_prec, 4),
    'Recall'     : round(g_rec,  4),
    'F1 / Dice'  : round(g_f1,   4),
    'Accuracy'   : round(g_acc,  4),
}

metrics = list(prev.keys())
x       = np.arange(len(metrics))
w       = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - w/2, [prev[m] for m in metrics],
               w, label='Baseline U-Net',
               color='#B4B2A9', edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x + w/2, [new[m]  for m in metrics],
               w, label='Improved U-Net (ours)',
               color='#185FA5', edgecolor='white', linewidth=0.5)

# Value labels on bars
for bar in [*bars1, *bars2]:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f'{h:.3f}', ha='center', va='bottom', fontsize=10)

# Improvement arrows
for i, m in enumerate(metrics):
    diff = new[m] - prev[m]
    color = '#3B6D11' if diff >= 0 else '#A32D2D'
    sign  = '+' if diff >= 0 else ''
    ax.text(i, max(prev[m], new[m]) + 0.04,
            f'{sign}{diff:.3f}',
            ha='center', fontsize=10,
            fontweight='bold', color=color)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.12)
ax.set_title(
    'Baseline vs Improved U-Net — Test Set Comparison',
    fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/paper_fig3_model_comparison.png',
            dpi=300, bbox_inches='tight')
plt.show()
print("Saved → paper_fig3_model_comparison.png")

In [ ]:
# ============================================================
#  Section 6 — Cell 5: Results Summary Table (Paper Table 1)
#  Prints LaTeX-ready table + saves as image
# ============================================================

import matplotlib.pyplot as plt
import json

# Final results
results = {
    "test_precision"   : round(g_prec,  4),
    "test_recall"      : round(g_rec,   4),
    "test_f1_dice"     : round(g_f1,    4),
    "test_iou"         : round(g_iou,   4),
    "test_accuracy"    : round(g_acc,   4),
    "test_specificity" : round(g_spec,  4),
}

# Save JSON
with open(f'{OUTPUT_DIR}/test_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Print table
print("=" * 52)
print(f"  FINAL TEST SET RESULTS — Improved U-Net")
print(f"  Dataset : Sen1Floods11 (90 test images)")
print("=" * 52)
print(f"  Metric        Previous   Improved   Change")
print("-" * 52)
rows = [
    ("Precision",   0.71,  g_prec),
    ("Recall",      0.61,  g_rec),
    ("F1 / Dice",   0.65,  g_f1),
    ("IoU",         0.48,  g_iou),
    ("Accuracy",    0.95,  g_acc),
    ("Specificity", 0.97,  g_spec),
]
for name, prev_v, new_v in rows:
    diff  = new_v - prev_v
    arrow = "▲" if diff > 0 else "▼"
    print(f"  {name:<14} {prev_v:.4f}     {new_v:.4f}   {arrow}{abs(diff):.4f}")
print("=" * 52)

# LaTeX table for paper
print("\nLaTeX table (copy into paper):")
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\caption{Flood Detection Results on Sen1Floods11 Test Set}")
print(r"\begin{tabular}{lcc}")
print(r"\hline")
print(r"Metric & Baseline U-Net & Improved U-Net \\")
print(r"\hline")
for name, prev_v, new_v in rows:
    print(f"{name} & {prev_v:.4f} & {new_v:.4f} \\\\")
print(r"\hline")
print(r"\end{tabular}")
print(r"\end{table}")

print(f"\nSection 6 complete!")
print(f"Paper figures saved to /content/outputs/")
print(f"  paper_fig1_confusion_matrix.png")
print(f"  paper_fig2_prediction_grid.png")
print(f"  paper_fig3_model_comparison.png")
print(f"  fig6_loss_curves.png")
print(f"  fig7_metric_curves.png")

In [ ]:
# ============================================================
#  Section 7 — Cell 1: Flood Prediction Pipeline
#  predict_flood() takes any Sentinel-1 .tif file and returns
#  flood mask + area statistics + visualization
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
import os

THRESHOLD      = 0.5
PIXEL_AREA_KM2 = 0.0001   # 10m × 10m Sentinel-1 GRD
OUTPUT_DIR     = "/content/outputs"


def predict_flood(s1_path, save_name=None, threshold=THRESHOLD):
    """
    Full flood prediction pipeline for any Sentinel-1 image.

    Args:
        s1_path   : path to Sentinel-1 .tif file (VV + VH bands)
        save_name : output filename prefix (optional)
        threshold : flood probability threshold (default 0.5)

    Returns:
        dict with keys:
            pred_prob     : (512,512) float — flood probability map
            pred_mask     : (512,512) uint8 — binary flood mask
            flood_px      : int — number of flood pixels
            total_valid_px: int — total valid pixels
            flood_pct     : float — flood percentage
            flood_area_km2: float — estimated flood area in km²
    """
    # Step 1: Load & preprocess
    vv, vh, valid = load_s1_image(s1_path)
    vv_n, vh_n    = normalize_bands(vv, vh, valid)
    image         = np.stack([vv_n, vh_n], axis=-1)[np.newaxis]  # (1,512,512,2)

    # Step 2: Model inference
    pred_prob = model.predict(image, verbose=0)[0, ..., 0]       # (512,512)

    # Step 3: Apply threshold → binary mask
    pred_mask = (pred_prob > threshold).astype(np.uint8)

    # Step 4: Area calculation (valid pixels only)
    valid_px       = int(valid.sum())
    flood_px       = int((pred_mask * valid).sum())
    total_px       = 512 * 512
    flood_pct      = 100 * flood_px / valid_px if valid_px > 0 else 0
    flood_area_km2 = flood_px * PIXEL_AREA_KM2

    results = {
        'name'          : os.path.basename(s1_path),
        'pred_prob'     : pred_prob,
        'pred_mask'     : pred_mask,
        'image_vv'      : vv_n,
        'image_vh'      : vh_n,
        'valid'         : valid,
        'total_px'      : total_px,
        'valid_px'      : valid_px,
        'flood_px'      : flood_px,
        'flood_pct'     : round(flood_pct,    2),
        'flood_area_km2': round(flood_area_km2, 4),
    }

    # Step 5: Print report
    print(f"\nFlood Prediction Report")
    print(f"{'='*45}")
    print(f"  File            : {os.path.basename(s1_path)}")
    print(f"  Total pixels    : {total_px:,}")
    print(f"  Valid pixels    : {valid_px:,}")
    print(f"  Flood pixels    : {flood_px:,}")
    print(f"  Flood coverage  : {flood_pct:.2f}%")
    print(f"  Flood area      : {flood_area_km2:.4f} km²")
    print(f"  Threshold used  : {threshold}")
    print(f"{'='*45}")

    return results


print("predict_flood() function ready!")
print("Usage: results = predict_flood('path/to/image_S1Hand.tif')")

In [ ]:
# ============================================================
#  Section 7 — Cell 2: Run Predictions on 5 Test Images
#  Shows flood area statistics for different flood events
# ============================================================

import pandas as pd

# Use flood-rich test images for demonstration
test_flood_rich = []
for i, lf in enumerate(test_lbl):
    import rasterio
    with rasterio.open(lf) as src:
        m = src.read(1)
    if (m == 1).sum() > 5000:
        test_flood_rich.append(i)

# Run on top 5 flood-rich test images
demo_indices   = test_flood_rich[:5]
all_predictions = []

print("Running flood predictions...\n")
for idx in demo_indices:
    res = predict_flood(test_s1[idx])
    all_predictions.append(res)

# Summary table
print("\nFlood Area Summary:")
print(f"{'='*65}")
print(f"  {'Event':<30} {'Flood%':>8} {'Area(km²)':>12}")
print(f"{'-'*65}")
for r in all_predictions:
    name = r['name'].replace('_S1Hand.tif', '')[:30]
    print(f"  {name:<30} {r['flood_pct']:>7.2f}%  {r['flood_area_km2']:>10.4f}")
print(f"{'='*65}")

total_area = sum(r['flood_area_km2'] for r in all_predictions)
print(f"  {'Total estimated flood area':<30} {'':>8} {total_area:>10.4f} km²")

In [ ]:
# ============================================================
#  Section 7 — Cell 3: Flood Area Visualization (Paper Fig 4)
#  Shows VV band + flood mask + probability map + area stats
#  for each predicted flood event
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

n = len(all_predictions)
fig = plt.figure(figsize=(16, 4 * n))
fig.suptitle('Flood Area Prediction — Improved U-Net',
             fontsize=15, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(n, 4, figure=fig,
                        hspace=0.35, wspace=0.05)

col_titles = ['VV Band (SAR Input)', 'VH Band',
              'Flood Probability', 'Predicted Flood Mask']

for row, r in enumerate(all_predictions):
    vv   = r['image_vv']
    vh   = r['image_vh']
    prob = r['pred_prob']
    mask = r['pred_mask']
    name = r['name'].replace('_S1Hand.tif', '')

    axes = [fig.add_subplot(gs[row, c]) for c in range(4)]

    # Col titles only on first row
    if row == 0:
        for ax, t in zip(axes, col_titles):
            ax.set_title(t, fontsize=11, fontweight='bold')

    # VV band
    axes[0].imshow(vv, cmap='gray', vmin=0, vmax=1)
    axes[0].set_ylabel(name[:22], fontsize=8)

    # VH band
    axes[1].imshow(vh, cmap='gray', vmin=0, vmax=1)

    # Probability heatmap
    im = axes[2].imshow(prob, cmap='RdYlBu_r', vmin=0, vmax=1)

    # Binary flood mask with stats annotation
    flood_disp = np.zeros((*mask.shape, 3))
    flood_disp[mask == 1] = [0.22, 0.56, 1.0]   # blue flood
    flood_disp[mask == 0] = [0.95, 0.94, 0.92]   # light gray land
    axes[3].imshow(flood_disp)
    axes[3].text(
        0.02, 0.02,
        f"Flood: {r['flood_pct']:.1f}%\n{r['flood_area_km2']:.3f} km²",
        transform = axes[3].transAxes,
        fontsize   = 9, color='white',
        fontweight = 'bold',
        bbox       = dict(boxstyle='round', facecolor='#185FA5',
                          alpha=0.8, pad=0.3),
        va='bottom'
    )

    for ax in axes:
        ax.axis('off')

# Colorbar for probability
cbar_ax = fig.add_axes([0.52, -0.01, 0.22, 0.015])
plt.colorbar(im, cax=cbar_ax, orientation='horizontal',
             label='Flood probability')

plt.savefig(f'{OUTPUT_DIR}/paper_fig4_flood_area.png',
            dpi=300, bbox_inches='tight')
plt.show()
print("Saved → paper_fig4_flood_area.png")

In [ ]:
# ============================================================
#  Section 7 — Cell 4: Flood Area Bar Chart (Paper Figure 5)
#  Compares predicted vs actual flood area per event
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import rasterio

PIXEL_AREA_KM2 = 0.0001

# Get actual flood areas from ground truth
names, pred_areas, true_areas = [], [], []

for i, (r, idx) in enumerate(zip(all_predictions, demo_indices)):
    with rasterio.open(test_lbl[idx]) as src:
        true_mask = src.read(1)
    true_flood_px   = int((true_mask == 1).sum())
    true_area_km2   = true_flood_px * PIXEL_AREA_KM2

    name = r['name'].replace('_S1Hand.tif','')[:18]
    names.append(name)
    pred_areas.append(r['flood_area_km2'])
    true_areas.append(true_area_km2)

x = np.arange(len(names))
w = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Predicted vs Actual Flood Area',
             fontsize=13, fontweight='bold')

# Bar chart
ax1.bar(x - w/2, true_areas,  w, label='Ground Truth',
       color='#B4B2A9', edgecolor='white')
ax1.bar(x + w/2, pred_areas,  w, label='Predicted',
       color='#185FA5', edgecolor='white')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
ax1.set_ylabel('Flood Area (km²)')
ax1.set_title('Flood Area Comparison')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Scatter plot: predicted vs actual
ax2.scatter(true_areas, pred_areas,
            color='#185FA5', s=80, zorder=5)
max_val = max(max(true_areas), max(pred_areas)) * 1.1
ax2.plot([0, max_val], [0, max_val],
         '--', color='#E24B4A', lw=1.5, label='Perfect prediction')
for i, name in enumerate(names):
    ax2.annotate(name[:10],
                (true_areas[i], pred_areas[i]),
                fontsize=8, xytext=(5,5),
                textcoords='offset points')
ax2.set_xlabel('Actual Flood Area (km²)')
ax2.set_ylabel('Predicted Flood Area (km²)')
ax2.set_title('Predicted vs Actual (scatter)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/paper_fig5_flood_area_comparison.png',
            dpi=300, bbox_inches='tight')
plt.show()
print("Saved → paper_fig5_flood_area_comparison.png")
print("\nSection 7 complete!")
print("All paper figures saved to /content/outputs/")

In [ ]:
# ============================================================
#  Section 8 — Cell 1: Verify all files before download
# ============================================================

import os

OUTPUT_DIR = "/content/outputs"
MODEL_DIR  = "/content/models"

# Expected files
expected = [
    # Paper figures
    f"{OUTPUT_DIR}/paper_fig1_confusion_matrix.png",
    f"{OUTPUT_DIR}/paper_fig2_prediction_grid.png",
    f"{OUTPUT_DIR}/paper_fig3_model_comparison.png",
    f"{OUTPUT_DIR}/paper_fig4_flood_area.png",
    f"{OUTPUT_DIR}/paper_fig5_flood_area_comparison.png",
    f"{OUTPUT_DIR}/fig6_loss_curves.png",
    f"{OUTPUT_DIR}/fig7_metric_curves.png",
    # Data files
    f"{OUTPUT_DIR}/training_log.csv",
    f"{OUTPUT_DIR}/training_summary.json",
    f"{OUTPUT_DIR}/test_results.json",
    # Trained model
    f"{MODEL_DIR}/best_unet_flood.keras",
]

print("Checking files...")
print(f"{'='*55}")
all_ok = True
for path in expected:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1024 if exists else 0
    status = "OK" if exists else "MISSING"
    icon   = "✓" if exists else "✗"
    name   = os.path.basename(path)
    print(f"  {icon} {name:<45} {size:>7.1f} KB")
    if not exists:
        all_ok = False

print(f"{'='*55}")
if all_ok:
    print("  All files present — ready to download!")
else:
    print("  Some files missing — check above sections")

In [ ]:
# ============================================================
#  Section 8 — Cell 2: Package everything into one zip
#  Creates: flood_monitoring_project.zip
#  Contains: all figures + model + logs + results JSON
# ============================================================

import shutil, os

OUTPUT_DIR  = "/content/outputs"
MODEL_DIR   = "/content/models"
PACKAGE_DIR = "/content/flood_monitoring_project"

# Create clean package folder
if os.path.exists(PACKAGE_DIR):
    shutil.rmtree(PACKAGE_DIR)
os.makedirs(PACKAGE_DIR)

# Sub-folders
os.makedirs(f"{PACKAGE_DIR}/paper_figures")
os.makedirs(f"{PACKAGE_DIR}/model")
os.makedirs(f"{PACKAGE_DIR}/logs")

# Copy paper figures
figures = [
    "paper_fig1_confusion_matrix.png",
    "paper_fig2_prediction_grid.png",
    "paper_fig3_model_comparison.png",
    "paper_fig4_flood_area.png",
    "paper_fig5_flood_area_comparison.png",
    "fig6_loss_curves.png",
    "fig7_metric_curves.png",
    "fig1_band_visualization.png",
    "fig2_class_distribution.png",
    "fig3_backscatter_distribution.png",
    "fig4_augmentation_check.png",
]
for f in figures:
    src = f"{OUTPUT_DIR}/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"{PACKAGE_DIR}/paper_figures/{f}")
        print(f"  ✓ {f}")

# Copy trained model
model_src = f"{MODEL_DIR}/best_unet_flood.keras"
if os.path.exists(model_src):
    shutil.copy(model_src, f"{PACKAGE_DIR}/model/best_unet_flood.keras")
    print(f"  ✓ best_unet_flood.keras")

# Copy logs and results
logs = ["training_log.csv", "training_summary.json", "test_results.json"]
for f in logs:
    src = f"{OUTPUT_DIR}/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"{PACKAGE_DIR}/logs/{f}")
        print(f"  ✓ {f}")

# Create zip
zip_path = "/content/flood_monitoring_project"
shutil.make_archive(zip_path, 'zip', '/content',
                      'flood_monitoring_project')

zip_size = os.path.getsize(zip_path + '.zip') / (1024**2)
print(f"\nZip created: flood_monitoring_project.zip")
print(f"Size: {zip_size:.1f} MB")
print(f"\nFolder structure:")
print("  flood_monitoring_project/")
print("  ├── paper_figures/   ← all 11 PNG figures (300 DPI)")
print("  ├── model/           ← best_unet_flood.keras")
print("  └── logs/            ← training_log.csv + results JSON")

In [ ]:
# ============================================================
#  Section 8 — Cell 3: Download to your computer
#  Browser will prompt you to save the zip file
# ============================================================

from google.colab import files

print("Starting download...")
print("Browser will show a save dialog — choose your folder.")
print("If no dialog appears, check your browser popup settings.\n")

files.download('/content/flood_monitoring_project.zip')

print("Download started!")

In [ ]:
# ============================================================
#  Section 8 — Cell 4: Save to Google Drive
#  Safer than local download — files persist after session ends
#  Run this as BACKUP even if Cell 3 worked
# ============================================================

from google.colab import drive
import shutil, os

# Mount Google Drive
drive.mount('/content/drive')

# Create project folder in Drive
DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copy zip to Drive
shutil.copy(
    '/content/flood_monitoring_project.zip',
    f'{DRIVE_DIR}/flood_monitoring_project.zip'
)

# Also copy model separately to Drive (easy to reload later)
os.makedirs(f"{DRIVE_DIR}/model", exist_ok=True)
shutil.copy(
    '/content/models/best_unet_flood.keras',
    f'{DRIVE_DIR}/model/best_unet_flood.keras'
)

# Copy all figures individually to Drive
os.makedirs(f"{DRIVE_DIR}/figures", exist_ok=True)
import glob
for f in glob.glob('/content/outputs/*.png'):
    shutil.copy(f, f"{DRIVE_DIR}/figures/")

print("Saved to Google Drive:")
print(f"  {DRIVE_DIR}/")
print("  ├── flood_monitoring_project.zip")
print("  ├── model/best_unet_flood.keras")
print("  └── figures/  ← all PNG figures")
print(f"\nFiles are now safe in your Google Drive!")
print(f"You can access them at: drive.google.com")

In [ ]:
# ============================================================
#  Section 8 — Cell 5: Reload Model in Future Session
#  Use this code next time you open Colab to load your
#  trained model without retraining from scratch
# ============================================================

# ── Step 1: Mount Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Step 2: Install dependencies ─────────────────────────
!pip install rasterio albumentations -q

# ── Step 3: Copy model from Drive to Colab ───────────────
import shutil, os
os.makedirs('/content/models', exist_ok=True)
shutil.copy(
    '/content/drive/MyDrive/FloodMonitoring_Project/model/best_unet_flood.keras',
    '/content/models/best_unet_flood.keras'
)

# ── Step 4: Redefine custom functions ────────────────────
# (paste your loss functions from Section 4 Cell 1 here)
# weighted_dice_bce_loss, dice_coefficient, iou_score
import tensorflow as tf
import tensorflow.keras.backend as K

def dice_loss(y_true, y_pred, smooth=1e-6):
    """
    Dice loss — directly optimizes F1/Dice score.
    Works well with class imbalance.
    """
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    dice = (2.0 * intersection + smooth) / \
           (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)
    return 1.0 - dice


def weighted_dice_bce_loss(y_true, y_pred):
    """
    Combined weighted Dice + BCE loss.
    - BCE handles per-pixel classification
    - Dice handles overall overlap (fixes imbalance)
    - sample_weight (8.43x for flood) already applied via
      model.fit(..., sample_weight=wt) in Section 5
    """
    smooth   = 1e-6
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.reshape(y_pred, [-1])

    # BCE component
    bce = tf.keras.losses.binary_crossentropy(
        y_true_f, y_pred_f)

    # Dice component
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    dice_coef    = (2.0 * intersection + smooth) / \
                   (tf.reduce_sum(y_true_f) +
                    tf.reduce_sum(y_pred_f) + smooth)
    dice = 1.0 - dice_coef

    # 50/50 combination
    return 0.5 * bce + 0.5 * dice


def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """Dice coefficient metric (= F1 score for binary segmentation)."""
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(
        tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / \
           (tf.reduce_sum(y_true_f) +
            tf.reduce_sum(y_pred_f) + smooth)


def iou_score(y_true, y_pred, smooth=1e-6):
    """Intersection over Union metric."""
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(
        tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union        = tf.reduce_sum(y_true_f) + \
                   tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


print("Loss functions defined:")
print("  weighted_dice_bce_loss  ← training loss")
print("  dice_coefficient        ← F1 metric")
print("  iou_score               ← IoU metric")

# Quick sanity check
dummy_true = tf.constant([[[1.0],[0.0]],[[1.0],[1.0]]])
dummy_pred = tf.constant([[[0.9],[0.1]],[[0.8],[0.7]]])
print(f"\nSanity check (near-perfect pred):")
print(f"  dice_loss      : {dice_loss(dummy_true, dummy_pred):.4f}  ← close to 0")
print(f"  dice_coeff     : {dice_coefficient(dummy_true, dummy_pred):.4f}  ← close to 1")
print(f"  iou_score      : {iou_score(dummy_true, dummy_pred):.4f}  ← close to 1")

# ── Step 5: Load model ───────────────────────────────────
import tensorflow as tf

model = tf.keras.models.load_model(
    '/content/models/best_unet_flood.keras',
    custom_objects={
        'weighted_dice_bce_loss' : weighted_dice_bce_loss,
        'dice_coefficient'       : dice_coefficient,
        'iou_score'              : iou_score,
    }
)

print("Model loaded successfully!")
print(f"Input  : {model.input_shape}")
print(f"Output : {model.output_shape}")
print("\nReady to run predict_flood() on any new SAR image!")

In [ ]:
# ============================================================
#  Inspect your custom SAR images before prediction
#  Checks bands, shape, value range, NaN count
# ============================================================

import rasterio
import numpy as np

PRE_PATH  = "/content/preflood_flood_RAW.tif"
POST_PATH = "/content/postflood_flood_RAW.tif"

for label, path in [("PRE-FLOOD", PRE_PATH),
                      ("POST-FLOOD", POST_PATH)]:
    with rasterio.open(path) as src:
        print(f"\n{label}: {path.split('/')[-1]}")
        print(f"{'='*50}")
        print(f"  Shape (bands,H,W) : {src.count} × {src.height} × {src.width}")
        print(f"  CRS               : {src.crs}")
        print(f"  Resolution        : {src.res}")
        print(f"  Dtype             : {src.dtypes[0]}")
        print(f"  Nodata value      : {src.nodata}")

        for b in range(1, src.count + 1):
            band = src.read(b).astype(np.float32)
            valid = band[np.isfinite(band)]
            print(f"  Band {b} — min:{valid.min():.2f}  "
                  f"max:{valid.max():.2f}  "
                  f"mean:{valid.mean():.2f}  "
                  f"NaN:{np.sum(np.isnan(band)):,}")

print(f"\nBoth files found and readable!")

In [ ]:
# ============================================================
#  Cell 2: Smart loader for any image size
#  If image is not 512×512, tiles it into patches,
#  runs prediction on each, stitches back together
#  Works for any Sentinel-1 image size
# ============================================================

import numpy as np
import rasterio
from skimage.transform import resize as sk_resize

def predict_any_size(path, threshold=0.5):
    """
    Predict flood on any size Sentinel-1 image.
    - If 512×512 → direct prediction
    - If larger  → tile into 512×512 patches → predict each → stitch
    - If smaller → resize to 512×512 → predict → resize back
    """
    PIXEL_AREA_KM2 = 0.0001

    # Load
    with rasterio.open(path) as src:
        vv_raw = src.read(1).astype(np.float32)
        vh_raw = src.read(2).astype(np.float32)
        H, W   = src.height, src.width

    print(f"  Image size: {H} × {W}")

    # Fix NaN
    valid  = np.isfinite(vv_raw) & np.isfinite(vh_raw)
    vv_raw = np.where(np.isfinite(vv_raw), vv_raw, 0.0)
    vh_raw = np.where(np.isfinite(vh_raw), vh_raw, 0.0)

    # Normalize
    vv_n = np.clip((vv_raw - (-30.0)) / (0.0 - (-30.0)), 0, 1)
    vh_n = np.clip((vh_raw - (-35.0)) / (-5.0 - (-35.0)), 0, 1)
    vv_n = np.where(valid, vv_n, 0.0)
    vh_n = np.where(valid, vh_n, 0.0)

    if H == 512 and W == 512:
        # Direct prediction
        print(f"  Mode: direct (512×512)")
        img   = np.stack([vv_n, vh_n], axis=-1)[np.newaxis]
        prob  = model.predict(img, verbose=0)[0,...,0]

    elif H < 512 or W < 512:
        # Resize up to 512×512
        print(f"  Mode: resize to 512×512")
        vv_r  = sk_resize(vv_n, (512,512), preserve_range=True)
        vh_r  = sk_resize(vh_n, (512,512), preserve_range=True)
        img   = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]
        prob_r = model.predict(img, verbose=0)[0,...,0]
        prob  = sk_resize(prob_r, (H, W), preserve_range=True)

    else:
        # Tile large image into 512×512 patches
        print(f"  Mode: tiled prediction")
        prob = np.zeros((H, W), dtype=np.float32)
        count = np.zeros((H, W), dtype=np.float32)
        stride = 256   # 50% overlap between tiles

        for y in range(0, H - 512 + 1, stride):
            for x in range(0, W - 512 + 1, stride):
                vv_t = vv_n[y:y+512, x:x+512]
                vh_t = vh_n[y:y+512, x:x+512]
                img_t = np.stack([vv_t, vh_t], axis=-1)[np.newaxis]
                p = model.predict(img_t, verbose=0)[0,...,0]
                prob[y:y+512, x:x+512]  += p
                count[y:y+512, x:x+512] += 1

        prob = prob / np.maximum(count, 1)

    # Binary mask + stats
    pred_mask      = (prob > threshold).astype(np.uint8)
    valid_px       = int(valid.sum())
    flood_px       = int((pred_mask & valid).sum())
    flood_pct      = 100 * flood_px / max(valid_px, 1)
    flood_area_km2 = flood_px * PIXEL_AREA_KM2

    return {
        'name'          : path.split('/')[-1],
        'vv'            : vv_n,
        'vh'            : vh_n,
        'prob'          : prob,
        'mask'          : pred_mask,
        'valid'         : valid,
        'flood_px'      : flood_px,
        'valid_px'      : valid_px,
        'flood_pct'     : round(flood_pct,    2),
        'flood_area_km2': round(flood_area_km2, 4),
    }

print("predict_any_size() ready!")

In [ ]:
# ============================================================
#  Cell 3: Run Flood Prediction on Pre & Post Flood Images
# ============================================================

PRE_PATH  = "/content/preflood_flood_RAW.tif"
POST_PATH = "/content/postflood_flood_RAW.tif"

print("Running prediction on PRE-FLOOD image...")
pre_result  = predict_any_size(PRE_PATH)

print("\nRunning prediction on POST-FLOOD image...")
post_result = predict_any_size(POST_PATH)

# Summary
print(f"\n{'='*52}")
print(f"  FLOOD DETECTION RESULTS — YOUR IMAGES")
print(f"{'='*52}")
print(f"  {'Metric':<22} {'Pre-Flood':>12} {'Post-Flood':>12}")
print(f"{'-'*52}")
print(f"  {'Valid pixels':<22} {pre_result['valid_px']:>12,} {post_result['valid_px']:>12,}")
print(f"  {'Flood pixels':<22} {pre_result['flood_px']:>12,} {post_result['flood_px']:>12,}")
print(f"  {'Flood coverage (%)':<22} {pre_result['flood_pct']:>11.2f}% {post_result['flood_pct']:>11.2f}%")
print(f"  {'Flood area (km²)':<22} {pre_result['flood_area_km2']:>12.4f} {post_result['flood_area_km2']:>12.4f}")
print(f"{'='*52}")

# Flood change
change_km2  = post_result['flood_area_km2'] - pre_result['flood_area_km2']
change_pct  = post_result['flood_pct']     - pre_result['flood_pct']
print(f"\n  Flood area INCREASE : {change_km2:+.4f} km²")
print(f"  Flood coverage change: {change_pct:+.2f}%")

In [ ]:
# ============================================================
#  Cell 4: Pre vs Post Flood Comparison Figure (Paper Fig 6)
#  This is the strongest real-world result figure for your paper
#  Shows exactly what happened during the flood event
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle(
    'Flood Monitoring — Pre vs Post Flood Detection\n'
    'Sentinel-1 SAR  |  Improved U-Net Prediction',
    fontsize=15, fontweight='bold'
)

col_titles = ['VV Band (SAR)', 'VH Band (SAR)',
              'Flood Probability', 'Flood Mask']
row_labels = ['Pre-Flood', 'Post-Flood']

for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=12, fontweight='bold')

for row, (res, rlabel) in enumerate(
        zip([pre_result, post_result], row_labels)):

    axes[row, 0].imshow(res['vv'], cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_ylabel(
        f"{rlabel}\nFlood: {res['flood_pct']:.1f}%\n{res['flood_area_km2']:.3f} km²",
        fontsize=10, fontweight='bold'
    )

    axes[row, 1].imshow(res['vh'], cmap='gray', vmin=0, vmax=1)

    im = axes[row, 2].imshow(
        res['prob'], cmap='RdYlBu_r', vmin=0, vmax=1)

    # Flood mask — blue=flood, light gray=land
    disp = np.zeros((*res['mask'].shape, 3))
    disp[res['mask'] == 1] = [0.13, 0.47, 0.71]
    disp[res['mask'] == 0] = [0.95, 0.94, 0.92]
    axes[row, 3].imshow(disp)
    axes[row, 3].text(
        0.03, 0.04,
        f"{res['flood_area_km2']:.3f} km²\n({res['flood_pct']:.1f}%)",
        transform   = axes[row, 3].transAxes,
        fontsize    = 11, color='white', fontweight='bold',
        bbox        = dict(boxstyle='round',
                          facecolor='#185FA5', alpha=0.85),
        va='bottom'
    )

    for ax in axes[row]:
        ax.axis('off')

# Colorbar
cbar_ax = fig.add_axes([0.52, 0.02, 0.22, 0.02])
plt.colorbar(im, cax=cbar_ax, orientation='horizontal',
             label='Flood probability')

# Flood change annotation
change = post_result['flood_area_km2'] - pre_result['flood_area_km2']
fig.text(0.5, -0.01,
         f"Flood area increase: {change:+.4f} km²  "
          f"({pre_result['flood_area_km2']:.4f} → {post_result['flood_area_km2']:.4f} km²)",
         ha='center', fontsize=12, fontweight='bold',
         color='#185FA5')

plt.tight_layout()
plt.savefig('/content/outputs/paper_fig6_prepost_comparison.png',
            dpi=300, bbox_inches='tight')
plt.show()
print("Saved → paper_fig6_prepost_comparison.png")
print("\nThis is your strongest real-world result figure!")

In [ ]:

# ============================================================
#  CELL 1 — Install libraries, mount Drive, auth GEE
#  Run this first every new Colab session
# ============================================================

!pip install rasterio scikit-image geemap earthengine-api -q

from google.colab import drive
import ee, os, json
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds
from skimage.transform import resize as sk_resize
import geemap

# Mount Drive
drive.mount('/content/drive')

# Authenticate GEE
ee.Authenticate()
ee.Initialize(project='supple-tracker-472516-b6')  # ← replace with yours

# ── All paths ────────────────────────────────────────────
DRIVE_DIR  = "/content/drive/MyDrive/FloodMonitoring_Project/vijayawada_sar"
OUTPUT_DIR = "/content/drive/MyDrive/FloodMonitoring_Project/vijayawada_figures"
os.makedirs(DRIVE_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Time periods ─────────────────────────────────────────
PERIODS = {
    "Pre_Flood"    : ('2024-08-01', '2024-08-31'),
    "During_Flood" : ('2024-09-01', '2024-09-15'),
    "Post_Flood"   : ('2024-09-16', '2024-10-15'),
}

# ── Vijayawada full AOI ──────────────────────────────────
VIJAYAWADA_AOI = ee.Geometry.Rectangle([
    80.30, 16.40, 80.75, 16.65
])

# ── 10 Locality bounding boxes ───────────────────────────
LOCALITIES = {
    "Vijayawada City Core" : [80.60, 16.50, 80.68, 16.56],
    "Nunna"                : [80.67, 16.52, 80.72, 16.56],
    "Kundavari Kandrika"   : [80.66, 16.55, 80.72, 16.60],
    "Ambapuram"            : [80.60, 16.56, 80.66, 16.60],
    "Jakkampudi"           : [80.58, 16.56, 80.63, 16.60],
    "Gollapudi"            : [80.52, 16.52, 80.58, 16.56],
    "Undavalli"            : [80.60, 16.46, 80.66, 16.50],
    "Tadepalli"            : [80.60, 16.43, 80.66, 16.47],
    "Ajit Singh Nagar"     : [80.62, 16.50, 80.66, 16.53],
    "Budameru Floodplain"  : [80.58, 16.52, 80.64, 16.57],
}

# TIF file paths
TIF_PATHS = {p: f"{DRIVE_DIR}/vijayawada_{p}_S1.tif"
             for p in PERIODS}
PERM_WATER_PATH = f"{DRIVE_DIR}/permanent_water.tif"

print("Setup complete!")
print(f"  Drive dir  : {DRIVE_DIR}")
print(f"  Output dir : {OUTPUT_DIR}")
print(f"  Periods    : {list(PERIODS.keys())}")
print(f"  Localities : {len(LOCALITIES)}")

In [ ]:
# ============================================================
#  CELL 2 FIXED — Download SAR with correct scale
#  Problem: AOI was 251 MB, GEE limit is 50 MB
#  Fix 1: scale=30m (was 10m) → 9x smaller file
#  Fix 2: tighter AOI covering only flood-affected area
#  Note: 30m is still sufficient for flood mapping
# ============================================================

import geemap, os, ee

# ── Tighter AOI — covers all 10 localities exactly ───────
VIJAYAWADA_AOI = ee.Geometry.Rectangle([
    80.50, 16.42,   # tighter than before (was 80.30, 16.40)
    80.75, 16.62    # covers all 10 localities
])

# Update global AOI
SCALE = 30   # metres — 30m is fine for flood mapping

def get_s1(aoi, start, end):
    return ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(aoi) \
        .filterDate(start, end) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.listContains(
            'transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains(
            'transmitterReceiverPolarisation', 'VH')) \
        .select(['VV', 'VH']) \
        .mean() \
        .clip(aoi)

# ── Check size before downloading ────────────────────────
def check_size(img, aoi, scale):
    """Estimate download size in MB before downloading."""
    info  = img.getInfo()
    bands = len(info['bands'])
    dims  = aoi.bounds().getInfo()['coordinates'][0]
    lons  = [c[0] for c in dims]
    lats  = [c[1] for c in dims]
    w_m   = (max(lons)-min(lons)) * 111000
    h_m   = (max(lats)-min(lats)) * 111000
    px    = (w_m/scale) * (h_m/scale)
    mb    = (px * bands * 4) / (1024**2)
    return mb

# Test size first
test_img = get_s1(VIJAYAWADA_AOI,
                   '2024-08-01', '2024-08-31')
est_mb   = check_size(test_img, VIJAYAWADA_AOI, SCALE)
print(f"Estimated download size at {SCALE}m: {est_mb:.1f} MB")
print(f"GEE limit: 50 MB")
print(f"Status: {'OK' if est_mb < 48 else 'STILL TOO LARGE — reduce scale'}")

# ── Download all 3 periods + water mask ──────────────────
for period, (start, end) in PERIODS.items():
    path = TIF_PATHS[period]
    if os.path.exists(path):
        mb = os.path.getsize(path)/(1024**2)
        print(f"  SKIP {period} — exists ({mb:.1f} MB)")
        continue
    print(f"\n  Downloading {period} ({start} → {end})...")
    try:
        geemap.ee_export_image(
            get_s1(VIJAYAWADA_AOI, start, end),
            filename      = path,
            scale         = SCALE,
            region        = VIJAYAWADA_AOI,
            file_per_band = False
        )
        mb = os.path.getsize(path)/(1024**2)
        print(f"  Saved {period} → {mb:.1f} MB")
    except Exception as e:
        print(f"  ERROR: {e}")

# ── Download JRC permanent water mask ────────────────────
if os.path.exists(PERM_WATER_PATH):
    mb = os.path.getsize(PERM_WATER_PATH)/(1024**2)
    print(f"\n  SKIP permanent water — exists ({mb:.1f} MB)")
else:
    print(f"\n  Downloading JRC permanent water mask...")
    perm = ee.Image('JRC/GSW1_4/GlobalSurfaceWater') \
        .select('seasonality') \
        .gte(10) \
        .clip(VIJAYAWADA_AOI)
    try:
        geemap.ee_export_image(
            perm,
            filename      = PERM_WATER_PATH,
            scale         = SCALE,
            region        = VIJAYAWADA_AOI,
            file_per_band = False
        )
        mb = os.path.getsize(PERM_WATER_PATH)/(1024**2)
        print(f"  Saved permanent water → {mb:.1f} MB")
    except Exception as e:
        print(f"  ERROR: {e}")

# ── Final check ──────────────────────────────────────────
print(f"\nDownload summary:")
all_ok = True
for label, path in {**TIF_PATHS,
        'Permanent_Water': PERM_WATER_PATH}.items():
    exists = os.path.exists(path)
    mb     = os.path.getsize(path)/(1024**2) if exists else 0
    icon   = "✓" if exists else "✗"
    print(f"  {icon} {label:<20}: {mb:.1f} MB")
    if not exists: all_ok = False

print(f"\n{'All files ready!' if all_ok else 'Some files missing!'}")
print(f"Proceed to Cell 3 for visualization.")

In [ ]:
# ============================================================
#  Fix: Re-download JRC permanent water mask
#  Problem: 0.0 MB means export silently failed
#  Fix: delete empty file → force fresh download
# ============================================================

import os, geemap, ee
import rasterio
import numpy as np

# Delete empty file so skip check doesn't trigger
if os.path.exists(PERM_WATER_PATH):
    os.remove(PERM_WATER_PATH)
    print("Deleted empty permanent water file")

# Re-download with force
print("Re-downloading JRC permanent water mask...")

perm_img = ee.Image('JRC/GSW1_4/GlobalSurfaceWater') \
    .select('seasonality') \
    .gte(10) \
    .toFloat() \
    .clip(VIJAYAWADA_AOI)

geemap.ee_export_image(
    perm_img,
    filename      = PERM_WATER_PATH,
    scale         = SCALE,
    region        = VIJAYAWADA_AOI,
    file_per_band = False
)

# Verify
mb = os.path.getsize(PERM_WATER_PATH) / (1024**2)
print(f"File size: {mb:.2f} MB")

if mb < 0.01:
    print("Still empty — using fallback method...")

    # Fallback: use occurrence > 80% instead of seasonality
    perm_img2 = ee.Image('JRC/GSW1_4/GlobalSurfaceWater') \
        .select('occurrence') \
        .gte(80) \
        .toFloat() \
        .clip(VIJAYAWADA_AOI)

    if os.path.exists(PERM_WATER_PATH):
        os.remove(PERM_WATER_PATH)

    geemap.ee_export_image(
        perm_img2,
        filename      = PERM_WATER_PATH,
        scale         = SCALE,
        region        = VIJAYAWADA_AOI,
        file_per_band = False
    )
    mb = os.path.getsize(PERM_WATER_PATH) / (1024**2)
    print(f"Fallback file size: {mb:.2f} MB")

# Inspect the downloaded mask
with rasterio.open(PERM_WATER_PATH) as src:
    perm_data = src.read(1)
    print(f"\nPermanent water mask info:")
    print(f"  Shape      : {perm_data.shape}")
    print(f"  Unique vals: {np.unique(perm_data)}")
    print(f"  Water px   : {(perm_data==1).sum():,}")
    print(f"  Water area : {(perm_data==1).sum() * 0.0009:.2f} km²")
    print(f"  CRS        : {src.crs}")

In [ ]:
# ============================================================
#  Verify: Plot permanent water mask over SAR image
#  Krishna River must appear as permanent water (teal)
#  If it looks correct → proceed to Cell 3
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.windows import from_bounds
from skimage.transform import resize as sk_resize

# Load full AOI VV band from during-flood image
with rasterio.open(TIF_PATHS['During_Flood']) as src:
    vv_full = src.read(1).astype(np.float32)

with rasterio.open(PERM_WATER_PATH) as src:
    pw_full = src.read(1)

# Normalize SAR for display
vv_n = np.clip(
    np.where(np.isfinite(vv_full), vv_full, 0),
    -30, 0)
vv_n = (vv_n + 30) / 30

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Permanent Water Verification — Full Vijayawada AOI',
    fontsize=13, fontweight='bold'
)

# SAR VV
axes[0].imshow(vv_n, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('SAR VV Band\n(During Flood)',
                   fontsize=11, fontweight='bold')

# Permanent water mask
axes[1].imshow(pw_full, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('JRC Permanent Water Mask\n(blue=permanent water)',
                   fontsize=11, fontweight='bold')

# Overlay — SAR + permanent water highlighted
axes[2].imshow(vv_n, cmap='gray', vmin=0, vmax=1)
overlay = np.zeros((*pw_full.shape, 4))
overlay[pw_full == 1] = [0.25, 0.76, 0.65, 0.7]
axes[2].imshow(overlay)
axes[2].set_title('Overlay\n(teal = will be excluded from flood)',
                   fontsize=11, fontweight='bold')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/fig0_permanent_water_check.png",
    dpi=300, bbox_inches='tight')
plt.show()
print("Saved → fig0_permanent_water_check.png")
print("\nCheck: Krishna River should appear as teal in overlay.")
print("If correct → run Cell 3!")

In [ ]:
import os

files = {
    'Pre_Flood SAR'    : f"{DRIVE_DIR}/vijayawada_Pre_Flood_S1.tif",
    'During_Flood SAR' : f"{DRIVE_DIR}/vijayawada_During_Flood_S1.tif",
    'Post_Flood SAR'   : f"{DRIVE_DIR}/vijayawada_Post_Flood_S1.tif",
    'Permanent Water'  : PERM_WATER_PATH,
}

print("Drive file status:")
print("="*45)
for name, path in files.items():
    exists = os.path.exists(path)
    mb = os.path.getsize(path)/(1024**2) if exists else 0
    icon = "✓" if exists and mb > 0.01 else "✗"
    print(f"  {icon} {name:<20}: {mb:.2f} MB")

In [ ]:
# ============================================================
#  CELL 3 — Neat visualization of all downloaded SAR images
#  Shows VV + VH for all 3 periods side by side
#  Saves as paper-ready figure
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import rasterio
from rasterio.windows import from_bounds
from skimage.transform import resize as sk_resize

# Use Vijayawada City Core as demo area
DEMO  = LOCALITIES["Vijayawada City Core"]
PSIZE = 512

def load_crop(tif_path, bounds, band=1):
    with rasterio.open(tif_path) as src:
        win = from_bounds(
            bounds[0], bounds[1],
            bounds[2], bounds[3],
            src.transform)
        arr = src.read(band, window=win).astype(np.float32)
    arr = np.where(np.isfinite(arr), arr, 0.0)
    return sk_resize(arr, (PSIZE, PSIZE), preserve_range=True)

period_labels = {
    "Pre_Flood"    : "Pre-Flood\n(Aug 2024)",
    "During_Flood" : "During Flood\n(Sep 1–15, 2024)",
    "Post_Flood"   : "Post-Flood\n(Sep 16 – Oct 2024)",
}

fig, axes = plt.subplots(3, 3, figsize=(16, 15))
fig.suptitle(
    'Sentinel-1 SAR Imagery — Vijayawada City Core\n'
    'Pre-Flood | During Flood | Post-Flood',
    fontsize=15, fontweight='bold', y=1.01
)

col_titles = ['VV Polarisation', 'VH Polarisation', 'VV–VH Difference']
for col, t in enumerate(col_titles):
    axes[0, col].set_title(t, fontsize=12,
                            fontweight='bold', pad=10)

for row, (period, label) in \
        enumerate(period_labels.items()):
    vv = load_crop(TIF_PATHS[period], DEMO, band=1)
    vh = load_crop(TIF_PATHS[period], DEMO, band=2)

    # Normalize for display
    vv_n = np.clip((vv + 30) / 30, 0, 1)
    vh_n = np.clip((vh + 35) / 30, 0, 1)
    diff = vv_n - vh_n

    axes[row, 0].imshow(vv_n, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].imshow(vh_n, cmap='gray', vmin=0, vmax=1)
    im = axes[row, 2].imshow(diff, cmap='RdBu', vmin=-0.5, vmax=0.5)

    axes[row, 0].set_ylabel(label, fontsize=11,
                              fontweight='bold', labelpad=10)

    # Stats annotation on VV panel
    mean_vv = vv[vv > -40].mean()
    axes[row, 0].text(0.02, 0.04,
        f"Mean VV: {mean_vv:.1f} dB",
        transform=axes[row,0].transAxes,
        fontsize=9, color='white', fontweight='bold',
        bbox=dict(facecolor='black', alpha=0.6,
                  boxstyle='round'), va='bottom')

    for ax in axes[row]:
        ax.axis('off')

# Colorbar for diff panel
cbar_ax = fig.add_axes([0.68, 0.02, 0.20, 0.015])
plt.colorbar(im, cax=cbar_ax, orientation='horizontal',
             label='VV − VH (dB difference)')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig1_SAR_all_periods.png",
            dpi=300, bbox_inches='tight')
plt.show()
print("Saved → fig1_SAR_all_periods.png")

In [ ]:
# ============================================================
#  CELL 4 — Load trained U-Net + prediction function
#  Includes permanent water removal inside predict function
# ============================================================

import tensorflow as tf

# Redefine custom loss functions
def weighted_dice_bce_loss(y_true, y_pred):
    smooth   = 1e-6
    yt = tf.cast(tf.reshape(y_true,[-1]),tf.float32)
    yp = tf.reshape(y_pred,[-1])
    bce  = tf.keras.losses.binary_crossentropy(yt,yp)
    inter= tf.reduce_sum(yt*yp)
    dice = 1.0-(2.0*inter+smooth)/(
           tf.reduce_sum(yt)+tf.reduce_sum(yp)+smooth)
    return 0.5*bce+0.5*dice

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true,[-1]),tf.float32)
    yp = tf.cast(tf.reshape(
        tf.cast(y_pred>0.5,tf.float32),[-1]),tf.float32)
    inter=tf.reduce_sum(yt*yp)
    return (2.0*inter+smooth)/(
           tf.reduce_sum(yt)+tf.reduce_sum(yp)+smooth)

def iou_score(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true,[-1]),tf.float32)
    yp = tf.cast(tf.reshape(
        tf.cast(y_pred>0.5,tf.float32),[-1]),tf.float32)
    inter=tf.reduce_sum(yt*yp)
    union=tf.reduce_sum(yt)+tf.reduce_sum(yp)-inter
    return (inter+smooth)/(union+smooth)

# Load model from Drive
MODEL_PATH = "/content/drive/MyDrive/FloodMonitoring_Project/model/best_unet_flood.keras"
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        'weighted_dice_bce_loss': weighted_dice_bce_loss,
        'dice_coefficient'      : dice_coefficient,
        'iou_score'             : iou_score,
    }
)
print(f"Model loaded! Input: {model.input_shape}")

# ── Core predict function ────────────────────────────────
def predict_locality(tif_path, bounds, perm_water_path):
    """
    Crop locality → normalize → resize to 512×512
    → U-Net predict → remove permanent water → area stats
    """
    with rasterio.open(tif_path) as src:
        win = from_bounds(
            bounds[0],bounds[1],
            bounds[2],bounds[3],
            src.transform)
        vv = src.read(1,window=win).astype(np.float32)
        vh = src.read(2,window=win).astype(np.float32)

    with rasterio.open(perm_water_path) as src:
        pw_win = from_bounds(
            bounds[0],bounds[1],
            bounds[2],bounds[3],
            src.transform)
        perm = src.read(1,window=pw_win)

    # Fix NaN + normalize
    valid = np.isfinite(vv) & np.isfinite(vh)
    vv = np.where(valid, vv, 0.0)
    vh = np.where(valid, vh, 0.0)
    vv_n = np.clip((vv+30)/30,0,1)
    vh_n = np.clip((vh+35)/30,0,1)

    # Resize to 512×512
    vv_r  = sk_resize(vv_n,(512,512),preserve_range=True)
    vh_r  = sk_resize(vh_n,(512,512),preserve_range=True)
    perm_r= sk_resize(perm.astype(np.float32),
                      (512,512),preserve_range=True)>0.5

    # Predict
    img  = np.stack([vv_r,vh_r],axis=-1)[np.newaxis]
    prob = model.predict(img,verbose=0)[0,...,0]
    mask_raw   = (prob>0.5).astype(np.uint8)

    # Remove permanent water
    mask_clean = mask_raw.copy()
    mask_clean[perm_r] = 0

    # Area calculation
    w_km       = (bounds[2]-bounds[0])*111
    h_km       = (bounds[3]-bounds[1])*111
    total_area = w_km*h_km
    flood_pct  = float(mask_clean.mean())*100
    flood_area = round(total_area*flood_pct/100,3)

    return {
        'vv'        : vv_r,
        'vh'        : vh_r,
        'prob'      : prob,
        'mask_raw'  : mask_raw,
        'mask'      : mask_clean,
        'perm'      : perm_r,
        'flood_pct' : round(flood_pct,2),
        'flood_area': flood_area,
        'total_area': round(total_area,3),
    }

print("predict_locality() ready — permanent water removal included!")

In [ ]:
# ============================================================
#  CELL 5 — Run all 30 predictions (10 localities × 3 periods)
#  Saves results JSON to Drive immediately — crash safe
# ============================================================

import json

RESULTS_PATH = f"{DRIVE_DIR}/flood_results.json"

# Load existing results if crash happened
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        saved = json.load(f)
    print(f"Loaded existing results from Drive")
else:
    saved = {}

all_results = {}
print("Running 30 predictions...\n")

for period, tif_path in TIF_PATHS.items():
    all_results[period] = {}
    print(f"  Period: {period}")
    for name, bounds in LOCALITIES.items():
        r = predict_locality(tif_path, bounds, PERM_WATER_PATH)
        all_results[period][name] = r
        print(f"    ✓ {name:<25}"
              f" flood={r['flood_area']:>6.3f} km²"
              f" ({r['flood_pct']:.1f}%)")

    # Save after each period — crash safe
    save_data = {}
    for p in all_results:
        save_data[p] = {}
        for n in all_results[p]:
            rv = all_results[p][n]
            save_data[p][n] = {
                'flood_area': rv['flood_area'],
                'flood_pct' : rv['flood_pct'],
                'total_area': rv['total_area'],
            }
    with open(RESULTS_PATH,'w') as f:
        json.dump(save_data, f, indent=2)
    print(f"    → Saved to Drive after {period}\n")

# Print summary table
print(f"\n{'='*68}")
print(f"  {'Locality':<25} {'Pre':>8} {'During':>8} {'Post':>8} {'Peak':>8}")
print(f"{'-'*68}")
for name in LOCALITIES:
    pre_a  = all_results['Pre_Flood'][name]['flood_area']
    dur_a  = all_results['During_Flood'][name]['flood_area']
    post_a = all_results['Post_Flood'][name]['flood_area']
    peak   = max(pre_a, dur_a, post_a)
    print(f"  {name:<25} {pre_a:>7.3f}  {dur_a:>7.3f}  {post_a:>7.3f}  {peak:>7.3f}")
print(f"{'='*68}")

In [ ]:
# ============================================================
#  CELL 6 — Publication-quality flood maps
#  For each locality: VV | Flood mask | Perm water removed
#  3 panels × 3 periods = neat 3×3 grid per locality
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

for loc_name in LOCALITIES:
    fig, axes = plt.subplots(3, 3, figsize=(15, 14))
    fig.suptitle(
        f'Flood Detection — {loc_name}\n'
        'VV Band  |  Flood Probability  |  Flood Mask (water removed)',
        fontsize=13, fontweight='bold', y=1.01
    )

    col_titles = ['SAR VV Band',
                  'Flood Probability',
                  'Flood Mask\n(permanent water removed)']
    for col, t in enumerate(col_titles):
        axes[0,col].set_title(t, fontsize=11,
                               fontweight='bold')

    row_labels = ['Pre-Flood\n(Aug 2024)',
                  'During Flood\n(Sep 1–15)',
                  'Post-Flood\n(Sep 16–Oct)']

    for row, period in enumerate(PERIODS):
        r    = all_results[period][loc_name]
        vv   = r['vv']
        prob = r['prob']
        mask = r['mask']
        perm = r['perm']

        # Col 0: VV band
        axes[row,0].imshow(vv, cmap='gray', vmin=0, vmax=1)
        axes[row,0].set_ylabel(row_labels[row],
                                fontsize=10, fontweight='bold')

        # Col 1: probability heatmap
        im = axes[row,1].imshow(
            prob, cmap='RdYlBu_r', vmin=0, vmax=1)

        # Col 2: clean flood mask
        disp = np.zeros((*mask.shape,3))
        disp[mask==1]  = [0.13,0.47,0.71]  # blue = flood
        disp[perm==1]  = [0.40,0.76,0.65]  # teal = perm water
        disp[(mask==0)&(perm==0)] = [0.95,0.95,0.92]
        axes[row,2].imshow(disp)

        # Stats box
        axes[row,2].text(
            0.02, 0.04,
            f"Flood: {r['flood_area']:.3f} km²\n({r['flood_pct']:.1f}%)",
            transform=axes[row,2].transAxes,
            fontsize=9, color='white', fontweight='bold',
            bbox=dict(facecolor='#185FA5', alpha=0.85,
                      boxstyle='round'), va='bottom')

        for ax in axes[row]: ax.axis('off')

    # Legend
    legend = [
        mpatches.Patch(color=[0.13,0.47,0.71],
                        label='Flood water (new)'),
        mpatches.Patch(color=[0.40,0.76,0.65],
                        label='Permanent water (excluded)'),
        mpatches.Patch(color=[0.95,0.95,0.92],
                        label='Land'),
    ]
    fig.legend(handles=legend,
               loc='lower center',
               bbox_to_anchor=(0.5, -0.02),
               ncol=3, fontsize=11)

    plt.tight_layout()
    fname = loc_name.replace(' ','_')
    plt.savefig(
        f"{OUTPUT_DIR}/fig2_{fname}_flood_map.png",
        dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved fig2_{fname}_flood_map.png")

In [ ]:
# ============================================================
#  CELL 7 — Final summary figures
#  Fig 3: Bar chart all localities × 3 periods
#  Fig 4: Flood progression chart (Pre→During→Post)
#  Fig 5: Heatmap table
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

names      = list(LOCALITIES.keys())
short      = [n.replace(' ','\n') for n in names]
pre_areas  = [all_results['Pre_Flood'][n]['flood_area']    for n in names]
dur_areas  = [all_results['During_Flood'][n]['flood_area']  for n in names]
post_areas = [all_results['Post_Flood'][n]['flood_area']   for n in names]
x = np.arange(len(names)); w=0.25

# ── Fig 3: Grouped bar chart ─────────────────────────────
fig3, ax = plt.subplots(figsize=(18,6))
ax.bar(x-w, pre_areas,  w, label='Pre-Flood',
       color='#B4B2A9', edgecolor='white')
ax.bar(x,   dur_areas,  w, label='During Flood',
       color='#185FA5', edgecolor='white')
ax.bar(x+w, post_areas, w, label='Post-Flood',
       color='#9FE1CB', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(short, fontsize=9)
ax.set_ylabel('Flood Area (km²)', fontsize=12)
ax.set_title(
    'Flood Area by Locality — Pre / During / Post Flood\n'
    'Vijayawada Sep 2024 | Improved U-Net on Sentinel-1 SAR',
    fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig3_locality_bar_chart.png",
            dpi=300, bbox_inches='tight')
plt.show(); plt.close()

# ── Fig 4: Flood progression lines ──────────────────────
fig4, ax2 = plt.subplots(figsize=(12,6))
period_x  = [0, 1, 2]
colors    = plt.cm.tab10(np.linspace(0,1,len(names)))
for i, name in enumerate(names):
    vals = [pre_areas[i], dur_areas[i], post_areas[i]]
    ax2.plot(period_x, vals, 'o-',
            color=colors[i], lw=2,
            label=name, markersize=7)
ax2.set_xticks([0,1,2])
ax2.set_xticklabels(['Pre-Flood\n(Aug 2024)',
                      'During Flood\n(Sep 1–15)',
                      'Post-Flood\n(Sep 16–Oct)'],
                     fontsize=11)
ax2.set_ylabel('Flood Area (km²)', fontsize=12)
ax2.set_title(
    'Flood Progression — All Localities\n'
    'Vijayawada Sep 2024',
    fontsize=13, fontweight='bold')
ax2.legend(fontsize=8, bbox_to_anchor=(1.01,1),
          loc='upper left')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig4_flood_progression.png",
            dpi=300, bbox_inches='tight')
plt.show(); plt.close()

# ── Fig 5: Heatmap table ─────────────────────────────────
data = np.array([pre_areas, dur_areas, post_areas])
fig5, ax3 = plt.subplots(figsize=(16,4))
im = ax3.imshow(data, cmap='YlOrRd', aspect='auto')
ax3.set_xticks(range(len(names)))
ax3.set_xticklabels(names, rotation=30,
                     ha='right', fontsize=10)
ax3.set_yticks([0,1,2])
ax3.set_yticklabels(['Pre-Flood','During Flood',
                      'Post-Flood'], fontsize=11)
for i in range(3):
    for j in range(len(names)):
        ax3.text(j, i, f"{data[i,j]:.2f}",
                ha='center', va='center',
                fontsize=9, fontweight='bold',
                color='white' if data[i,j]>data.max()*0.5
                       else 'black')
ax3.set_title(
    'Flood Area (km²) Heatmap — All Localities × All Periods',
    fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax3, label='Flood Area (km²)')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig5_heatmap_table.png",
            dpi=300, bbox_inches='tight')
plt.show(); plt.close()

print("\nAll figures saved to Drive!")
print(f"  fig1_SAR_all_periods.png")
print(f"  fig2_[locality]_flood_map.png  (×10)")
print(f"  fig3_locality_bar_chart.png")
print(f"  fig4_flood_progression.png")
print(f"  fig5_heatmap_table.png")

In [ ]:
# ============================================================
#  Color Fix: Red flood water instead of blue
#  Red = danger/flood (more intuitive)
#  Blue = permanent water (kept as reference)
#  Gray = land (unchanged)
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Color scheme ─────────────────────────────────────────
COLOR_FLOOD = [0.85, 0.13, 0.13]   # RED   — new flood water
COLOR_PERM  = [0.20, 0.47, 0.71]   # BLUE  — permanent water (excluded)
COLOR_LAND  = [0.95, 0.94, 0.92]   # GRAY  — land

names        = list(LOCALITIES.keys())
period_keys  = list(PERIODS.keys())
period_labels= ['Pre-Flood\n(Aug 2024)',
                'During Flood\n(Sep 1–15)',
                'Post-Flood\n(Sep 16–Oct)']

# ── Part 1: 10 individual figures (one per locality) ─────
print("Generating individual locality figures...")
for loc_name in names:
    fig, axes = plt.subplots(3, 3, figsize=(15, 14))
    fig.suptitle(
        f'Flood Detection — {loc_name}\n'
        'VV Band  |  Flood Probability  |  Flood Mask',
        fontsize=13, fontweight='bold', y=1.01
    )
    col_titles = ['SAR VV Band',
                  'Flood Probability',
                  'Flood Mask\n(permanent water removed)']
    for c, t in enumerate(col_titles):
        axes[0, c].set_title(t, fontsize=11, fontweight='bold')

    for row, period in enumerate(period_keys):
        r    = all_results[period][loc_name]
        vv   = r['vv']
        prob = r['prob']
        mask = r['mask']
        perm = r['perm']

        # Col 0: VV SAR band
        axes[row, 0].imshow(vv, cmap='gray', vmin=0, vmax=1)
        axes[row, 0].set_ylabel(
            period_labels[row], fontsize=10, fontweight='bold')

        # Col 1: Flood probability heatmap
        im = axes[row, 1].imshow(
            prob, cmap='RdYlBu_r', vmin=0, vmax=1)

        # Col 2: Flood mask — RED flood, BLUE perm water, GRAY land
        disp = np.zeros((*mask.shape, 3))
        disp[(mask==0)&(perm==0)] = COLOR_LAND
        disp[perm==1]              = COLOR_PERM
        disp[mask==1]              = COLOR_FLOOD   # ← RED
        axes[row, 2].imshow(disp)

        # Stats box
        axes[row, 2].text(
            0.02, 0.04,
            f"Flood: {r['flood_area']:.3f} km²\n({r['flood_pct']:.1f}%)",
            transform  = axes[row, 2].transAxes,
            fontsize   = 9, color='white', fontweight='bold',
            bbox       = dict(facecolor='#A32D2D',
                             alpha=0.85, boxstyle='round'),
            va='bottom'
        )

        for ax in axes[row]: ax.axis('off')

    # Legend
    legend = [
        mpatches.Patch(color=COLOR_FLOOD,
                        label='Flood water (new)'),
        mpatches.Patch(color=COLOR_PERM,
                        label='Permanent water (excluded)'),
        mpatches.Patch(color=COLOR_LAND,
                        label='Land'),
    ]
    fig.legend(
        handles=legend, loc='lower center',
        bbox_to_anchor=(0.5, -0.02),
        ncol=3, fontsize=11)
    plt.tight_layout()
    fname = loc_name.replace(' ', '_')
    plt.savefig(
        f"{OUTPUT_DIR}/fig2_{fname}_flood_map.png",
        dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved fig2_{fname}_flood_map.png")

# ── Part 2: Combined overview figure ─────────────────────
print("\nGenerating combined overview figure...")
fig2, axes2 = plt.subplots(
    3, 10, figsize=(36, 12),
    gridspec_kw={'hspace':0.08, 'wspace':0.05}
)
fig2.suptitle(
    'Flood Detection — All Localities × Pre / During / Post\n'
    'Vijayawada Sep 2024 | U-Net on Sentinel-1 SAR',
    fontsize=16, fontweight='bold', y=1.02
)

for col, name in enumerate(names):
    axes2[0, col].set_title(
        name.replace(' ', '\n'),
        fontsize=8, fontweight='bold', pad=4)

    for row, period in enumerate(period_keys):
        r    = all_results[period][name]
        mask = r['mask']
        perm = r['perm']

        disp = np.zeros((*mask.shape, 3))
        disp[(mask==0)&(perm==0)] = COLOR_LAND
        disp[perm==1]              = COLOR_PERM
        disp[mask==1]              = COLOR_FLOOD   # ← RED
        axes2[row, col].imshow(disp)
        axes2[row, col].text(
            0.05, 0.04,
            f"{r['flood_area']:.2f}km²",
            transform  = axes2[row, col].transAxes,
            fontsize   = 7, color='white', fontweight='bold',
            bbox       = dict(facecolor='#A32D2D',
                             alpha=0.8, boxstyle='round',
                             pad=0.2),
            va='bottom'
        )
        axes2[row, col].axis('off')

for row in range(3):
    axes2[row, 0].set_ylabel(
        period_labels[row],
        fontsize=10, fontweight='bold')

legend2 = [
    mpatches.Patch(color=COLOR_FLOOD, label='Flood water (new)'),
    mpatches.Patch(color=COLOR_PERM,  label='Permanent water (excluded)'),
    mpatches.Patch(color=COLOR_LAND,  label='Land'),
]
fig2.legend(
    handles=legend2, loc='lower center',
    bbox_to_anchor=(0.5, -0.03),
    ncol=3, fontsize=12)

plt.savefig(
    f"{OUTPUT_DIR}/fig2_all_localities_combined.png",
    dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print("Saved → fig2_all_localities_combined.png")
print("\nAll figures saved with RED flood color!")

In [ ]:
# ============================================================
#  Flood Area Table — All Localities × Pre/During/Post
#  Vijayawada Sep 2024
# ============================================================

import pandas as pd

names        = list(LOCALITIES.keys())
period_keys  = list(PERIODS.keys())

# Build data
rows = []
for name in names:
    pre_a  = all_results['Pre_Flood'][name]['flood_area']
    dur_a  = all_results['During_Flood'][name]['flood_area']
    post_a = all_results['Post_Flood'][name]['flood_area']
    change = dur_a - pre_a
    total  = all_results['Pre_Flood'][name]['total_area']

    rows.append({
        'Locality'              : name,
        'Total Area (km²)'      : round(total, 2),
        'Pre-Flood (km²)'       : pre_a,
        'During Flood (km²)'    : dur_a,
        'Post-Flood (km²)'      : post_a,
        'Flood Increase (km²)'  : round(change, 3),
        'Peak Flood %'          : round(max(
            all_results['Pre_Flood'][name]['flood_pct'],
            all_results['During_Flood'][name]['flood_pct'],
            all_results['Post_Flood'][name]['flood_pct']), 2)
    })

df = pd.DataFrame(rows)

# Add totals row
totals = {
    'Locality'             : 'TOTAL',
    'Total Area (km²)'     : round(df['Total Area (km²)'].sum(), 2),
    'Pre-Flood (km²)'      : round(df['Pre-Flood (km²)'].sum(), 3),
    'During Flood (km²)'   : round(df['During Flood (km²)'].sum(), 3),
    'Post-Flood (km²)'     : round(df['Post-Flood (km²)'].sum(), 3),
    'Flood Increase (km²)' : round(df['Flood Increase (km²)'].sum(), 3),
    'Peak Flood %'         : round(df['Peak Flood %'].mean(), 2)
}
df = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)

# Print table
print("=" * 90)
print("  FLOOD AREA TABLE — Vijayawada Sep 2024")
print("  Source: Improved U-Net on Sentinel-1 SAR | Permanent water excluded")
print("=" * 90)
print(df.to_string(index=False))
print("=" * 90)

# Save as CSV to Drive
csv_path = f"{OUTPUT_DIR}/flood_area_table.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved → flood_area_table.csv")

# Save as nice image table
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(18, 5))
ax.axis('off')

table = ax.table(
    cellText  = df.values,
    colLabels = df.columns,
    cellLoc   = 'center',
    loc       = 'center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.8)

# Style header
for j in range(len(df.columns)):
    table[0, j].set_facecolor('#185FA5')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Style totals row
for j in range(len(df.columns)):
    table[len(df), j].set_facecolor('#E6F1FB')
    table[len(df), j].set_text_props(fontweight='bold')

# Alternate row colors
for i in range(1, len(df)):
    for j in range(len(df.columns)):
        if i % 2 == 0:
            table[i, j].set_facecolor('#F8F8F8')

plt.title(
    'Flood Area (km²) — Vijayawada Sep 2024\n'
    'Improved U-Net | Sentinel-1 SAR | Permanent Water Excluded',
    fontsize=12, fontweight='bold', pad=20
)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/flood_area_table_figure.png",
            dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print("Saved → flood_area_table_figure.png")

In [ ]:
# ============================================================
#  Save Everything to Google Drive — Skip existing files
# ============================================================

from google.colab import drive
import shutil, os, json
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"
os.makedirs(f"{DRIVE_DIR}/model",   exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/outputs", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/logs",    exist_ok=True)

def save_if_missing(src, dst):
    if os.path.exists(dst):
        mb = os.path.getsize(dst)/(1024**2)
        print(f"  SKIP  {os.path.basename(dst)} ({mb:.1f} MB)")
    elif os.path.exists(src):
        shutil.copy(src, dst)
        mb = os.path.getsize(dst)/(1024**2)
        print(f"  SAVED {os.path.basename(dst)} ({mb:.1f} MB)")
    else:
        print(f"  MISS  {os.path.basename(src)} not found")

print("=" * 50)
print("  Saving to Google Drive...")
print("=" * 50)

# Model
print("\nModel:")
save_if_missing(
    '/content/models/best_unet_flood.keras',
    f'{DRIVE_DIR}/model/best_unet_flood.keras')

# All output figures
print("\nFigures:")
import glob
for f in sorted(glob.glob('/content/outputs/*.png')):
    save_if_missing(f, f"{DRIVE_DIR}/outputs/{os.path.basename(f)}")

# Logs and results
print("\nLogs:")
for f in ['training_log.csv',
          'training_summary.json',
          'test_results.json']:
    save_if_missing(
        f'/content/outputs/{f}',
        f'{DRIVE_DIR}/logs/{f}')

# Final summary
print("\n" + "=" * 50)
total = sum(
    os.path.getsize(os.path.join(r,f))
    for r,d,files in os.walk(DRIVE_DIR)
    for f in files
) / (1024**2)
print(f"  Total Drive usage: {total:.1f} MB")
print(f"  Location: {DRIVE_DIR}")
print("=" * 50)
print("\nEverything saved! Have a great day!")

In [ ]:
# ============================================================
#  Save Everything Needed to Rerun Model on New Images
#  Saves: model + all helper functions as a .py script
# ============================================================

from google.colab import drive
import os, shutil
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"
os.makedirs(f"{DRIVE_DIR}/rerun_kit", exist_ok=True)

# ── Save the complete rerun script ───────────────────────
rerun_script = '''
# ============================================================
#  COMPLETE RERUN SCRIPT — Flood Detection on New Images
#  Run this in a fresh Colab session to predict on new images
#  Everything you need is in this one file
# ============================================================

# ── STEP 1: Install ──────────────────────────────────────
!pip install rasterio scikit-image tensorflow -q

# ── STEP 2: Mount Drive ──────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── STEP 3: Imports ──────────────────────────────────────
import tensorflow as tf
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from skimage.transform import resize as sk_resize

# ── STEP 4: Redefine loss functions ──────────────────────
def weighted_dice_bce_loss(y_true, y_pred):
    smooth   = 1e-6
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.reshape(y_pred, [-1])
    bce  = tf.keras.losses.binary_crossentropy(yt, yp)
    inter= tf.reduce_sum(yt * yp)
    dice = 1.0 - (2.0*inter+smooth) / (
           tf.reduce_sum(yt)+tf.reduce_sum(yp)+smooth)
    return 0.5*bce + 0.5*dice

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(
        tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    inter = tf.reduce_sum(yt * yp)
    return (2.0*inter+smooth) / (
           tf.reduce_sum(yt)+tf.reduce_sum(yp)+smooth)

def iou_score(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(
        tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    inter = tf.reduce_sum(yt * yp)
    union = tf.reduce_sum(yt)+tf.reduce_sum(yp)-inter
    return (inter+smooth)/(union+smooth)

# ── STEP 5: Load model ───────────────────────────────────
MODEL_PATH = "/content/drive/MyDrive/FloodMonitoring_Project/model/best_unet_flood.keras"
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "weighted_dice_bce_loss" : weighted_dice_bce_loss,
        "dice_coefficient"       : dice_coefficient,
        "iou_score"              : iou_score,
    }
)
print(f"Model loaded! Input: {model.input_shape}")

# ── STEP 6: Helper functions ─────────────────────────────
def load_s1_image(s1_path):
    with rasterio.open(s1_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)
    valid = np.isfinite(vv) & np.isfinite(vh)
    vv = np.where(valid, vv, 0.0)
    vh = np.where(valid, vh, 0.0)
    return vv, vh, valid

def normalize_bands(vv, vh, valid=None):
    vv_n = np.clip((vv + 30) / 30, 0, 1)
    vh_n = np.clip((vh + 35) / 30, 0, 1)
    if valid is not None:
        vv_n = np.where(valid, vv_n, 0.0)
        vh_n = np.where(valid, vh_n, 0.0)
    return vv_n, vh_n

def predict_flood(s1_path, threshold=0.5):
    """
    Run flood prediction on any Sentinel-1 .tif file.
    Handles any image size automatically.
    """
    PIXEL_AREA_KM2 = 0.0001

    vv, vh, valid = load_s1_image(s1_path)
    vv_n, vh_n    = normalize_bands(vv, vh, valid)
    H, W          = vv_n.shape

    # Resize to 512x512 if needed
    if H != 512 or W != 512:
        vv_r = sk_resize(vv_n, (512,512), preserve_range=True)
        vh_r = sk_resize(vh_n, (512,512), preserve_range=True)
    else:
        vv_r, vh_r = vv_n, vh_n

    # Predict
    img       = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]
    prob      = model.predict(img, verbose=0)[0,...,0]
    mask      = (prob > threshold).astype(np.uint8)

    # Area stats
    valid_px       = int(valid.sum())
    flood_px       = int(mask.sum())
    flood_pct      = 100 * flood_px / max(valid_px, 1)
    flood_area_km2 = flood_px * PIXEL_AREA_KM2

    print(f"\\nFlood Prediction Report")
    print(f"{'='*45}")
    print(f"  File         : {os.path.basename(s1_path)}")
    print(f"  Image size   : {H} x {W}")
    print(f"  Flood pixels : {flood_px:,}")
    print(f"  Flood area   : {flood_area_km2:.4f} km²")
    print(f"  Flood %      : {flood_pct:.2f}%")
    print(f"{'='*45}")

    return {
        "vv"           : vv_r,
        "vh"           : vh_r,
        "prob"         : prob,
        "mask"         : mask,
        "flood_px"     : flood_px,
        "flood_area"   : round(flood_area_km2, 4),
        "flood_pct"    : round(flood_pct, 2),
    }

def visualize_result(result, title="Flood Detection Result"):
    """Show neat visualization of prediction result."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    axes[0].imshow(result["vv"], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("VV Band (SAR Input)")

    axes[1].imshow(result["vh"], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("VH Band (SAR Input)")

    axes[2].imshow(result["prob"], cmap="RdYlBu_r", vmin=0, vmax=1)
    axes[2].set_title("Flood Probability")

    # Flood mask — RED flood, gray land
    disp = np.zeros((*result["mask"].shape, 3))
    disp[result["mask"] == 0] = [0.95, 0.94, 0.92]
    disp[result["mask"] == 1] = [0.85, 0.13, 0.13]
    axes[3].imshow(disp)
    axes[3].text(
        0.02, 0.04,
        f"Flood: {result[\'flood_area\']:.4f} km²\\n({result[\'flood_pct\']:.1f}%)",
        transform=axes[3].transAxes,
        fontsize=9, color="white", fontweight="bold",
        bbox=dict(facecolor="#A32D2D", alpha=0.85, boxstyle="round"),
        va="bottom"
    )
    axes[3].set_title("Flood Mask (red=flood)")

    legend = [
        mpatches.Patch(color=[0.85,0.13,0.13], label="Flood water"),
        mpatches.Patch(color=[0.95,0.94,0.92], label="Land"),
    ]
    fig.legend(handles=legend, loc="lower center",
               bbox_to_anchor=(0.5, -0.05), ncol=2, fontsize=11)

    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()
    return fig

# ── STEP 7: Run on your image ────────────────────────────
# Replace with your actual image path
# result = predict_flood("/content/your_image_S1.tif")
# visualize_result(result, title="Your Flood Image")

print("\\nAll functions ready!")
print("Usage:")
print("  result = predict_flood('/content/your_image.tif')")
print("  visualize_result(result)")
'''

# Save script to Drive
script_path = f"{DRIVE_DIR}/rerun_kit/flood_predict.py"
with open(script_path, 'w') as f:
    f.write(rerun_script)
print(f"  SAVED flood_predict.py")

# Save AOI definitions and locality bounds
localities_script = '''
# ============================================================
#  Vijayawada Locality AOIs and Period Definitions
#  Import this when running on Vijayawada data
# ============================================================

PERIODS = {
    "Pre_Flood"    : ("2024-08-01", "2024-08-31"),
    "During_Flood" : ("2024-09-01", "2024-09-15"),
    "Post_Flood"   : ("2024-09-16", "2024-10-15"),
}

VIJAYAWADA_AOI = [80.50, 16.42, 80.75, 16.62]

LOCALITIES = {
    "Vijayawada City Core" : [80.60, 16.50, 80.68, 16.56],
    "Nunna"                : [80.67, 16.52, 80.72, 16.56],
    "Kundavari Kandrika"   : [80.66, 16.55, 80.72, 16.60],
    "Ambapuram"            : [80.60, 16.56, 80.66, 16.60],
    "Jakkampudi"           : [80.58, 16.56, 80.63, 16.60],
    "Gollapudi"            : [80.52, 16.52, 80.58, 16.56],
    "Undavalli"            : [80.60, 16.46, 80.66, 16.50],
    "Tadepalli"            : [80.60, 16.43, 80.66, 16.47],
    "Ajit Singh Nagar"     : [80.62, 16.50, 80.66, 16.53],
    "Budameru Floodplain"  : [80.58, 16.52, 80.64, 16.57],
}

# SAR file paths (after downloading from GEE)
DRIVE_DIR  = "/content/drive/MyDrive/FloodMonitoring_Project"
TIF_PATHS  = {p: f"{DRIVE_DIR}/vijayawada_sar/vijayawada_{p}_S1.tif"
              for p in PERIODS}
PERM_WATER = f"{DRIVE_DIR}/vijayawada_sar/permanent_water.tif"
OUTPUT_DIR = f"{DRIVE_DIR}/vijayawada_figures"
'''

with open(f"{DRIVE_DIR}/rerun_kit/vijayawada_config.py", 'w') as f:
    f.write(localities_script)
print(f"  SAVED vijayawada_config.py")

# Verify
print(f"\nRerun kit saved to Drive:")
print(f"  {DRIVE_DIR}/rerun_kit/")
for f in os.listdir(f"{DRIVE_DIR}/rerun_kit"):
    size = os.path.getsize(
        f"{DRIVE_DIR}/rerun_kit/{f}")/(1024)
    print(f"  ✓ {f} ({size:.1f} KB)")

print(f"\nNext time — just run flood_predict.py")
print(f"and call predict_flood('your_image.tif')")

In [ ]:
# ── STEP 1: Install ──────────────────────────────────────
!pip install rasterio scikit-image tensorflow -q

In [ ]:
# ── STEP 2: Imports ──────────────────────────────────────
import tensorflow as tf
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from skimage.transform import resize as sk_resize

# ── STEP 3: Redefine loss functions ──────────────────────
def weighted_dice_bce_loss(y_true, y_pred):
    smooth = 1e-6
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.reshape(y_pred, [-1])
    bce  = tf.keras.losses.binary_crossentropy(yt, yp)
    inter = tf.reduce_sum(yt * yp)
    dice = 1.0 - (2.0*inter+smooth) / (
           tf.reduce_sum(yt)+tf.reduce_sum(yp)+smooth)
    return 0.5*bce + 0.5*dice

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(
        tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    inter = tf.reduce_sum(yt * yp)
    return (2.0*inter+smooth) / (
           tf.reduce_sum(yt)+tf.reduce_sum(yp)+smooth)

def iou_score(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(
        tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    inter = tf.reduce_sum(yt * yp)
    union = tf.reduce_sum(yt)+tf.reduce_sum(yp)-inter
    return (inter+smooth)/(union+smooth)

# ── STEP 4: Load model ───────────────────────────────────
MODEL_PATH = "/content/best_unet_flood.keras"  # ✅ Fixed path
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "weighted_dice_bce_loss" : weighted_dice_bce_loss,
        "dice_coefficient"       : dice_coefficient,
        "iou_score"              : iou_score,
    }
)
print(f"Model loaded! Input: {model.input_shape}")

In [ ]:
# ── STEP 5: Helper functions ─────────────────────────────
def load_s1_image(s1_path):
    with rasterio.open(s1_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)
    valid = np.isfinite(vv) & np.isfinite(vh)
    vv = np.where(valid, vv, 0.0)
    vh = np.where(valid, vh, 0.0)
    return vv, vh, valid

def normalize_bands(vv, vh, valid=None):
    vv_n = np.clip((vv + 30) / 30, 0, 1)
    vh_n = np.clip((vh + 35) / 30, 0, 1)
    if valid is not None:
        vv_n = np.where(valid, vv_n, 0.0)
        vh_n = np.where(valid, vh_n, 0.0)
    return vv_n, vh_n

def predict_flood(s1_path, threshold=0.5):
    PIXEL_AREA_KM2 = 0.0001
    vv, vh, valid = load_s1_image(s1_path)
    vv_n, vh_n    = normalize_bands(vv, vh, valid)
    H, W          = vv_n.shape

    if H != 512 or W != 512:
        vv_r = sk_resize(vv_n, (512,512), preserve_range=True)
        vh_r = sk_resize(vh_n, (512,512), preserve_range=True)
    else:
        vv_r, vh_r = vv_n, vh_n

    img  = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]
    prob = model.predict(img, verbose=0)[0,...,0]
    mask = (prob > threshold).astype(np.uint8)

    valid_px       = int(valid.sum())
    flood_px       = int(mask.sum())
    flood_pct      = 100 * flood_px / max(valid_px, 1)
    flood_area_km2 = flood_px * PIXEL_AREA_KM2

    print(f"\nFlood Prediction Report")
    print(f"{'='*45}")
    print(f"  File         : {os.path.basename(s1_path)}")
    print(f"  Image size   : {H} x {W}")
    print(f"  Flood pixels : {flood_px:,}")
    print(f"  Flood area   : {flood_area_km2:.4f} km²")
    print(f"  Flood %      : {flood_pct:.2f}%")
    print(f"{'='*45}")

    return {
        "vv": vv_r, "vh": vh_r, "prob": prob, "mask": mask,
        "flood_px": flood_px,
        "flood_area": round(flood_area_km2, 4),
        "flood_pct": round(flood_pct, 2),
    }

def visualize_result(result, title="Flood Detection Result"):
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    axes[0].imshow(result["vv"], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("VV Band (SAR Input)")

    axes[1].imshow(result["vh"], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("VH Band (SAR Input)")

    axes[2].imshow(result["prob"], cmap="RdYlBu_r", vmin=0, vmax=1)
    axes[2].set_title("Flood Probability")

    disp = np.zeros((*result["mask"].shape, 3))
    disp[result["mask"] == 0] = [0.95, 0.94, 0.92]
    disp[result["mask"] == 1] = [0.85, 0.13, 0.13]
    axes[3].imshow(disp)
    axes[3].text(
        0.02, 0.04,
        f"Flood: {result['flood_area']:.4f} km²\n({result['flood_pct']:.1f}%)",
        transform=axes[3].transAxes,
        fontsize=9, color="white", fontweight="bold",
        bbox=dict(facecolor="#A32D2D", alpha=0.85, boxstyle="round"),
        va="bottom"
    )
    axes[3].set_title("Flood Mask (red=flood)")

    legend = [
        mpatches.Patch(color=[0.85,0.13,0.13], label="Flood water"),
        mpatches.Patch(color=[0.95,0.94,0.92], label="Land"),
    ]
    fig.legend(handles=legend, loc="lower center",
               bbox_to_anchor=(0.5, -0.05), ncol=2, fontsize=11)

    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()
    return fig
# ── STEP 5b: Permanent Water Mask ────────────────────────
import urllib.request

def download_jrc_water_mask(tif_path, save_path="/content/permanent_water_mask.tif"):
    """
    Creates a permanent water mask using a simple threshold approach.
    For production use, replace with JRC Global Surface Water mask.
    """
    with rasterio.open(tif_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)
        profile = src.profile

    # Permanent water bodies have consistently very low backscatter
    # Typical threshold: VV < -18 dB AND VH < -25 dB
    permanent_water = (vv < -18) & (vh < -25)

    # Save mask as tif
    profile.update(count=1, dtype=rasterio.uint8)
    with rasterio.open(save_path, 'w', **profile) as dst:
        dst.write(permanent_water.astype(np.uint8), 1)

    print(f"Permanent water pixels found: {permanent_water.sum():,}")
    return permanent_water

def predict_flood_with_mask(s1_path, threshold=0.5,
                             use_preflood_as_water_ref=True,
                             preflood_path=None):
    """
    Flood prediction that removes permanent water bodies.

    Strategy: pixels flooded in BOTH pre and post = permanent water → exclude
    """
    PIXEL_AREA_KM2 = 0.0001

    # ── Load & normalize both images ──
    vv, vh, valid = load_s1_image(s1_path)
    vv_n, vh_n    = normalize_bands(vv, vh, valid)
    H, W          = vv_n.shape

    if H != 512 or W != 512:
        vv_r = sk_resize(vv_n, (512,512), preserve_range=True)
        vh_r = sk_resize(vh_n, (512,512), preserve_range=True)
    else:
        vv_r, vh_r = vv_n, vh_n

    # ── Predict flood mask ──
    img  = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]
    prob = model.predict(img, verbose=0)[0,...,0]
    mask = (prob > threshold).astype(np.uint8)

    # ── Build permanent water mask from pre-flood image ──
    if preflood_path:
        vv_pre, vh_pre, _ = load_s1_image(preflood_path)
        # Permanent water = very low backscatter in pre-flood image
        # (these were already water BEFORE the flood event)
        perm_water = (vv_pre < -18) & (vh_pre < -25)

        if perm_water.shape != (512, 512):
            perm_water = sk_resize(
                perm_water.astype(np.float32),
                (512, 512), preserve_range=True
            ) > 0.5

        print(f"Permanent water pixels masked: {perm_water.sum():,}")
    else:
        # Fallback: use raw backscatter threshold on current image
        vv_raw, vh_raw, _ = load_s1_image(s1_path)
        perm_water = (vv_raw < -18) & (vh_raw < -25)
        if perm_water.shape != (512, 512):
            perm_water = sk_resize(
                perm_water.astype(np.float32),
                (512, 512), preserve_range=True
            ) > 0.5

    # ── Remove permanent water from flood mask ──
    flood_only_mask = mask.copy()
    flood_only_mask[perm_water] = 0   # ← key line: exclude permanent water

    # ── Stats ──
    valid_px       = int(valid.sum())
    flood_px       = int(flood_only_mask.sum())
    perm_px        = int(perm_water.sum())
    flood_pct      = 100 * flood_px / max(valid_px, 1)
    flood_area_km2 = flood_px * PIXEL_AREA_KM2

    print(f"\nFlood Prediction Report (Permanent Water Removed)")
    print(f"{'='*50}")
    print(f"  File              : {os.path.basename(s1_path)}")
    print(f"  Image size        : {H} x {W}")
    print(f"  Permanent water   : {perm_px:,} px (excluded)")
    print(f"  Flood-only pixels : {flood_px:,}")
    print(f"  Flood area        : {flood_area_km2:.4f} km²")
    print(f"  Flood %           : {flood_pct:.2f}%")
    print(f"{'='*50}")

    return {
        "vv": vv_r, "vh": vh_r, "prob": prob,
        "mask": flood_only_mask,
        "perm_water": perm_water,
        "raw_mask": mask,          # original mask before water removal
        "flood_px": flood_px,
        "flood_area": round(flood_area_km2, 4),
        "flood_pct": round(flood_pct, 2),
    }

def visualize_result_with_water(result, title="Flood Detection Result"):
    """5-panel visualization showing permanent water separately."""
    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    axes[0].imshow(result["vv"], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("VV Band")

    axes[1].imshow(result["vh"], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("VH Band")

    axes[2].imshow(result["prob"], cmap="RdYlBu_r", vmin=0, vmax=1)
    axes[2].set_title("Flood Probability")

    # Raw mask (before water removal)
    disp_raw = np.zeros((*result["raw_mask"].shape, 3))
    disp_raw[result["raw_mask"] == 0] = [0.95, 0.94, 0.92]
    disp_raw[result["raw_mask"] == 1] = [0.85, 0.13, 0.13]
    axes[3].imshow(disp_raw)
    axes[3].set_title("Raw Mask (with perm. water)")

    # Final mask — 3 classes: land, permanent water, flood
    disp = np.zeros((*result["mask"].shape, 3))
    disp[result["mask"] == 0]       = [0.95, 0.94, 0.92]  # land = gray
    disp[result["perm_water"]]      = [0.13, 0.45, 0.85]  # permanent = BLUE
    disp[result["mask"] == 1]       = [0.85, 0.13, 0.13]  # flood = RED
    axes[4].imshow(disp)
    axes[4].text(
        0.02, 0.04,
        f"Flood: {result['flood_area']:.4f} km²\n({result['flood_pct']:.1f}%)",
        transform=axes[4].transAxes,
        fontsize=9, color="white", fontweight="bold",
        bbox=dict(facecolor="#A32D2D", alpha=0.85, boxstyle="round"),
        va="bottom"
    )
    axes[4].set_title("Final: Flood Only (water removed)")

    legend = [
        mpatches.Patch(color=[0.85,0.13,0.13], label="Flood water"),
        mpatches.Patch(color=[0.13,0.45,0.85], label="Permanent water"),
        mpatches.Patch(color=[0.95,0.94,0.92], label="Land"),
    ]
    fig.legend(handles=legend, loc="lower center",
               bbox_to_anchor=(0.5, -0.05), ncol=3, fontsize=11)

    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
# ── STEP 6: Run with permanent water removal ─────────────

PRE  = "/content/preflood_flood_RAW.tif"
POST = "/content/postflood_flood_RAW.tif"

# Pre-flood (use itself as reference — shows baseline water)
result_pre = predict_flood_with_mask(PRE, preflood_path=PRE)
visualize_result_with_water(result_pre, title="Pre-Flood (Permanent Water Removed)")

# Post-flood (use pre-flood as permanent water reference)
result_post = predict_flood_with_mask(POST, preflood_path=PRE)
visualize_result_with_water(result_post, title="Post-Flood (Permanent Water Removed)")

# ── Comparison ───────────────────────────────────────────
print("\n--- Flood Comparison (Permanent Water Excluded) ---")
print(f"Pre-flood  area : {result_pre['flood_area']} km²  ({result_pre['flood_pct']}%)")
print(f"Post-flood area : {result_post['flood_area']} km²  ({result_post['flood_pct']}%)")
print(f"New flood area  : {result_post['flood_area'] - result_pre['flood_area']:.4f} km²")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

# ── Publication style settings ────────────────────────────
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 10,
    'figure.dpi'       : 300,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'savefig.pad_inches': 0.1,
})

SAVE_DIR = "/content/"  # change if needed

# ══════════════════════════════════════════════════════════
# FIGURE 1 — Pre vs Post SAR Input Bands (4-panel)
# ══════════════════════════════════════════════════════════
def plot_figure1(result_pre, result_post):
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    fig.suptitle(
        "Figure 1: Sentinel-1 SAR Input Bands — Pre-Flood vs Post-Flood",
        fontsize=13, fontweight='bold', y=1.02
    )
    panels = [
        (result_pre["vv"],  "Pre-Flood VV Band",  "gray"),
        (result_pre["vh"],  "Pre-Flood VH Band",  "gray"),
        (result_post["vv"], "Post-Flood VV Band", "gray"),
        (result_post["vh"], "Post-Flood VH Band", "gray"),
    ]
    for ax, (data, title, cmap) in zip(axes, panels):
        im = ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
        ax.set_title(title, pad=8)
        ax.axis('off')
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=8)
        cbar.set_label("Normalized backscatter", fontsize=8)

    plt.tight_layout()
    path = SAVE_DIR + "fig1_SAR_bands.png"
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved → {path}")

# ══════════════════════════════════════════════════════════
# FIGURE 2 — Flood Probability Map + Binary Mask (2-panel)
# ══════════════════════════════════════════════════════════
def plot_figure2(result_post):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(
        "Figure 2: U-Net Flood Probability Map and Binary Flood Mask",
        fontsize=13, fontweight='bold', y=1.02
    )

    # Panel A — Probability heatmap
    im = axes[0].imshow(result_post["prob"], cmap="RdYlBu_r", vmin=0, vmax=1)
    axes[0].set_title("(a) Flood Probability Map", pad=8)
    axes[0].axis('off')
    cbar = fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
    cbar.set_label("Flood probability", fontsize=9)
    # Draw threshold line on colorbar
    cbar.ax.axhline(y=0.5, color='black', linewidth=1.5, linestyle='--')
    cbar.ax.text(1.6, 0.5, 'threshold\n(0.5)', va='center',
                 fontsize=7, transform=cbar.ax.transAxes)

    # Panel B — Binary flood mask (3-class)
    disp = np.zeros((*result_post["mask"].shape, 3))
    disp[result_post["mask"] == 0]  = [0.93, 0.93, 0.91]   # land
    if "perm_water" in result_post:
        disp[result_post["perm_water"]] = [0.13, 0.45, 0.85]  # permanent water
    disp[result_post["mask"] == 1]  = [0.85, 0.13, 0.13]   # flood

    axes[1].imshow(disp)
    axes[1].set_title("(b) Classified Flood Map", pad=8)
    axes[1].axis('off')

    # Stats text box
    stats = (f"Flood area: {result_post['flood_area']:.4f} km²\n"
             f"Flood extent: {result_post['flood_pct']:.2f}%\n"
             f"Flood pixels: {result_post['flood_px']:,}")
    axes[1].text(0.02, 0.03, stats,
                 transform=axes[1].transAxes,
                 fontsize=8.5, color='white', fontweight='bold',
                 verticalalignment='bottom',
                 bbox=dict(boxstyle='round,pad=0.4',
                           facecolor='#1a1a1a', alpha=0.75, edgecolor='none'))

    # Legend
    legend_patches = [
        mpatches.Patch(color=[0.85, 0.13, 0.13], label='Flood water'),
        mpatches.Patch(color=[0.13, 0.45, 0.85], label='Permanent water'),
        mpatches.Patch(color=[0.93, 0.93, 0.91], label='Land / no-flood'),
    ]
    axes[1].legend(handles=legend_patches, loc='upper right',
                   fontsize=8, framealpha=0.85, edgecolor='gray')

    plt.tight_layout()
    path = SAVE_DIR + "fig2_flood_map.png"
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved → {path}")

# ══════════════════════════════════════════════════════════
# FIGURE 3 — Permanent Water vs Flood Separation (3-panel)
# ══════════════════════════════════════════════════════════
def plot_figure3(result_pre, result_post):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(
        "Figure 3: Permanent Water Body Removal from Flood Detection",
        fontsize=13, fontweight='bold', y=1.02
    )

    # Panel A — Raw mask (before permanent water removal)
    raw = np.zeros((*result_post["raw_mask"].shape, 3))
    raw[result_post["raw_mask"] == 0] = [0.93, 0.93, 0.91]
    raw[result_post["raw_mask"] == 1] = [0.85, 0.13, 0.13]
    axes[0].imshow(raw)
    axes[0].set_title("(a) Raw Model Output\n(includes permanent water)", pad=8)
    axes[0].axis('off')

    # Panel B — Permanent water mask
    perm_disp = np.zeros((*result_post["perm_water"].shape, 3))
    perm_disp[~result_post["perm_water"]] = [0.93, 0.93, 0.91]
    perm_disp[ result_post["perm_water"]] = [0.13, 0.45, 0.85]
    axes[1].imshow(perm_disp)
    axes[1].set_title("(b) Permanent Water Mask\n(JRC-derived threshold)", pad=8)
    axes[1].axis('off')

    # Panel C — Final: flood only
    final = np.zeros((*result_post["mask"].shape, 3))
    final[result_post["mask"] == 0]   = [0.93, 0.93, 0.91]
    final[result_post["perm_water"]]  = [0.13, 0.45, 0.85]
    final[result_post["mask"] == 1]   = [0.85, 0.13, 0.13]
    axes[2].imshow(final)
    axes[2].set_title("(c) Final Flood Map\n(permanent water removed)", pad=8)
    axes[2].axis('off')

    legend_patches = [
        mpatches.Patch(color=[0.85, 0.13, 0.13], label='Flood water'),
        mpatches.Patch(color=[0.13, 0.45, 0.85], label='Permanent water body'),
        mpatches.Patch(color=[0.93, 0.93, 0.91], label='Land'),
    ]
    fig.legend(handles=legend_patches, loc='lower center',
               bbox_to_anchor=(0.5, -0.08), ncol=3, fontsize=10,
               framealpha=0.9, edgecolor='gray')

    plt.tight_layout()
    path = SAVE_DIR + "fig3_water_separation.png"
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved → {path}")

# ══════════════════════════════════════════════════════════
# FIGURE 4 — Metrics Bar Chart
# ══════════════════════════════════════════════════════════
def plot_figure4(result_pre, result_post):
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle(
        "Figure 4: Flood Detection Metrics — Pre-Flood vs Post-Flood",
        fontsize=13, fontweight='bold', y=1.02
    )

    labels   = ['Pre-Flood', 'Post-Flood']
    colors   = ['#2196F3', '#E53935']
    new_flood = round(result_post['flood_area'] - result_pre['flood_area'], 4)

    # Subplot 1 — Flood area km²
    vals1 = [result_pre['flood_area'], result_post['flood_area']]
    bars = axes[0].bar(labels, vals1, color=colors, width=0.5, edgecolor='white')
    axes[0].set_title("Flood Area (km²)", pad=8)
    axes[0].set_ylabel("Area (km²)")
    axes[0].spines[['top','right']].set_visible(False)
    for bar, val in zip(bars, vals1):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.0005,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=9)

    # Subplot 2 — Flood %
    vals2 = [result_pre['flood_pct'], result_post['flood_pct']]
    bars2 = axes[1].bar(labels, vals2, color=colors, width=0.5, edgecolor='white')
    axes[1].set_title("Flood Extent (%)", pad=8)
    axes[1].set_ylabel("Percentage (%)")
    axes[1].spines[['top','right']].set_visible(False)
    for bar, val in zip(bars2, vals2):
        axes[1].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.05,
                     f'{val:.2f}%', ha='center', va='bottom', fontsize=9)

    # Subplot 3 — New flood area (difference)
    axes[2].bar(['New flood area'], [new_flood],
                color='#FF7043', width=0.4, edgecolor='white')
    axes[2].set_title("New Inundation (km²)\nPost − Pre", pad=8)
    axes[2].set_ylabel("Area (km²)")
    axes[2].spines[['top','right']].set_visible(False)
    axes[2].text(0, new_flood + 0.0005,
                 f'{new_flood:.4f} km²',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    path = SAVE_DIR + "fig4_metrics_chart.png"
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved → {path}")

# ══════════════════════════════════════════════════════════
# RUN ALL FIGURES
# ══════════════════════════════════════════════════════════
plot_figure1(result_pre, result_post)
plot_figure2(result_post)
plot_figure3(result_pre, result_post)
plot_figure4(result_pre, result_post)

print("\n✅ All 4 figures saved to /content/")
print("  fig1_SAR_bands.png")
print("  fig2_flood_map.png")
print("  fig3_water_separation.png")
print("  fig4_metrics_chart.png")

In [ ]:
# ── Fix font without restarting ──────────────────────────
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# Force matplotlib to rebuild font cache in current session
fm._load_fontmanager(try_read_cache=False)

# Check available serif fonts
available = sorted(set([f.name for f in fm.fontManager.ttflist]))
serif_candidates = ['Times New Roman', 'DejaVu Serif',
                    'Georgia', 'Palatino', 'FreeSerif',
                    'Liberation Serif', 'Nimbus Roman']

chosen_font = None
for font in serif_candidates:
    if font in available:
        chosen_font = font
        print(f"✅ Using font: {font}")
        break

if not chosen_font:
    chosen_font = 'DejaVu Serif'  # guaranteed fallback in Colab
    print(f"⚠️  Using fallback: {chosen_font}")

# Apply globally — affects all subsequent plt calls
plt.rcParams.update({
    'font.family' : 'serif',
    'font.serif'  : [chosen_font, 'DejaVu Serif', 'serif'],
})

print("Font settings applied — re-run your figure functions now.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Apply the correct font — no restart needed
plt.rcParams.update({
    'font.family'       : 'serif',
    'font.serif'        : ['STIXGeneral', 'Liberation Serif',
                           'DejaVu Serif', 'serif'],
    'font.size'         : 10,
    'axes.titlesize'    : 10,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 9,
    'xtick.labelsize'   : 8,
    'ytick.labelsize'   : 8,
    'legend.fontsize'   : 8,
    'figure.dpi'        : 300,
    'savefig.dpi'       : 300,
    'savefig.bbox'      : 'tight',
    'savefig.pad_inches': 0.05,
    'axes.linewidth'    : 0.8,
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
})

# Verify
test = fm.findfont(fm.FontProperties(family='STIXGeneral'))
print(f"✅ Font resolved to: {test}")
print("Now re-run save_figure1, save_figure2, save_figure3")

In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 1 — Set font ONCE (no Times New Roman, no warnings)
# ══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import numpy as np

plt.rcParams.update({
    'font.family'       : 'STIXGeneral',   # ✅ available in Colab, no warning
    'font.size'         : 10,
    'axes.titlesize'    : 10,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 9,
    'xtick.labelsize'   : 8,
    'ytick.labelsize'   : 8,
    'legend.fontsize'   : 8,
    'figure.dpi'        : 300,
    'savefig.dpi'       : 300,
    'savefig.bbox'      : 'tight',
    'savefig.pad_inches': 0.05,
    'axes.linewidth'    : 0.8,
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
})

SAVE_DIR  = "/content/"
SAVE_OPTS = dict(dpi=300, bbox_inches='tight',
                 facecolor='white', edgecolor='none')

print("✅ Font set to STIXGeneral — no warnings")

# ══════════════════════════════════════════════════════════
# FIGURE 1 — Pre vs Post SAR Bands (4 panels)
# ══════════════════════════════════════════════════════════
def save_figure1(result_pre, result_post):

    panels = [
        (result_pre["vv"],  "(a) Pre-flood VV band"),
        (result_pre["vh"],  "(b) Pre-flood VH band"),
        (result_post["vv"], "(c) Post-flood VV band"),
        (result_post["vh"], "(d) Post-flood VH band"),
    ]

    # ── Combined 4-panel ──────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    fig.suptitle(
        "Sentinel-1 SAR Backscatter: Pre-Flood vs. Post-Flood",
        fontsize=11, fontweight='bold', y=1.01
    )
    for ax, (data, title) in zip(axes, panels):
        im = ax.imshow(data, cmap='gray', vmin=0, vmax=1,
                       interpolation='bilinear')
        ax.set_title(title, pad=5, loc='left', fontsize=9)
        ax.axis('off')
        cbar = fig.colorbar(im, ax=ax, fraction=0.046,
                            pad=0.03, shrink=0.95)
        cbar.ax.tick_params(labelsize=7)
        cbar.set_label("Normalised backscatter", fontsize=7, labelpad=4)

    plt.tight_layout(pad=0.8)
    fig.savefig(SAVE_DIR + "fig1_SAR_comparison.png", **SAVE_OPTS)
    plt.show()
    print("✅ Saved: fig1_SAR_comparison.png")

    # ── Individual panels ─────────────────────────────────
    names = ['fig1a_pre_VV', 'fig1b_pre_VH',
             'fig1c_post_VV', 'fig1d_post_VH']
    for (data, title), name in zip(panels, names):
        fig_s, ax_s = plt.subplots(figsize=(4, 4))
        im = ax_s.imshow(data, cmap='gray', vmin=0, vmax=1,
                         interpolation='bilinear')
        ax_s.set_title(title, pad=5, loc='left', fontsize=9)
        ax_s.axis('off')
        cbar = fig_s.colorbar(im, ax=ax_s, fraction=0.046,
                              pad=0.03, shrink=0.95)
        cbar.ax.tick_params(labelsize=7)
        cbar.set_label("Normalised backscatter", fontsize=7)
        fig_s.savefig(SAVE_DIR + name + ".png", **SAVE_OPTS)
        plt.close(fig_s)
        print(f"   Saved: {name}.png")

# ══════════════════════════════════════════════════════════
# FIGURE 2 — Flood Extent Map (3 panels, NO permanent water)
# ══════════════════════════════════════════════════════════
def save_figure2(result_pre, result_post):

    def make_mask(result):
        """2-class RGB: flood=red, land=gray. No permanent water."""
        disp = np.ones((*result["mask"].shape, 3))
        disp[result["mask"] == 0] = [0.92, 0.92, 0.92]  # land
        disp[result["mask"] == 1] = [0.84, 0.10, 0.10]  # flood
        return disp

    prob_map  = result_post["prob"]
    pre_mask  = make_mask(result_pre)
    post_mask = make_mask(result_post)

    legend_patches = [
        mpatches.Patch(color=[0.84, 0.10, 0.10], label='Flood water'),
        mpatches.Patch(color=[0.92, 0.92, 0.92], label='Non-flooded land'),
    ]

    panels = [
        (prob_map,  "(a) Flood probability map",  "RdYlBu_r", True),
        (pre_mask,  "(b) Pre-flood extent",        None,        False),
        (post_mask, "(c) Post-flood extent",       None,        False),
    ]

    # ── Combined 3-panel ──────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    fig.suptitle(
        "Flood Extent Mapping: Probability and Binary Classification",
        fontsize=11, fontweight='bold', y=1.01
    )
    for ax, (data, title, cmap, has_cbar) in zip(axes, panels):
        if has_cbar:
            im = ax.imshow(data, cmap=cmap, vmin=0, vmax=1,
                           interpolation='bilinear')
            cbar = fig.colorbar(im, ax=ax, fraction=0.046,
                                pad=0.03, shrink=0.95)
            cbar.ax.tick_params(labelsize=7)
            cbar.set_label("Flood probability", fontsize=7, labelpad=4)
            cbar.ax.axhline(y=0.5, color='black',
                            linewidth=1.2, linestyle='--')
            cbar.ax.text(1.7, 0.5, 'threshold\n(0.5)',
                         va='center', fontsize=6,
                         transform=cbar.ax.transAxes)
        else:
            ax.imshow(data, interpolation='bilinear')
            ax.legend(handles=legend_patches, loc='lower right',
                      fontsize=7, frameon=True,
                      edgecolor='#cccccc', fancybox=False)

        ax.set_title(title, pad=5, loc='left', fontsize=9)
        ax.axis('off')

        # Stats overlay on post-flood panel
        if title == "(c) Post-flood extent":
            stats = (f"Flood area : {result_post['flood_area']:.4f} km²\n"
                     f"Flood extent: {result_post['flood_pct']:.2f}%\n"
                     f"Flood pixels: {result_post['flood_px']:,}")
            ax.text(0.02, 0.03, stats,
                    transform=ax.transAxes,
                    fontsize=7.5, color='white', fontweight='bold',
                    verticalalignment='bottom',
                    bbox=dict(boxstyle='round,pad=0.4',
                              facecolor='#1a1a1a',
                              alpha=0.72, edgecolor='none'))

    plt.tight_layout(pad=0.8)
    fig.savefig(SAVE_DIR + "fig2_flood_extent_map.png", **SAVE_OPTS)
    plt.show()
    print("✅ Saved: fig2_flood_extent_map.png")

    # ── Individual panels ─────────────────────────────────
    ind = [
        (prob_map,  "fig2a_probability",  "RdYlBu_r", True),
        (pre_mask,  "fig2b_pre_extent",   None,        False),
        (post_mask, "fig2c_post_extent",  None,        False),
    ]
    titles_ind = ["(a) Flood probability map",
                  "(b) Pre-flood extent",
                  "(c) Post-flood extent"]

    for (data, name, cmap, has_cbar), title in zip(ind, titles_ind):
        fig_s, ax_s = plt.subplots(figsize=(4.5, 4.5))
        if has_cbar:
            im = ax_s.imshow(data, cmap=cmap, vmin=0, vmax=1,
                             interpolation='bilinear')
            cbar = fig_s.colorbar(im, ax=ax_s, fraction=0.046,
                                  pad=0.03, shrink=0.95)
            cbar.ax.tick_params(labelsize=7)
            cbar.set_label("Flood probability", fontsize=7)
            cbar.ax.axhline(y=0.5, color='black',
                            linewidth=1.2, linestyle='--')
        else:
            ax_s.imshow(data, interpolation='bilinear')
            ax_s.legend(handles=legend_patches, loc='lower right',
                        fontsize=7, frameon=True,
                        edgecolor='#cccccc', fancybox=False)
        ax_s.set_title(title, pad=5, loc='left', fontsize=9)
        ax_s.axis('off')
        fig_s.savefig(SAVE_DIR + name + ".png", **SAVE_OPTS)
        plt.close(fig_s)
        print(f"   Saved: {name}.png")

# ══════════════════════════════════════════════════════════
# FIGURE 3 — Metrics Bar Chart (3 panels)
# ══════════════════════════════════════════════════════════
def save_figure3(result_pre, result_post):

    COLORS = ['#2171B5', '#CB181D']   # blue=pre, red=post
    labels = ['Pre-Flood', 'Post-Flood']

    metrics = [
        ("(a) Flood area",    "Area (km²)",
         result_pre['flood_area'], result_post['flood_area'], "km²", 4),
        ("(b) Flood extent",  "Extent (%)",
         result_pre['flood_pct'],  result_post['flood_pct'],  "%",   2),
        ("(c) Flood pixels",  "Pixel count",
         result_pre['flood_px'],   result_post['flood_px'],   "px",  0),
    ]

    # ── Combined 3-panel ──────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))
    fig.suptitle(
        "Flood Detection Metrics: Pre-Flood vs. Post-Flood Comparison",
        fontsize=11, fontweight='bold', y=1.01
    )
    for ax, (title, ylabel, pre_v, post_v, unit, dp) in zip(axes, metrics):
        fmt  = f".{dp}f"
        bars = ax.bar(labels, [pre_v, post_v],
                      color=COLORS, width=0.45,
                      edgecolor='black', linewidth=0.6)
        ax.set_title(title, pad=5, loc='left')
        ax.set_ylabel(ylabel)
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['left', 'bottom']].set_linewidth(0.8)
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.tick_params(axis='both', which='major', length=3)

        for bar, val in zip(bars, [pre_v, post_v]):
            offset = max(pre_v, post_v) * 0.02
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + offset,
                    f"{val:{fmt}}", ha='center',
                    va='bottom', fontsize=8.5, fontweight='bold')

        delta = post_v - pre_v
        ax.annotate(
            f"Δ = {delta:{fmt}} {unit}",
            xy=(0.97, 0.96), xycoords='axes fraction',
            ha='right', va='top', fontsize=7.5,
            bbox=dict(boxstyle='round,pad=0.3',
                      fc='#f5f5f5', ec='#cccccc', lw=0.7)
        )

    plt.tight_layout(pad=1.0)
    fig.savefig(SAVE_DIR + "fig3_metrics_chart.png", **SAVE_OPTS)
    plt.show()
    print("✅ Saved: fig3_metrics_chart.png")

    # ── Individual panels ─────────────────────────────────
    names = ['fig3a_flood_area', 'fig3b_flood_extent', 'fig3c_flood_pixels']
    for (title, ylabel, pre_v, post_v, unit, dp), name in zip(metrics, names):
        fmt  = f".{dp}f"
        fig_s, ax_s = plt.subplots(figsize=(4, 4))
        bars = ax_s.bar(labels, [pre_v, post_v],
                        color=COLORS, width=0.45,
                        edgecolor='black', linewidth=0.6)
        ax_s.set_title(title, pad=5, loc='left', fontsize=9)
        ax_s.set_ylabel(ylabel)
        ax_s.spines[['top', 'right']].set_visible(False)
        ax_s.spines[['left', 'bottom']].set_linewidth(0.8)
        ax_s.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        for bar, val in zip(bars, [pre_v, post_v]):
            offset = max(pre_v, post_v) * 0.02
            ax_s.text(bar.get_x() + bar.get_width() / 2,
                      bar.get_height() + offset,
                      f"{val:{fmt}}", ha='center',
                      va='bottom', fontsize=9, fontweight='bold')
        delta = post_v - pre_v
        ax_s.annotate(
            f"Δ = {delta:{fmt}} {unit}",
            xy=(0.97, 0.96), xycoords='axes fraction',
            ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3',
                      fc='#f5f5f5', ec='#cccccc', lw=0.7)
        )
        fig_s.tight_layout()
        fig_s.savefig(SAVE_DIR + name + ".png", **SAVE_OPTS)
        plt.close(fig_s)
        print(f"   Saved: {name}.png")

# ══════════════════════════════════════════════════════════
# RUN ALL 3 FIGURES
# ══════════════════════════════════════════════════════════
save_figure1(result_pre, result_post)
save_figure2(result_pre, result_post)
save_figure3(result_pre, result_post)

print("\n✅ All done! Files saved to /content/")
print("\nCombined:"); print("  fig1_SAR_comparison.png")
print("  fig2_flood_extent_map.png"); print("  fig3_metrics_chart.png")
print("\nIndividual panels:")
print("  fig1a–fig1d | fig2a–fig2c | fig3a–fig3c")

In [ ]:
# ============================================================
#  RELOAD CELL — Run this first in every new session
#  Reloads all paths from Drive — no re-downloading needed
# ============================================================

from google.colab import drive
import os
drive.mount('/content/drive')

# ── All Drive paths ───────────────────────────────────────
DRIVE_DIR  = "/content/drive/MyDrive/FloodMonitoring_Project"
DRIVE_SAR  = f"{DRIVE_DIR}/vijayawada_sar"
OUTPUT_DIR = f"{DRIVE_DIR}/vijayawada_figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Period definitions ────────────────────────────────────
PERIODS = {
    "Pre_Flood"    : ("2024-08-01", "2024-08-31"),
    "During_Flood" : ("2024-09-01", "2024-09-15"),
    "Post_Flood"   : ("2024-09-16", "2024-10-15"),
}

# ── TIF file paths (already in Drive) ────────────────────
TIF_PATHS = {
    "Pre_Flood"    : f"{DRIVE_SAR}/vijayawada_Pre_Flood_S1.tif",
    "During_Flood" : f"{DRIVE_SAR}/vijayawada_During_Flood_S1.tif",
    "Post_Flood"   : f"{DRIVE_SAR}/vijayawada_Post_Flood_S1.tif",
}
PERM_WATER_PATH = f"{DRIVE_SAR}/permanent_water.tif"

# ── 3 selected localities ─────────────────────────────────
SELECTED = {
    "Budameru Floodplain"  : [80.58, 16.52, 80.64, 16.57],
    "Vijayawada City Core" : [80.60, 16.50, 80.68, 16.56],
    "Nunna"                : [80.67, 16.52, 80.72, 16.56],
}

# ── Verify all files exist ────────────────────────────────
print("Checking Drive files...")
print("=" * 50)
all_ok = True
for name, path in {**TIF_PATHS,
                   "Permanent_Water": PERM_WATER_PATH}.items():
    exists = os.path.exists(path)
    mb     = os.path.getsize(path)/(1024**2) if exists else 0
    icon   = "✓" if exists and mb > 0.01 else "✗"
    print(f"  {icon} {name:<20}: {mb:.1f} MB")
    if not exists or mb < 0.01:
        all_ok = False

print("=" * 50)
if all_ok:
    print("  All files ready — TIF_PATHS defined!")
    print("  Now run the VV/VH visualization code!")
else:
    print("  Some files missing — need to re-download")

In [ ]:
# ============================================================
#  Clean VV & VH Images — No text on images
#  Area names and period labels as titles only
#  Maximum image clarity
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.windows import from_bounds
from skimage.transform import resize as sk_resize
import warnings
warnings.filterwarnings('ignore')  # suppress rasterio warnings

SELECTED = {
    "Budameru Floodplain"  : [80.58, 16.52, 80.64, 16.57],
    "Vijayawada City Core" : [80.60, 16.50, 80.68, 16.56],
    "Nunna"                : [80.67, 16.52, 80.72, 16.56],
}

PERIOD_LABELS = {
    "Pre_Flood"    : "Pre-Flood\n(Aug 2024)",
    "During_Flood" : "During Flood\n(Sep 1–15, 2024)",
    "Post_Flood"   : "Post-Flood\n(Sep 16–Oct 2024)",
}

def load_band(tif_path, bounds, band=1, size=512):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with rasterio.open(tif_path) as src:
            win = from_bounds(
                bounds[0], bounds[1],
                bounds[2], bounds[3],
                src.transform)
            arr = src.read(band, window=win).astype(np.float32)
    arr = np.where(np.isfinite(arr), arr, np.nan)
    return sk_resize(
        np.nan_to_num(arr, nan=0.0),
        (size, size),
        preserve_range=True,
        anti_aliasing=True)

def enhance(arr, lo=2, hi=98):
    """Percentile stretch for best visual clarity."""
    valid = arr[arr != 0]
    if len(valid) == 0:
        return arr
    p_lo = np.percentile(valid, lo)
    p_hi = np.percentile(valid, hi)
    stretched = np.clip((arr - p_lo) / (p_hi - p_lo + 1e-9), 0, 1)
    return stretched

# ── Figure 1: VV — 3 localities × 3 periods ──────────────
print("Generating VV band figure...")
fig, axes = plt.subplots(
    3, 3, figsize=(20, 20),
    gridspec_kw={'hspace': 0.05, 'wspace': 0.04}
)
fig.suptitle(
    'VV Polarisation — Vijayawada Sep 2024\nSentinel-1 SAR',
    fontsize=18, fontweight='bold', y=1.01
)

for col, (period, plabel) in enumerate(PERIOD_LABELS.items()):
    axes[0, col].set_title(
        plabel, fontsize=13, fontweight='bold', pad=12)

for row, (name, bounds) in enumerate(SELECTED.items()):
    axes[row, 0].set_ylabel(
        name, fontsize=12, fontweight='bold', labelpad=12)
    for col, (period, _) in enumerate(PERIOD_LABELS.items()):
        vv  = load_band(TIF_PATHS[period], bounds, band=1)
        vv_e = enhance(vv)
        axes[row, col].imshow(
            vv_e, cmap='gray', vmin=0, vmax=1,
            interpolation='lanczos')
        axes[row, col].axis('off')

plt.savefig(
    f"{OUTPUT_DIR}/fig_VV_clean.png",
    dpi=300, bbox_inches='tight',
    facecolor='white')
plt.show()
plt.close()
print("Saved → fig_VV_clean.png")

# ── Figure 2: VH — 3 localities × 3 periods ──────────────
print("Generating VH band figure...")
fig2, axes2 = plt.subplots(
    3, 3, figsize=(20, 20),
    gridspec_kw={'hspace': 0.05, 'wspace': 0.04}
)
fig2.suptitle(
    'VH Polarisation — Vijayawada Sep 2024\nSentinel-1 SAR',
    fontsize=18, fontweight='bold', y=1.01
)

for col, (period, plabel) in enumerate(PERIOD_LABELS.items()):
    axes2[0, col].set_title(
        plabel, fontsize=13, fontweight='bold', pad=12)

for row, (name, bounds) in enumerate(SELECTED.items()):
    axes2[row, 0].set_ylabel(
        name, fontsize=12, fontweight='bold', labelpad=12)
    for col, (period, _) in enumerate(PERIOD_LABELS.items()):
        vh   = load_band(TIF_PATHS[period], bounds, band=2)
        vh_e = enhance(vh)
        axes2[row, col].imshow(
            vh_e, cmap='gray', vmin=0, vmax=1,
            interpolation='lanczos')
        axes2[row, col].axis('off')

plt.savefig(
    f"{OUTPUT_DIR}/fig_VH_clean.png",
    dpi=300, bbox_inches='tight',
    facecolor='white')
plt.show()
plt.close()
print("Saved → fig_VH_clean.png")

# ── Figure 3: VV + VH per locality ───────────────────────
print("Generating per-locality figures...")
for name, bounds in SELECTED.items():
    fig3, axes3 = plt.subplots(
        3, 2, figsize=(14, 18),
        gridspec_kw={'hspace': 0.05, 'wspace': 0.04}
    )
    fig3.suptitle(
        f'{name}\nSentinel-1 SAR — Vijayawada Sep 2024',
        fontsize=15, fontweight='bold', y=1.01
    )
    axes3[0, 0].set_title(
        'VV Polarisation', fontsize=13, fontweight='bold', pad=12)
    axes3[0, 1].set_title(
        'VH Polarisation', fontsize=13, fontweight='bold', pad=12)

    for row, (period, plabel) in enumerate(PERIOD_LABELS.items()):
        vv   = load_band(TIF_PATHS[period], bounds, band=1)
        vh   = load_band(TIF_PATHS[period], bounds, band=2)
        vv_e = enhance(vv)
        vh_e = enhance(vh)

        axes3[row, 0].imshow(
            vv_e, cmap='gray', vmin=0, vmax=1,
            interpolation='lanczos')
        axes3[row, 1].imshow(
            vh_e, cmap='gray', vmin=0, vmax=1,
            interpolation='lanczos')

        axes3[row, 0].set_ylabel(
            plabel, fontsize=11,
            fontweight='bold', labelpad=10)

        axes3[row, 0].axis('off')
        axes3[row, 1].axis('off')

    plt.savefig(
        f"{OUTPUT_DIR}/fig_VVVH_{name.replace(' ','_')}_clean.png",
        dpi=300, bbox_inches='tight',
        facecolor='white')
    plt.show()
    plt.close()
    print(f"  Saved → fig_VVVH_{name.replace(' ','_')}_clean.png")

print("\nAll clean figures saved!")
print("  fig_VV_clean.png")
print("  fig_VH_clean.png")
print("  fig_VVVH_Budameru_Floodplain_clean.png")
print("  fig_VVVH_Vijayawada_City_Core_clean.png")
print("  fig_VVVH_Nunna_clean.png")

In [ ]:
# ============================================================
#  Flood Probability Maps — Fixed with correct colormap
#  Purple→Cyan→Yellow style (like your reference image)
#  No axis ticks, no text on image, clean output
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds
from skimage.transform import resize as sk_resize
import warnings
warnings.filterwarnings('ignore')

SELECTED = {
    "Budameru Floodplain"  : [80.58, 16.52, 80.64, 16.57],
    "Vijayawada City Core" : [80.60, 16.50, 80.68, 16.56],
    "Nunna"                : [80.67, 16.52, 80.72, 16.56],
}

PERIOD_LABELS = {
    "Pre_Flood"    : "Pre-Flood\n(Aug 2024)",
    "During_Flood" : "During Flood\n(Sep 1–15, 2024)",
    "Post_Flood"   : "Post-Flood\n(Sep 16–Oct 2024)",
}

def load_band(tif_path, bounds, band=1, size=512):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with rasterio.open(tif_path) as src:
            win = from_bounds(
                bounds[0], bounds[1],
                bounds[2], bounds[3],
                src.transform)
            arr = src.read(band, window=win).astype(np.float32)
    arr = np.where(np.isfinite(arr), arr, 0.0)
    return sk_resize(arr, (size, size),
                     preserve_range=True,
                     anti_aliasing=True)

def get_perm_water(bounds, size=512):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with rasterio.open(PERM_WATER_PATH) as src:
            win = from_bounds(
                bounds[0], bounds[1],
                bounds[2], bounds[3],
                src.transform)
            perm = src.read(1, window=win).astype(np.float32)
    return sk_resize(perm, (size, size),
                     preserve_range=True,
                     anti_aliasing=False) > 0.5

def predict_prob(bounds, period):
    vv   = load_band(TIF_PATHS[period], bounds, band=1)
    vh   = load_band(TIF_PATHS[period], bounds, band=2)
    vv_n = np.clip((vv + 30) / 30, 0, 1)
    vh_n = np.clip((vh + 35) / 30, 0, 1)
    img  = np.stack([vv_n, vh_n], axis=-1)[np.newaxis]
    prob = model.predict(img, verbose=0)[0, ..., 0]

    # Remove permanent water
    perm = get_perm_water(bounds)
    prob[perm] = 0.0

    return prob

# ── Figure 1: Combined 3×3 overview ──────────────────────
print("Generating combined probability map...")

fig, axes = plt.subplots(
    3, 3, figsize=(20, 20),
    gridspec_kw={'hspace': 0.06, 'wspace': 0.04}
)
fig.suptitle(
    'Flood Probability Maps — Vijayawada Sep 2024\n'
    'Improved U-Net | Sentinel-1 SAR | Permanent Water Removed',
    fontsize=18, fontweight='bold', y=1.01
)

# Column titles
for col, (_, plabel) in enumerate(PERIOD_LABELS.items()):
    axes[0, col].set_title(
        plabel, fontsize=13, fontweight='bold', pad=12)

# Row labels + images
for row, (name, bounds) in enumerate(SELECTED.items()):
    axes[row, 0].set_ylabel(
        name, fontsize=12, fontweight='bold', labelpad=12)

    for col, (period, _) in enumerate(PERIOD_LABELS.items()):
        prob = predict_prob(bounds, period)

        axes[row, col].imshow(
            prob,
            cmap='plasma',        # ← matches your reference image
            vmin=0, vmax=1,
            interpolation='lanczos'
        )
        # Remove ALL axis ticks and labels
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        axes[row, col].axis('off')

# Shared colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(
    cmap='plasma',
    norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cb = plt.colorbar(sm, cax=cbar_ax)
cb.set_label('Flood Probability', fontsize=12)
cb.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cb.set_ticklabels(['0%', '25%', '50%', '75%', '100%'])

plt.savefig(
    f"{OUTPUT_DIR}/fig_flood_prob_combined.png",
    dpi=300, bbox_inches='tight',
    facecolor='white')
plt.show()
plt.close()
print("Saved → fig_flood_prob_combined.png")

# ── Figure 2: One per locality ────────────────────────────
print("\nGenerating per-locality maps...")

for name, bounds in SELECTED.items():
    fig2, axes2 = plt.subplots(
        1, 3, figsize=(21, 7),
        gridspec_kw={'wspace': 0.04}
    )
    fig2.suptitle(
        f'Flood Probability — {name}\n'
        'Improved U-Net | Sentinel-1 SAR | Permanent Water Removed',
        fontsize=14, fontweight='bold', y=1.02
    )

    for col, (period, plabel) in enumerate(PERIOD_LABELS.items()):
        prob = predict_prob(bounds, period)
        im2  = axes2[col].imshow(
            prob,
            cmap='plasma',
            vmin=0, vmax=1,
            interpolation='lanczos'
        )
        axes2[col].set_title(
            plabel, fontsize=12,
            fontweight='bold', pad=10)
        axes2[col].set_xticks([])
        axes2[col].set_yticks([])
        axes2[col].axis('off')

    # Colorbar
    cbar_ax2 = fig2.add_axes([0.92, 0.15, 0.015, 0.7])
    sm2 = plt.cm.ScalarMappable(
        cmap='plasma',
        norm=plt.Normalize(vmin=0, vmax=1))
    sm2.set_array([])
    cb2 = plt.colorbar(sm2, cax=cbar_ax2)
    cb2.set_label('Flood Probability', fontsize=11)
    cb2.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
    cb2.set_ticklabels(['0%', '25%', '50%', '75%', '100%'])

    fname = name.replace(' ', '_')
    plt.savefig(
        f"{OUTPUT_DIR}/fig_prob_{fname}.png",
        dpi=300, bbox_inches='tight',
        facecolor='white')
    plt.show()
    plt.close()
    print(f"  Saved → fig_prob_{fname}.png")

print("\nAll probability maps saved!")
print("  fig_flood_prob_combined.png")
print("  fig_prob_Budameru_Floodplain.png")
print("  fig_prob_Vijayawada_City_Core.png")
print("  fig_prob_Nunna.png")

In [ ]:
# ============================================================
#  Binary Flood Classification Maps
#  Black = Non-flood | White = Flood
#  3 localities × 3 periods | No text on image
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds
from skimage.transform import resize as sk_resize
import warnings
warnings.filterwarnings('ignore')

def predict_binary(bounds, period, threshold=0.5):
    """Predict binary flood mask — 1=flood, 0=non-flood."""
    vv   = load_band(TIF_PATHS[period], bounds, band=1)
    vh   = load_band(TIF_PATHS[period], bounds, band=2)
    vv_n = np.clip((vv + 30) / 30, 0, 1)
    vh_n = np.clip((vh + 35) / 30, 0, 1)
    img  = np.stack([vv_n, vh_n], axis=-1)[np.newaxis]
    prob = model.predict(img, verbose=0)[0, ..., 0]

    # Binary threshold
    mask = (prob > threshold).astype(np.float32)

    # Remove permanent water
    perm = get_perm_water(bounds)
    mask[perm] = 0.0

    return mask

# ── Figure 1: Combined 3×3 overview ──────────────────────
print("Generating binary classification maps...")

fig, axes = plt.subplots(
    3, 3, figsize=(20, 20),
    gridspec_kw={'hspace': 0.06, 'wspace': 0.04}
)
fig.suptitle(
    'Flood Binary Classification — Vijayawada Sep 2024\n'
    'Improved U-Net | Sentinel-1 SAR | Permanent Water Removed',
    fontsize=18, fontweight='bold', y=1.01
)

# Column titles
for col, (_, plabel) in enumerate(PERIOD_LABELS.items()):
    axes[0, col].set_title(
        plabel, fontsize=13, fontweight='bold', pad=12)

# Row labels + images
for row, (name, bounds) in enumerate(SELECTED.items()):
    axes[row, 0].set_ylabel(
        name, fontsize=12, fontweight='bold', labelpad=12)

    for col, (period, _) in enumerate(PERIOD_LABELS.items()):
        mask = predict_binary(bounds, period)

        axes[row, col].imshow(
            mask,
            cmap='gray',      # black=0=non-flood, white=1=flood
            vmin=0, vmax=1,
            interpolation='nearest'  # sharp edges, no blur
        )
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        axes[row, col].axis('off')

# Legend
from matplotlib.patches import Patch
legend = [
    Patch(facecolor='white', edgecolor='black', label='Flood'),
    Patch(facecolor='black', edgecolor='white', label='Non-flood'),
]
fig.legend(
    handles=legend,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.02),
    ncol=2, fontsize=13,
    frameon=True
)

plt.savefig(
    f"{OUTPUT_DIR}/fig_binary_combined.png",
    dpi=300, bbox_inches='tight',
    facecolor='white')
plt.show()
plt.close()
print("Saved → fig_binary_combined.png")

# ── Figure 2: One per locality ────────────────────────────
print("\nGenerating per-locality binary maps...")

for name, bounds in SELECTED.items():
    fig2, axes2 = plt.subplots(
        1, 3, figsize=(21, 7),
        gridspec_kw={'wspace': 0.04}
    )
    fig2.suptitle(
        f'Flood Detection — {name}\n'
        'Black = Non-flood  |  White = Flood',
        fontsize=14, fontweight='bold', y=1.02
    )

    for col, (period, plabel) in enumerate(PERIOD_LABELS.items()):
        mask = predict_binary(bounds, period)

        axes2[col].imshow(
            mask,
            cmap='gray',
            vmin=0, vmax=1,
            interpolation='nearest'
        )
        axes2[col].set_title(
            plabel, fontsize=12,
            fontweight='bold', pad=10)
        axes2[col].set_xticks([])
        axes2[col].set_yticks([])
        axes2[col].axis('off')

    # Legend
    legend2 = [
        Patch(facecolor='white', edgecolor='black', label='Flood'),
        Patch(facecolor='black', edgecolor='white', label='Non-flood'),
    ]
    fig2.legend(
        handles=legend2,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.04),
        ncol=2, fontsize=12,
        frameon=True
    )

    fname = name.replace(' ', '_')
    plt.savefig(
        f"{OUTPUT_DIR}/fig_binary_{fname}.png",
        dpi=300, bbox_inches='tight',
        facecolor='white')
    plt.show()
    plt.close()
    print(f"  Saved → fig_binary_{fname}.png")

print("\nAll binary maps saved!")
print("  fig_binary_combined.png")
print("  fig_binary_Budameru_Floodplain.png")
print("  fig_binary_Vijayawada_City_Core.png")
print("  fig_binary_Nunna.png")

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"

print("📁 Scanning your FloodMonitoring_Project folder...\n")

for root, dirs, files in os.walk(DRIVE_DIR):
    # Show folder name
    level = root.replace(DRIVE_DIR, '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    print(f"{indent}📂 {folder_name}/")

    # Show files inside
    sub_indent = '  ' * (level + 1)
    for file in sorted(files):
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{sub_indent}📄 {file}  ({size_mb:.2f} MB)")

In [ ]:
# ============================================================
#  Sentinel-1 SAR — VV+VH Grayscale Composite
#  Publication-quality figure for research paper
#  Layout: 3 localities × 3 periods (Pre, During, Post)
#  Each image = (VV + VH) / 2  normalized to [0, 1]
# ============================================================

import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import logging

# ── Suppress rasterio TIF warnings ───────────────────────────
warnings.filterwarnings("ignore")
logging.getLogger("rasterio").setLevel(logging.ERROR)

# ── Config ───────────────────────────────────────────────────
DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"

TIF_PATHS = {
    "Pre-Flood"    : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Pre_Flood_S1.tif",
    "During Flood" : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_During_Flood_S1.tif",
    "Post-Flood"   : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Post_Flood_S1.tif",
}

LOCALITIES = {
    "Vijayawada City Core" : [80.60, 16.50, 80.68, 16.56],
    "Nunna"                : [80.67, 16.52, 80.72, 16.56],
    "Budameru Floodplain"  : [80.58, 16.52, 80.64, 16.57],
}

PERIODS = ["Pre-Flood", "During Flood", "Post-Flood"]

PERIOD_DATES = {
    "Pre-Flood"    : "Aug 2024",
    "During Flood" : "Sep 1–15, 2024",
    "Post-Flood"   : "Sep 16–Oct 2024",
}

PERIOD_COLORS = {
    "Pre-Flood"    : "#2166AC",
    "During Flood" : "#B2182B",
    "Post-Flood"   : "#1A7B3E",
}

# ── Helper: normalize bands to [0, 1] ────────────────────────
def normalize_bands(vv, vh):
    vv_n = np.clip((vv + 30) / 30, 0, 1)
    vh_n = np.clip((vh + 35) / 30, 0, 1)
    return vv_n, vh_n

# ── Helper: crop tif to locality bounding box ────────────────
def crop_to_locality(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    with rasterio.open(tif_path) as src:
        band_count = src.count
        vv_full = src.read(1).astype(np.float32)
        vh_full = src.read(2).astype(np.float32) if band_count >= 2 else vv_full.copy()

        # Convert geographic bounds → pixel row/col
        row_min, col_min = src.index(min_lon, max_lat)
        row_max, col_max = src.index(max_lon, min_lat)

        # Ensure correct order (index() can return flipped values)
        row_min, row_max = sorted([row_min, row_max])
        col_min, col_max = sorted([col_min, col_max])

        # Clamp to image dimensions
        row_min = max(0, row_min)
        col_min = max(0, col_min)
        row_max = min(src.height, row_max)
        col_max = min(src.width,  col_max)

        vv_crop = vv_full[row_min:row_max, col_min:col_max]
        vh_crop = vh_full[row_min:row_max, col_min:col_max]

    # Replace nodata values
    vv_crop = np.where(np.isfinite(vv_crop), vv_crop, -30.0)
    vh_crop = np.where(np.isfinite(vh_crop), vh_crop, -35.0)

    return vv_crop, vh_crop

# ── Helper: VV+VH grayscale composite ────────────────────────
def make_composite(vv, vh):
    vv_n, vh_n = normalize_bands(vv, vh)
    return (vv_n + vh_n) / 2.0

# ── Figure setup using GridSpec ───────────────────────────────
n_rows = len(LOCALITIES)
n_cols = len(PERIODS)

fig = plt.figure(figsize=(13, 5.2 * n_rows), facecolor="white")

# GridSpec gives full control over spacing — avoids tight_layout issues
gs = fig.add_gridspec(
    n_rows, n_cols,
    hspace=0.35,
    wspace=0.06,
    top=0.93,
    bottom=0.06,
    left=0.04,
    right=0.98
)

# ── Plot each cell ───────────────────────────────────────────
for row_idx, (locality, bbox) in enumerate(LOCALITIES.items()):
    for col_idx, period in enumerate(PERIODS):

        ax = fig.add_subplot(gs[row_idx, col_idx])

        vv, vh    = crop_to_locality(TIF_PATHS[period], bbox)
        composite = make_composite(vv, vh)

        ax.imshow(
            composite,
            cmap="gray",
            vmin=0, vmax=1,
            interpolation="bicubic",
            aspect="equal"
        )
        ax.set_xticks([])
        ax.set_yticks([])

        # Clean thin border
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor("#aaaaaa")
            spine.set_linewidth(0.8)

        # ── Caption under each image ──────────────────────────
        caption = f"{locality}  —  {period}\n{PERIOD_DATES[period]}"
        ax.set_xlabel(
            caption,
            fontsize=9,
            color=PERIOD_COLORS[period],
            fontweight="bold",
            labelpad=7,
            linespacing=1.7
        )

# ── Main title ───────────────────────────────────────────────
fig.suptitle(
    "Sentinel-1 SAR Imagery — VV+VH Grayscale Composite\n"
    "Vijayawada Flood Event, Andhra Pradesh, India  (September 2024)",
    fontsize=13,
    fontweight="bold",
    color="#111111",
    y=0.98
)

# ── Legend ───────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=PERIOD_COLORS[p], label=p)
    for p in PERIODS
]
fig.legend(
    handles=legend_patches,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=3,
    fontsize=11,
    frameon=True,
    edgecolor="#cccccc",
    facecolor="white"
)

# ── Save ─────────────────────────────────────────────────────
OUT_PATH = f"{DRIVE_DIR}/vijayawada_figures/fig_VVVH_composite_publication.png"
plt.savefig(
    OUT_PATH,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
    format="png"
)
plt.show()
print(f"\nSaved → {OUT_PATH}")

In [ ]:
# ============================================================
#  Sentinel-1 SAR — VV+VH Grayscale Composite
#  Publication-quality | Same style as working version
#  Only change: percentile stretch + unsharp mask for clarity
# ============================================================

import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import logging
import cv2

warnings.filterwarnings("ignore")
logging.getLogger("rasterio").setLevel(logging.ERROR)

# ── Config ───────────────────────────────────────────────────
DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"

TIF_PATHS = {
    "Pre-Flood"    : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Pre_Flood_S1.tif",
    "During Flood" : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_During_Flood_S1.tif",
    "Post-Flood"   : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Post_Flood_S1.tif",
}

LOCALITIES = {
    "Vijayawada City Core" : [80.60, 16.50, 80.68, 16.56],
    "Nunna"                : [80.67, 16.52, 80.72, 16.56],
    "Budameru Floodplain"  : [80.58, 16.52, 80.64, 16.57],
}

PERIODS = ["Pre-Flood", "During Flood", "Post-Flood"]

PERIOD_DATES = {
    "Pre-Flood"    : "Aug 2024",
    "During Flood" : "Sep 1–15, 2024",
    "Post-Flood"   : "Sep 16–Oct 2024",
}

PERIOD_COLORS = {
    "Pre-Flood"    : "#2166AC",
    "During Flood" : "#B2182B",
    "Post-Flood"   : "#1A7B3E",
}

# ── Helper: normalize bands (UNCHANGED from original) ─────────
def normalize_bands(vv, vh):
    vv_n = np.clip((vv + 30) / 30, 0, 1)
    vh_n = np.clip((vh + 35) / 30, 0, 1)
    return vv_n, vh_n

# ── Helper: crop (UNCHANGED from original) ────────────────────
def crop_to_locality(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    with rasterio.open(tif_path) as src:
        band_count = src.count
        vv_full    = src.read(1).astype(np.float32)
        vh_full    = src.read(2).astype(np.float32) if band_count >= 2 else vv_full.copy()

        row_min, col_min = src.index(min_lon, max_lat)
        row_max, col_max = src.index(max_lon, min_lat)

        row_min, row_max = sorted([row_min, row_max])
        col_min, col_max = sorted([col_min, col_max])

        row_min = max(0, row_min)
        col_min = max(0, col_min)
        row_max = min(src.height, row_max)
        col_max = min(src.width,  col_max)

        vv_crop = vv_full[row_min:row_max, col_min:col_max]
        vh_crop = vh_full[row_min:row_max, col_min:col_max]

    vv_crop = np.where(np.isfinite(vv_crop), vv_crop, -30.0)
    vh_crop = np.where(np.isfinite(vh_crop), vh_crop, -35.0)

    return vv_crop, vh_crop

# ── Composite: same as original + clarity only ────────────────
def make_composite(vv, vh):
    # STEP 1: exact same as original
    vv_n, vh_n = normalize_bands(vv, vh)
    composite  = (vv_n + vh_n) / 2.0

    # STEP 2: percentile stretch — preserves look, extends contrast range
    # This keeps the same dark/bright SAR feel but uses full range
    p2  = np.percentile(composite, 2)
    p98 = np.percentile(composite, 98)
    composite = np.clip((composite - p2) / (p98 - p2 + 1e-8), 0, 1)

    # STEP 3: unsharp mask — sharpens edges, removes soft/blurry look
    # Very light sharpening so it doesn't change the SAR character
    img_u8    = (composite * 255).astype(np.uint8)
    blurred   = cv2.GaussianBlur(img_u8, (0, 0), sigmaX=1.5)
    sharpened = cv2.addWeighted(img_u8, 1.5, blurred, -0.5, 0)
    composite = np.clip(sharpened, 0, 255).astype(np.float32) / 255.0

    return composite

# ── Figure: identical to original ────────────────────────────
n_rows = len(LOCALITIES)
n_cols = len(PERIODS)

fig = plt.figure(figsize=(13, 5.2 * n_rows), facecolor="white")

gs = fig.add_gridspec(
    n_rows, n_cols,
    hspace=0.35,
    wspace=0.06,
    top=0.93,
    bottom=0.06,
    left=0.04,
    right=0.98
)

for row_idx, (locality, bbox) in enumerate(LOCALITIES.items()):
    for col_idx, period in enumerate(PERIODS):

        ax = fig.add_subplot(gs[row_idx, col_idx])

        vv, vh    = crop_to_locality(TIF_PATHS[period], bbox)
        composite = make_composite(vv, vh)

        # IDENTICAL imshow settings as original
        ax.imshow(
            composite,
            cmap="gray",
            vmin=0, vmax=1,
            interpolation="bicubic",
            aspect="equal"
        )
        ax.set_xticks([])
        ax.set_yticks([])

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor("#aaaaaa")
            spine.set_linewidth(0.8)

        caption = f"{locality}  —  {period}\n{PERIOD_DATES[period]}"
        ax.set_xlabel(
            caption,
            fontsize=9,
            color=PERIOD_COLORS[period],
            fontweight="bold",
            labelpad=7,
            linespacing=1.7
        )

fig.suptitle(
    "Sentinel-1 SAR Imagery — VV+VH Grayscale Composite\n"
    "Vijayawada Flood Event, Andhra Pradesh, India  (September 2024)",
    fontsize=13,
    fontweight="bold",
    color="#111111",
    y=0.98
)

legend_patches = [
    mpatches.Patch(color=PERIOD_COLORS[p], label=p)
    for p in PERIODS
]
fig.legend(
    handles=legend_patches,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=3,
    fontsize=11,
    frameon=True,
    edgecolor="#cccccc",
    facecolor="white"
)

OUT_PATH = f"{DRIVE_DIR}/vijayawada_figures/fig_VVVH_composite_publication.png"
plt.savefig(
    OUT_PATH,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
    format="png"
)
plt.show()
print(f"\nSaved → {OUT_PATH}")

In [ ]:
# ============================================================
#  Water vs Non-Water Pixel Visualization
#  Black & White Binary Mask
#  White = water | Black = non-water
#  Method 1: SAR Threshold
#  Method 2: U-Net Model Prediction
#  Layout: 3 localities × 3 periods
# ============================================================

import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
import warnings
import logging
from skimage.transform import resize as sk_resize

warnings.filterwarnings("ignore")
logging.getLogger("rasterio").setLevel(logging.ERROR)

# ── Config ───────────────────────────────────────────────────
DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"

TIF_PATHS = {
    "Pre-Flood"    : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Pre_Flood_S1.tif",
    "During Flood" : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_During_Flood_S1.tif",
    "Post-Flood"   : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Post_Flood_S1.tif",
}

LOCALITIES = {
    "Vijayawada City Core" : [80.60, 16.50, 80.68, 16.56],
    "Nunna"                : [80.67, 16.52, 80.72, 16.56],
    "Budameru Floodplain"  : [80.58, 16.52, 80.64, 16.57],
}

PERIODS      = ["Pre-Flood", "During Flood", "Post-Flood"]
PERIOD_DATES = {
    "Pre-Flood"    : "Aug 2024",
    "During Flood" : "Sep 1–15, 2024",
    "Post-Flood"   : "Sep 16–Oct 2024",
}
PERIOD_COLORS = {
    "Pre-Flood"    : "#2166AC",
    "During Flood" : "#B2182B",
    "Post-Flood"   : "#1A7B3E",
}

SAR_THRESHOLD = 0.35

# ── Load U-Net model ─────────────────────────────────────────
def weighted_dice_bce_loss(y_true, y_pred):
    smooth = 1e-6
    yt    = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp    = tf.reshape(y_pred, [-1])
    bce   = tf.keras.losses.binary_crossentropy(yt, yp)
    inter = tf.reduce_sum(yt * yp)
    dice  = 1.0 - (2.0 * inter + smooth) / (
            tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)
    return 0.5 * bce + 0.5 * dice

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    yt    = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp    = tf.cast(tf.reshape(tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    inter = tf.reduce_sum(yt * yp)
    return (2.0 * inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)

def iou_score(y_true, y_pred, smooth=1e-6):
    yt    = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp    = tf.cast(tf.reshape(tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    inter = tf.reduce_sum(yt * yp)
    union = tf.reduce_sum(yt) + tf.reduce_sum(yp) - inter
    return (inter + smooth) / (union + smooth)

print("Loading U-Net model...")
model = tf.keras.models.load_model(
    f"{DRIVE_DIR}/model/best_unet_flood.keras",
    custom_objects={
        "weighted_dice_bce_loss" : weighted_dice_bce_loss,
        "dice_coefficient"       : dice_coefficient,
        "iou_score"              : iou_score,
    }
)
print(f"Model loaded! Input: {model.input_shape}")

# ── Helper: normalize bands ───────────────────────────────────
def normalize_bands(vv, vh):
    vv_n = np.clip((vv + 30) / 30, 0, 1)
    vh_n = np.clip((vh + 35) / 30, 0, 1)
    return vv_n, vh_n

# ── Helper: crop to locality ──────────────────────────────────
def crop_to_locality(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    with rasterio.open(tif_path) as src:
        band_count = src.count
        vv_full    = src.read(1).astype(np.float32)
        vh_full    = src.read(2).astype(np.float32) if band_count >= 2 else vv_full.copy()

        row_min, col_min = src.index(min_lon, max_lat)
        row_max, col_max = src.index(max_lon, min_lat)

        row_min, row_max = sorted([row_min, row_max])
        col_min, col_max = sorted([col_min, col_max])

        row_min = max(0, row_min)
        col_min = max(0, col_min)
        row_max = min(src.height, row_max)
        col_max = min(src.width,  col_max)

        vv_crop = vv_full[row_min:row_max, col_min:col_max]
        vh_crop = vh_full[row_min:row_max, col_min:col_max]

    vv_crop = np.where(np.isfinite(vv_crop), vv_crop, -30.0)
    vh_crop = np.where(np.isfinite(vh_crop), vh_crop, -35.0)
    return vv_crop, vh_crop

# ── Method 1: SAR threshold ───────────────────────────────────
def sar_threshold_mask(vv, vh):
    vv_n, vh_n = normalize_bands(vv, vh)
    composite  = (vv_n + vh_n) / 2.0
    return (composite < SAR_THRESHOLD).astype(np.uint8)

# ── Method 2: U-Net prediction ───────────────────────────────
def model_prediction_mask(vv, vh, threshold=0.5):
    vv_n, vh_n = normalize_bands(vv, vh)
    H, W       = vv_n.shape
    vv_r = sk_resize(vv_n, (512, 512), preserve_range=True)
    vh_r = sk_resize(vh_n, (512, 512), preserve_range=True)
    img       = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]
    prob      = model.predict(img, verbose=0)[0, ..., 0]
    mask_512  = (prob > threshold).astype(np.uint8)
    mask      = sk_resize(mask_512, (H, W), preserve_range=True, order=0).astype(np.uint8)
    return mask

# ── Plot function ─────────────────────────────────────────────
def plot_binary_masks(mask_fn, method_title, out_filename):
    n_rows = len(LOCALITIES)
    n_cols = len(PERIODS)

    fig = plt.figure(figsize=(13, 5.2 * n_rows), facecolor="white")
    gs  = fig.add_gridspec(
        n_rows, n_cols,
        hspace=0.35, wspace=0.06,
        top=0.93, bottom=0.06,
        left=0.04, right=0.98
    )

    for row_idx, (locality, bbox) in enumerate(LOCALITIES.items()):
        for col_idx, period in enumerate(PERIODS):

            ax = fig.add_subplot(gs[row_idx, col_idx])

            vv, vh = crop_to_locality(TIF_PATHS[period], bbox)
            mask   = mask_fn(vv, vh)

            # White = water (1), Black = non-water (0)
            ax.imshow(
                mask,
                cmap="gray",
                vmin=0, vmax=1,
                interpolation="nearest",
                aspect="equal"
            )
            ax.set_xticks([])
            ax.set_yticks([])

            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor("#aaaaaa")
                spine.set_linewidth(0.8)

            caption = f"{locality}  —  {period}\n{PERIOD_DATES[period]}"
            ax.set_xlabel(
                caption,
                fontsize=9,
                color=PERIOD_COLORS[period],
                fontweight="bold",
                labelpad=7,
                linespacing=1.7
            )

    fig.suptitle(
        f"Water vs Non-Water Binary Mask — {method_title}\n"
        "Vijayawada Flood Event, Andhra Pradesh, India  (September 2024)",
        fontsize=13,
        fontweight="bold",
        color="#111111",
        y=0.98
    )

    legend_patches = [
        mpatches.Patch(color="white", label="Water pixels",
                       edgecolor="#aaaaaa", linewidth=0.8),
        mpatches.Patch(color="black", label="Non-water pixels"),
    ]
    fig.legend(
        handles=legend_patches,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=2,
        fontsize=11,
        frameon=True,
        edgecolor="#cccccc",
        facecolor="white"
    )

    out_path = f"{DRIVE_DIR}/vijayawada_figures/{out_filename}"
    plt.savefig(out_path, dpi=300, bbox_inches="tight",
                facecolor="white", format="png")
    plt.show()
    print(f"\nSaved → {out_path}")

# ── Figure 1: SAR Threshold ───────────────────────────────────
print("\nGenerating SAR threshold binary mask...")
plot_binary_masks(
    mask_fn      = sar_threshold_mask,
    method_title = "SAR Threshold (VV+VH < 0.35)",
    out_filename = "fig_binary_mask_SAR_threshold.png"
)

# ── Figure 2: U-Net Model ─────────────────────────────────────
print("\nGenerating U-Net model binary mask...")
plot_binary_masks(
    mask_fn      = model_prediction_mask,
    method_title = "U-Net Model Prediction",
    out_filename = "fig_binary_mask_UNet_model.png"
)

print("\nDone! Both figures saved to vijayawada_figures/")

In [ ]:
# ============================================================
#  U-Net Model Only - Water vs Non-Water + Pixel Counting
# ============================================================

import numpy as np
import rasterio
import matplotlib.pyplot as plt
import tensorflow as tf
from skimage.transform import resize as sk_resize
import warnings, logging

warnings.filterwarnings("ignore")
logging.getLogger("rasterio").setLevel(logging.ERROR)

# ── Config ───────────────────────────────────────────────────
DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"

TIF_PATHS = {
    "Pre-Flood"    : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Pre_Flood_S1.tif",
    "During Flood" : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_During_Flood_S1.tif",
    "Post-Flood"   : f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Post_Flood_S1.tif",
}

LOCALITIES = {
    "Vijayawada City Core" : [80.60, 16.50, 80.68, 16.56],
    "Nunna"                : [80.67, 16.52, 80.72, 16.56],
    "Budameru Floodplain"  : [80.58, 16.52, 80.64, 16.57],
}

PERIODS = ["Pre-Flood", "During Flood", "Post-Flood"]

# ── Load Model ───────────────────────────────────────────────
def weighted_dice_bce_loss(y_true, y_pred):
    smooth = 1e-6
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.reshape(y_pred, [-1])
    bce = tf.keras.losses.binary_crossentropy(yt, yp)
    inter = tf.reduce_sum(yt * yp)
    dice = 1.0 - (2.0 * inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)
    return 0.5 * bce + 0.5 * dice

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    inter = tf.reduce_sum(yt * yp)
    return (2.0 * inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)

def iou_score(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    inter = tf.reduce_sum(yt * yp)
    union = tf.reduce_sum(yt) + tf.reduce_sum(yp) - inter
    return (inter + smooth) / (union + smooth)

print("Loading U-Net model...")
model = tf.keras.models.load_model(
    f"{DRIVE_DIR}/model/best_unet_flood.keras",
    custom_objects={
        "weighted_dice_bce_loss": weighted_dice_bce_loss,
        "dice_coefficient": dice_coefficient,
        "iou_score": iou_score,
    }
)
print("Model loaded!")

# ── Normalize ────────────────────────────────────────────────
def normalize_bands(vv, vh):
    vv_n = np.clip((vv + 30) / 30, 0, 1)
    vh_n = np.clip((vh + 35) / 30, 0, 1)
    return vv_n, vh_n

# ── Crop ─────────────────────────────────────────────────────
def crop_to_locality(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    with rasterio.open(tif_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)

        row_min, col_min = src.index(min_lon, max_lat)
        row_max, col_max = src.index(max_lon, min_lat)

        row_min, row_max = sorted([row_min, row_max])
        col_min, col_max = sorted([col_min, col_max])

        vv_crop = vv[row_min:row_max, col_min:col_max]
        vh_crop = vh[row_min:row_max, col_min:col_max]

    return vv_crop, vh_crop

# ── Pixel Counting ───────────────────────────────────────────
def count_pixels(mask):
    total = mask.size
    water = np.sum(mask == 1)
    non_water = np.sum(mask == 0)

    water_per = (water / total) * 100
    non_water_per = (non_water / total) * 100

    return total, water, non_water, water_per, non_water_per

# ── U-Net Prediction ─────────────────────────────────────────
def model_prediction_mask(vv, vh):
    vv_n, vh_n = normalize_bands(vv, vh)
    H, W = vv_n.shape

    vv_r = sk_resize(vv_n, (512, 512))
    vh_r = sk_resize(vh_n, (512, 512))

    img = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]
    pred = model.predict(img, verbose=0)[0, ..., 0]

    mask = (pred > 0.5).astype(np.uint8)
    mask = sk_resize(mask, (H, W), order=0).astype(np.uint8)

    return mask

# ── Plot ─────────────────────────────────────────────────────
def plot_unet_results():
    fig, axes = plt.subplots(len(LOCALITIES), len(PERIODS), figsize=(12, 12))

    for i, (loc, bbox) in enumerate(LOCALITIES.items()):
        for j, period in enumerate(PERIODS):

            vv, vh = crop_to_locality(TIF_PATHS[period], bbox)
            mask = model_prediction_mask(vv, vh)

            total, water, non_water, wp, nwp = count_pixels(mask)

            print("\n===================================")
            print(f"{loc} — {period}")
            print("===================================")
            print(f"Total Pixels     : {total}")
            print(f"Water Pixels     : {water}")
            print(f"Non-Water Pixels : {non_water}")
            print(f"Water %          : {wp:.2f}%")
            print(f"Non-Water %      : {nwp:.2f}%")

            ax = axes[i, j]
            ax.imshow(mask, cmap="gray")
            ax.set_title(f"{loc}\n{period}", fontsize=9)
            ax.axis("off")

            # Overlay stats
            ax.text(
                0.02, 0.95,
                f"W:{water}\nNW:{non_water}",
                transform=ax.transAxes,
                color="yellow",
                fontsize=8,
                bbox=dict(facecolor="black", alpha=0.5)
            )

    plt.suptitle("U-Net Model Flood Detection", fontsize=14)
    plt.tight_layout()
    plt.show()

# ── Run ──────────────────────────────────────────────────────
plot_unet_results()

In [ ]:
# ============================================================
#  FLOOD ANALYSIS (U-NET) + GRAPHS + CHANGE MAP + COLOR MASKS
# ============================================================

# -------------------- IMPORTS --------------------
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import tensorflow as tf
from skimage.transform import resize
import matplotlib.patches as mpatches
import warnings, logging

warnings.filterwarnings("ignore")
logging.getLogger("rasterio").setLevel(logging.ERROR)

# -------------------- STYLE --------------------
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8
})

# -------------------- PATHS --------------------
DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"

TIF_PATHS = {
    "Pre-Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Pre_Flood_S1.tif",
    "During Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_During_Flood_S1.tif",
    "Post-Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Post_Flood_S1.tif",
}

# -------------------- LOCATIONS --------------------
LOCALITIES = {
    "City Core": [80.60, 16.50, 80.68, 16.56],
    "Nunna": [80.67, 16.52, 80.72, 16.56],
    "Budameru": [80.58, 16.52, 80.64, 16.57],
}

PERIODS = ["Pre-Flood", "During Flood", "Post-Flood"]
COLORS = ["#1f77b4", "#d62728", "#2ca02c"]

# -------------------- LOAD MODEL --------------------
def weighted_dice_bce_loss(y_true, y_pred):
    smooth = 1e-6
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.reshape(y_pred, [-1])
    bce = tf.keras.losses.binary_crossentropy(yt, yp)
    inter = tf.reduce_sum(yt * yp)
    dice = 1.0 - (2.0 * inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)
    return 0.5 * bce + 0.5 * dice

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    inter = tf.reduce_sum(yt * yp)
    return (2.0 * inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)

def iou_score(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(tf.cast(y_pred > 0.5, tf.float32), [-1]), tf.float32)
    inter = tf.reduce_sum(yt * yp)
    union = tf.reduce_sum(yt) + tf.reduce_sum(yp) - inter
    return (inter + smooth) / (union + smooth)

print("Loading model...")
model = tf.keras.models.load_model(
    f"{DRIVE_DIR}/model/best_unet_flood.keras",
    custom_objects={
        "weighted_dice_bce_loss": weighted_dice_bce_loss,
        "dice_coefficient": dice_coefficient,
        "iou_score": iou_score,
    }
)
print("Model Loaded ✅")

# -------------------- FUNCTIONS --------------------

def normalize(vv, vh):
    vv = np.clip((vv + 30) / 30, 0, 1)
    vh = np.clip((vh + 35) / 30, 0, 1)
    return vv, vh

def crop(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    with rasterio.open(tif_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)

        r1, c1 = src.index(min_lon, max_lat)
        r2, c2 = src.index(max_lon, min_lat)

        r1, r2 = sorted([r1, r2])
        c1, c2 = sorted([c1, c2])

        return vv[r1:r2, c1:c2], vh[r1:r2, c1:c2]

def predict_mask(vv, vh):
    vv, vh = normalize(vv, vh)
    H, W = vv.shape

    vv_r = resize(vv, (512, 512))
    vh_r = resize(vh, (512, 512))

    img = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]
    pred = model.predict(img, verbose=0)[0, ..., 0]

    mask = (pred > 0.5).astype(np.uint8)
    return resize(mask, (H, W), order=0).astype(np.uint8)

def to_km2(pixels):
    return (pixels * 100) / 1_000_000

def get_change(pre, post):
    return post - pre

# -------------------- DATA COLLECTION --------------------

pixels_data = {loc: [] for loc in LOCALITIES}
area_data = {loc: [] for loc in LOCALITIES}
masks_store = {}

for loc, bbox in LOCALITIES.items():
    masks_store[loc] = {}
    print(f"\n📍 {loc}")

    for period in PERIODS:
        vv, vh = crop(TIF_PATHS[period], bbox)
        mask = predict_mask(vv, vh)

        water = np.sum(mask == 1)
        area = to_km2(water)

        print(f"{period}: Pixels={water}, Area={area:.4f} km²")

        pixels_data[loc].append(water)
        area_data[loc].append(area)
        masks_store[loc][period] = mask

# -------------------- GRAPH FUNCTION --------------------

def plot_graph(data, title, ylabel, fmt):
    x = np.arange(len(LOCALITIES))
    width = 0.18

    fig, ax = plt.subplots(figsize=(6,3.5))

    for i, period in enumerate(PERIODS):
        values = [data[loc][i] for loc in LOCALITIES]
        bars = ax.bar(x + i*width, values, width,
                      label=period, color=COLORS[i])

        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2,
                    h,
                    format(h, fmt),
                    ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x + width)
    ax.set_xticklabels(list(LOCALITIES.keys()))
    ax.set_xlabel("Localities")
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

# -------------------- CHANGE MAP --------------------

def show_change_maps():
    for loc in LOCALITIES:
        pre = masks_store[loc]["Pre-Flood"]
        dur = masks_store[loc]["During Flood"]
        post = masks_store[loc]["Post-Flood"]

        change1 = get_change(pre, dur)
        change2 = get_change(dur, post)

        plt.figure(figsize=(8,3))

        plt.subplot(1,2,1)
        plt.imshow(change1, cmap='bwr', vmin=-1, vmax=1)
        plt.title(f"{loc}\nPre → During")
        plt.axis('off')

        plt.subplot(1,2,2)
        plt.imshow(change2, cmap='bwr', vmin=-1, vmax=1)
        plt.title(f"{loc}\nDuring → Post")
        plt.axis('off')

        plt.suptitle("Flood Change Detection", fontsize=10)
        plt.tight_layout()
        plt.show()

# -------------------- COLORED MASK --------------------

def show_colored_masks():
    for loc in LOCALITIES:

        plt.figure(figsize=(10,3))

        for i, period in enumerate(PERIODS):
            mask = masks_store[loc][period]

            colored = np.zeros((mask.shape[0], mask.shape[1], 3))
            colored[mask == 1] = [0, 0, 1]   # Blue = Water
            colored[mask == 0] = [0, 0, 0]   # Black = Land

            plt.subplot(1,3,i+1)
            plt.imshow(colored)
            plt.title(f"{loc}\n{period}")
            plt.axis("off")

        # Legend
        water_patch = mpatches.Patch(color='blue', label='Water')
        land_patch = mpatches.Patch(color='black', label='Land')

        plt.legend(handles=[water_patch, land_patch],
                   loc='lower center', ncol=2)

        plt.suptitle("Flood Segmentation (Water vs Land)", fontsize=10)
        plt.tight_layout()
        plt.show()

# -------------------- RUN EVERYTHING --------------------

plot_graph(pixels_data, "Flood Pixels (U-Net)", "Pixels", ".0f")
plot_graph(area_data, "Flood Area (km²)", "Area (km²)", ".3f")

show_change_maps()
show_colored_masks()

In [ ]:
try:
    value = all_results["During_Flood"]["Vijayawada City Core"]["flood_area"]
    print(value)
    print("all_results ready!")
except KeyError as e:
    print(f"Missing key: {e} — check JSON structure")
except NameError:
    print("all_results not loaded — reload from JSON")
except Exception as e:
    print(f"Unexpected error: {e}")

In [ ]:
import os

folder_path = "/content/drive/MyDrive/FloodMonitoring_Project"
print(os.listdir(folder_path))

In [ ]:
import json

file_path = "/content/drive/MyDrive/FloodMonitoring_Project/logs/test_results.json"

with open(file_path, "r") as f:
    all_results = json.load(f)


print("all_results loaded successfully!")
print(all_results.keys())

In [ ]:
# ============================================================
#  FLOOD PREDICTION — Risk Assessment Module
#  Combines U-Net flood detection + actual rainfall data
#  Rule-based risk: Low / Medium / High
#  Data source: Open-Meteo (free, no API key, actual IMD data)
# ============================================================

# ── STEP 1: Install & imports ─────────────────────────────
!pip install openmeteo-requests requests-cache retry-requests -q

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

In [ ]:
# ── STEP 2: Fetch actual rainfall data from Open-Meteo ────
# Vijayawada coordinates
LAT = 16.5193
LON = 80.6305

def get_rainfall_data(lat, lon, start_date, end_date):
    """
    Fetch actual daily rainfall data for Vijayawada
    from Open-Meteo (uses ERA5 reanalysis — matches IMD data)
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude"          : lat,
        "longitude"         : lon,
        "start_date"        : start_date,
        "end_date"          : end_date,
        "daily"             : "precipitation_sum",
        "timezone"          : "Asia/Kolkata"
    }
    response = requests.get(url, params=params)
    data     = response.json()

    df = pd.DataFrame({
        "date"     : data["daily"]["time"],
        "rainfall" : data["daily"]["precipitation_sum"]
    })
    df["date"]     = pd.to_datetime(df["date"])
    df["rainfall"] = df["rainfall"].fillna(0)
    return df
# Fetch all 3 periods
print("Fetching actual rainfall data for Vijayawada...")

pre_rain    = get_rainfall_data(LAT, LON,
                                "2024-08-01", "2024-08-31")
during_rain = get_rainfall_data(LAT, LON,
                                "2024-09-01", "2024-09-15")
post_rain   = get_rainfall_data(LAT, LON,
                                "2024-09-16", "2024-09-30")

print(f"\nRainfall data fetched!")
print(f"  Pre-Flood    : {len(pre_rain)} days, "
      f"total={pre_rain['rainfall'].sum():.1f} mm, "
      f"max={pre_rain['rainfall'].max():.1f} mm/day")
print(f"  During Flood : {len(during_rain)} days, "
      f"total={during_rain['rainfall'].sum():.1f} mm, "
      f"max={during_rain['rainfall'].max():.1f} mm/day")
print(f"  Post-Flood   : {len(post_rain)} days, "
      f"total={post_rain['rainfall'].sum():.1f} mm, "
      f"max={post_rain['rainfall'].max():.1f} mm/day")

In [ ]:
# ── STEP 3: Define thresholds ─────────────────────────────
# Based on IMD rainfall classification for Andhra Pradesh
RAIN_LOW    = 10    # mm/day — light rain
RAIN_MED    = 35    # mm/day — moderate rain
RAIN_HIGH   = 65    # mm/day — heavy rain (IMD threshold)

FLOOD_LOW   = 0.5   # km² — small flood
FLOOD_MED   = 2.0   # km² — moderate flood
FLOOD_HIGH  = 5.0   # km² — large flood

In [ ]:
# ============================================================
#  FLOOD AREA EXTRACTION (U-NET) — CLEAN + FORMATTED OUTPUT
# ============================================================

# ---- Suppress warnings ----
import os
os.environ["CPL_LOG"] = "ERROR"

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)

# ---- Imports ----
import numpy as np
import rasterio
import tensorflow as tf
from skimage.transform import resize
import json

# ---- Paths ----
DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"

TIF_PATHS = {
    "Pre_Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Pre_Flood_S1.tif",
    "During_Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_During_Flood_S1.tif",
    "Post_Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Post_Flood_S1.tif",
}

LOCALITIES = {
    "Budameru Floodplain": [80.58, 16.52, 80.64, 16.57],
    "Vijayawada City Core": [80.60, 16.50, 80.68, 16.56],
    "Nunna": [80.67, 16.52, 80.72, 16.56],
}

# ---- Load model ----
print("Loading model...")
model = tf.keras.models.load_model(
    f"{DRIVE_DIR}/model/best_unet_flood.keras",
    compile=False
)
print("Model Loaded ✅")

# ---- Helper functions ----
def normalize(vv, vh):
    vv = np.clip((vv + 30) / 30, 0, 1)
    vh = np.clip((vh + 35) / 30, 0, 1)
    return vv, vh

def crop(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox

    with rasterio.open(tif_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)

        r1, c1 = src.index(min_lon, max_lat)
        r2, c2 = src.index(max_lon, min_lat)

        r1, r2 = sorted([r1, r2])
        c1, c2 = sorted([c1, c2])

        return vv[r1:r2, c1:c2], vh[r1:r2, c1:c2]

def predict_mask(vv, vh):
    vv, vh = normalize(vv, vh)
    H, W = vv.shape

    vv_r = resize(vv, (512, 512))
    vh_r = resize(vh, (512, 512))

    img = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]

    pred = model.predict(img, verbose=0)[0, ..., 0]

    mask = (pred > 0.5).astype(np.uint8)

    return resize(mask, (H, W), order=0).astype(np.uint8)

def to_km2(pixels):
    return (pixels * 100) / 1_000_000

# ---- Run model ----
print("\nRunning flood detection...\n")

all_results = {}

for period, tif in TIF_PATHS.items():
    all_results[period] = {}

    for loc, bbox in LOCALITIES.items():

        vv, vh = crop(tif, bbox)
        mask = predict_mask(vv, vh)

        pixels = int(np.sum(mask == 1))
        area   = float(to_km2(pixels))

        all_results[period][loc] = {
            "pixels": pixels,
            "flood_area": area
        }

# ---- Clean formatted output ----
print("\n" + "="*65)
print("        FLOOD AREA SUMMARY (U-NET OUTPUT)")
print("="*65)

for loc in LOCALITIES:
    print(f"\n📍 {loc}")

    for period in ["Pre_Flood", "During_Flood", "Post_Flood"]:
        data = all_results[period][loc]

        area = data["flood_area"]
        pixels = data["pixels"]

        period_label = period.replace("_", "-")

        print(f"   {period_label:<13} → {area:.3f} km² ({pixels} pixels)")

print("\n" + "="*65)

# ---- Top flooded areas (During Flood) ----
print("\n🔥 Top Flooded Areas (During Flood):\n")

sorted_locs = sorted(
    LOCALITIES.keys(),
    key=lambda loc: all_results["During_Flood"][loc]["flood_area"],
    reverse=True
)

for loc in sorted_locs:
    area = all_results["During_Flood"][loc]["flood_area"]
    print(f"{loc:<25} → {area:.3f} km²")

# ---- Save results ----
save_path = f"{DRIVE_DIR}/outputs/flood_results.json"

with open(save_path, "w") as f:
    json.dump(all_results, f, indent=2)

print("\nSaved to:", save_path)

print("\nFlood area extraction complete! 🚀")

In [ ]:
# ── STEP 4: Flood area from your U-Net results ────────────
# Using results from all_results (already computed)
# If session restarted, enter manually from your CSV

FLOOD_AREAS = {
    "Budameru Floodplain" : {
        "Pre_Flood"    : all_results["Pre_Flood"]["Budameru Floodplain"]["flood_area"],
        "During_Flood" : all_results["During_Flood"]["Budameru Floodplain"]["flood_area"],
        "Post_Flood"   : all_results["Post_Flood"]["Budameru Floodplain"]["flood_area"],
    },
    "Vijayawada City Core" : {
        "Pre_Flood"    : all_results["Pre_Flood"]["Vijayawada City Core"]["flood_area"],
        "During_Flood" : all_results["During_Flood"]["Vijayawada City Core"]["flood_area"],
        "Post_Flood"   : all_results["Post_Flood"]["Vijayawada City Core"]["flood_area"],
    },
    "Nunna" : {
        "Pre_Flood"    : all_results["Pre_Flood"]["Nunna"]["flood_area"],
        "During_Flood" : all_results["During_Flood"]["Nunna"]["flood_area"],
        "Post_Flood"   : all_results["Post_Flood"]["Nunna"]["flood_area"],
    },
}

In [ ]:
# ── STEP 5: Risk classification function ─────────────────
def classify_risk(rainfall_mm, flood_area_km2):

    # Rain classification
    if rainfall_mm >= RAIN_HIGH:
        rain_level = "High"
    elif rainfall_mm >= RAIN_MED:
        rain_level = "Moderate"
    else:
        rain_level = "Low"

    # ✅ FIXED flood thresholds usage
    if flood_area_km2 >= FLOOD_HIGH:
        flood_level = "High"
    elif flood_area_km2 >= FLOOD_MED:
        flood_level = "Moderate"
    else:
        flood_level = "Low"

    # Risk rules
    if rain_level == "High" and flood_level == "High":
        risk  = "High Risk"
        reason= "Heavy rainfall + large flood area detected"
    elif rain_level == "High" and flood_level == "Moderate":
        risk  = "High Risk"
        reason= "Heavy rainfall with moderate flood extent"
    elif rain_level == "Moderate" and flood_level == "High":
        risk  = "High Risk"
        reason= "Moderate rainfall but large flood area persists"
    elif rain_level == "Moderate" or flood_level == "Moderate":
        risk  = "Medium Risk"
        reason= "Moderate rainfall or flood area detected"
    elif rain_level == "Low" and flood_level == "High":
        risk  = "Medium Risk"
        reason= "Low rainfall but residual flood area remains"
    else:
        risk  = "Low Risk"
        reason= "Low rainfall and minimal flood area"

    return risk, rain_level, flood_level, reason


# ── STEP 6: Rainfall stats ─────────────────
PERIOD_RAIN = {
    "Pre_Flood": pre_rain["rainfall"].mean(),
    "During_Flood": during_rain["rainfall"].mean(),
    "Post_Flood": post_rain["rainfall"].mean(),
}

PERIOD_MAX_RAIN = {
    "Pre_Flood": pre_rain["rainfall"].max(),
    "During_Flood": during_rain["rainfall"].max(),
    "Post_Flood": post_rain["rainfall"].max(),
}

# ✅ FIXED flood thresholds (VERY IMPORTANT)
FLOOD_LOW   = 0.1
FLOOD_MED   = 0.5
FLOOD_HIGH  = 1.0

results_risk = []

for locality, periods in FLOOD_AREAS.items():
    for period, flood_area in periods.items():

        # ✅ BETTER rainfall logic
        rain = 0.7 * PERIOD_MAX_RAIN[period] + 0.3 * PERIOD_RAIN[period]

        risk, rain_lvl, flood_lvl, reason = classify_risk(
            rain, flood_area)

        results_risk.append({
            "Locality": locality,
            "Period": period.replace("_", " "),
            "Rain (mm)": round(rain, 1),
            "Rain Level": rain_lvl,
            "Flood Area km²": round(flood_area, 3),
            "Flood Level": flood_lvl,
            "Risk": risk,
            "Reason": reason,
        })

df_risk = pd.DataFrame(results_risk)


# ── STEP 7: Clean Output ─────────────────
print("\n" + "=" * 80)
print("  FLOOD RISK ASSESSMENT — Vijayawada Sep 2024")
print("=" * 80)

for loc in df_risk["Locality"].unique():
    print(f"\n📍 {loc}")

    sub = df_risk[df_risk["Locality"] == loc]

    for _, row in sub.iterrows():
        print(f"   {row['Period']:<13} → {row['Risk']:<11} | "
              f"Rain: {row['Rain (mm)']} mm | "
              f"Flood: {row['Flood Area km²']} km²")

print("\n" + "=" * 80)


# ── STEP 8: Save CSV ─────────────────
OUTPUT_DIR = "/content/drive/MyDrive/FloodMonitoring_Project/outputs"

csv_path = f"{OUTPUT_DIR}/flood_risk_assessment.csv"
df_risk.to_csv(csv_path, index=False)

print(f"\nSaved → {csv_path}")

In [ ]:
# ============================================================
#  PHASE-WISE RAINFALL (15 DAYS EACH) — CLEAN VERSION
# ============================================================

import requests
import pandas as pd
import matplotlib.pyplot as plt

# ---- Config ----
LAT = 16.5193
LON = 80.6305
OUTPUT_DIR = "/content/drive/MyDrive/FloodMonitoring_Project/outputs"

# ---- Rainfall thresholds ----
RAIN_LOW  = 10
RAIN_MED  = 35
RAIN_HIGH = 65

# ---- Fetch rainfall ----
def get_rainfall_data(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "precipitation_sum",
        "timezone": "Asia/Kolkata"
    }
    response = requests.get(url, params=params)
    data = response.json()

    df = pd.DataFrame({
        "date": data["daily"]["time"],
        "rainfall": data["daily"]["precipitation_sum"]
    })
    df["date"] = pd.to_datetime(df["date"])
    df["rainfall"] = df["rainfall"].fillna(0)

    return df

# ============================================================
#  15-DAY PERIODS
# ============================================================

print("Fetching 15-day rainfall data...\n")

# ✅ Pre-Flood (Aug 15–31)
pre_rain = get_rainfall_data(LAT, LON, "2024-08-15", "2024-08-31")

# ✅ During Flood (Sep 1–15)
during_rain = get_rainfall_data(LAT, LON, "2024-09-01", "2024-09-15")

# ✅ Post-Flood (Sep 16–30)
post_rain = get_rainfall_data(LAT, LON, "2024-09-16", "2024-09-30")

print("Rainfall data ready ✅")

# ============================================================
#  PLOT FUNCTION (ONLY RESPECTIVE DATES)
# ============================================================

def plot_phase(df, title, color, filename):

    fig, ax = plt.subplots(figsize=(12, 4))

    # Plot only this phase data
    ax.bar(df["date"], df["rainfall"],
           color=color, edgecolor="white")

    # Threshold lines
    ax.axhline(RAIN_HIGH, color="#A32D2D", linestyle=":",
               lw=1.5, label=f"Heavy ({RAIN_HIGH} mm)")
    ax.axhline(RAIN_MED, color="#BA7517", linestyle=":",
               lw=1.5, label=f"Moderate ({RAIN_MED} mm)")

    # Labels
    ax.set_xlabel("Date")
    ax.set_ylabel("Rainfall (mm)")
    ax.set_title(title, fontsize=12, fontweight="bold")

    ax.grid(axis="y", alpha=0.3)
    ax.legend()

    # IMPORTANT: show only this date range
    ax.set_xlim(df["date"].min(), df["date"].max())

    plt.tight_layout()

    # Save image
    save_path = f"{OUTPUT_DIR}/{filename}"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"Saved → {filename}")

# ============================================================
#  GENERATE PHASE-WISE CHARTS
# ============================================================

# Pre-Flood
plot_phase(
    pre_rain,
    "Pre-Flood Rainfall (Aug 15–31, 2024)",
    "#B4B2A9",
    "rain_pre_flood.png"
)

# During Flood
plot_phase(
    during_rain,
    "During Flood Rainfall (Sep 1–15, 2024)",
    "#D85A30",
    "rain_during_flood.png"
)

# Post-Flood
plot_phase(
    post_rain,
    "Post-Flood Rainfall (Sep 16–30, 2024)",
    "#378ADD",
    "rain_post_flood.png"
)

print("\nAll phase-wise rainfall charts generated successfully 🚀")

In [ ]:
# ── STEP 9: Risk heatmap ──────────────────────────────────
print("Generating risk heatmap...")

localities  = ["Budameru Floodplain",
               "Vijayawada City Core", "Nunna"]
periods_lab = ["Pre Flood", "During Flood", "Post Flood"]
risk_map    = {"Low Risk": 1, "Medium Risk": 2, "High Risk": 3}
color_map   = {1: "#639922", 2: "#BA7517", 3: "#A32D2D"}
label_map   = {1: "Low Risk", 2: "Medium Risk", 3: "High Risk"}

fig2, ax2 = plt.subplots(figsize=(12, 5))
fig2.suptitle(
    "Flood Risk Assessment — Vijayawada Sep 2024\n"
    "U-Net Detection + Rainfall Data",
    fontsize=13, fontweight="bold")

for row, locality in enumerate(localities):
    for col, period in enumerate(periods_lab):
        r = df_risk[
            (df_risk["Locality"] == locality) &
            (df_risk["Period"]   == period)
        ]["Risk"].values[0]

        risk_num = risk_map[r]
        color    = color_map[risk_num]

        rect = plt.Rectangle(
            [col, row], 1, 1,
            facecolor=color, edgecolor="white", lw=2)
        ax2.add_patch(rect)
        ax2.text(
            col + 0.5, row + 0.5, r,
            ha="center", va="center",
            fontsize=12, fontweight="bold", color="white")

ax2.set_xlim(0, 3)
ax2.set_ylim(0, 3)
ax2.set_xticks([0.5, 1.5, 2.5])
ax2.set_xticklabels(periods_lab, fontsize=11, fontweight="bold")
ax2.set_yticks([0.5, 1.5, 2.5])
ax2.set_yticklabels(localities, fontsize=11, fontweight="bold")
ax2.tick_params(length=0)

legend3 = [
    mpatches.Patch(color="#639922", label="Low Risk"),
    mpatches.Patch(color="#BA7517", label="Medium Risk"),
    mpatches.Patch(color="#A32D2D", label="High Risk"),
]
ax2.legend(
    handles=legend3, loc="upper right",
    bbox_to_anchor=(1.25, 1), fontsize=11)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_risk_heatmap.png",
            dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Saved → fig_risk_heatmap.png")

print("\nFlood prediction module complete!")
print("Figures saved:")
print("  fig_rainfall_chart.png")
print("  fig_risk_heatmap.png")
print("  flood_risk_assessment.csv")

In [ ]:
import folium
from IPython.display import IFrame

OUTPUT_DIR = "/content/drive/MyDrive/FloodMonitoring_Project/outputs"
map_path = f"{OUTPUT_DIR}/flood_risk_map.html"

map_viz = folium.Map(
    location=[16.52, 80.63],
    zoom_start=12,
    tiles="CartoDB positron"
)

risk_color = {
    "Low Risk": "green",
    "Medium Risk": "orange",
    "High Risk": "red"
}

locations_coords = {
    "Budameru Floodplain": [16.545, 80.61],
    "Vijayawada City Core": [16.52, 80.64],
    "Nunna": [16.54, 80.70]
}

for _, row in df_risk.iterrows():

    loc = row.get("Locality", "Unknown")
    risk = row.get("Risk", "Low Risk")
    period = row.get("Period", "Unknown")

    # ✅ SAFE access (no crash)
    rain = row.get("Max Rain (mm)", row.get("Rainfall", "N/A"))
    flood_area = row.get("Flood Area km²", 0)

    folium.CircleMarker(
        location=locations_coords.get(loc, [16.52, 80.63]),
        radius=10,
        color=risk_color.get(risk, "green"),
        fill=True,
        fill_color=risk_color.get(risk, "green"),
        fill_opacity=0.7,

        popup=folium.Popup(
            f"""
            <b>Location:</b> {loc}<br>
            <b>Period:</b> {period}<br>
            <b>Risk:</b> {risk}<br>
            <b>Rainfall:</b> {rain}<br>
            <b>Flood Area:</b> {flood_area}
            """,
            max_width=300
        )
    ).add_to(map_viz)

map_viz.save(map_path)

print(f"Map saved → {map_path}")

display(map_viz)

In [ ]:
# ============================================================
#  FULL PIPELINE: U-NET → MASK → MAP OVERLAY
# ============================================================

import numpy as np
import rasterio
import tensorflow as tf
from skimage.transform import resize
import folium
from PIL import Image
from IPython.display import display
import warnings
warnings.filterwarnings("ignore")

# -------------------- PATHS --------------------
DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"

TIF_PATHS = {
    "Pre Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Pre_Flood_S1.tif",
    "During Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_During_Flood_S1.tif",
    "Post Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Post_Flood_S1.tif",
}

LOCALITIES = {
    "Budameru Floodplain": [80.58, 16.52, 80.64, 16.57],
    "Vijayawada City Core": [80.60, 16.50, 80.68, 16.56],
    "Nunna": [80.67, 16.52, 80.72, 16.56],
}

# -------------------- LOAD MODEL --------------------
print("Loading model...")
model = tf.keras.models.load_model(
    f"{DRIVE_DIR}/model/best_unet_flood.keras",
    compile=False
)
print("Model loaded ✅")

# -------------------- FUNCTIONS --------------------

def normalize(vv, vh):
    vv = np.clip((vv + 30) / 30, 0, 1)
    vh = np.clip((vh + 35) / 30, 0, 1)
    return vv, vh

def crop(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    with rasterio.open(tif_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)

        r1, c1 = src.index(min_lon, max_lat)
        r2, c2 = src.index(max_lon, min_lat)

        r1, r2 = sorted([r1, r2])
        c1, c2 = sorted([c1, c2])

        return vv[r1:r2, c1:c2], vh[r1:r2, c1:c2]

def predict_mask(vv, vh):
    vv, vh = normalize(vv, vh)
    H, W = vv.shape

    vv_r = resize(vv, (512, 512))
    vh_r = resize(vh, (512, 512))

    img = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]
    pred = model.predict(img, verbose=0)[0, ..., 0]

    mask = (pred > 0.5).astype(np.uint8)
    return resize(mask, (H, W), order=0).astype(np.uint8)

# ============================================================
#  STEP 1: GENERATE MASK (SELECT ONE)
# ============================================================

SELECTED_LOCATION = "Budameru Floodplain"
SELECTED_PERIOD   = "During Flood"

print(f"\nGenerating mask for {SELECTED_LOCATION} ({SELECTED_PERIOD})...")

bbox = LOCALITIES[SELECTED_LOCATION]
tif  = TIF_PATHS[SELECTED_PERIOD]

vv, vh = crop(tif, bbox)
mask = predict_mask(vv, vh)

print("Mask generated ✅")

# ============================================================
#  STEP 2: CONVERT MASK → IMAGE
# ============================================================

colored = np.zeros((mask.shape[0], mask.shape[1], 4))  # RGBA

# Water = Blue
colored[mask == 1] = [0, 0, 255, 180]

# Land = Transparent
colored[mask == 0] = [0, 0, 0, 0]

img_path = f"{OUTPUT_DIR}/temp_mask.png"
Image.fromarray(colored.astype(np.uint8)).save(img_path)

# ============================================================
#  STEP 3: CREATE MAP
# ============================================================

min_lon, min_lat, max_lon, max_lat = bbox

map_viz = folium.Map(
    location=[(min_lat + max_lat)/2, (min_lon + max_lon)/2],
    zoom_start=13,
    tiles="CartoDB positron"
)

# ============================================================
#  STEP 4: OVERLAY MASK
# ============================================================

folium.raster_layers.ImageOverlay(
    image=img_path,
    bounds=[[min_lat, min_lon], [max_lat, max_lon]],
    opacity=0.6
).add_to(map_viz)

# Add label marker
folium.Marker(
    location=[(min_lat + max_lat)/2, (min_lon + max_lon)/2],
    popup=f"{SELECTED_LOCATION} ({SELECTED_PERIOD})"
).add_to(map_viz)

# ============================================================
#  STEP 5: SAVE & DISPLAY
# ============================================================

map_path = f"{OUTPUT_DIR}/flood_mask_map.html"
map_viz.save(map_path)

print(f"Saved → {map_path}")

display(map_viz)

In [ ]:
# ============================================================
#  U-NET FLOOD DETECTION (WITHOUT PERMANENT WATER)
# ============================================================

import numpy as np
import rasterio
import tensorflow as tf
from skimage.transform import resize
import json
import os
import warnings
warnings.filterwarnings("ignore")

# -------------------- PATHS --------------------
DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TIF_PATHS = {
    "Pre_Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Pre_Flood_S1.tif",
    "During_Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_During_Flood_S1.tif",
    "Post_Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Post_Flood_S1.tif",
}

PERMANENT_WATER_PATH = f"{DRIVE_DIR}/vijayawada_sar/permanent_water.tif"

LOCALITIES = {
    "Budameru Floodplain": [80.58, 16.52, 80.64, 16.57],
    "Vijayawada City Core": [80.60, 16.50, 80.68, 16.56],
    "Nunna": [80.67, 16.52, 80.72, 16.56],
}

# -------------------- LOAD MODEL --------------------
print("Loading model...")
model = tf.keras.models.load_model(
    f"{DRIVE_DIR}/model/best_unet_flood.keras",
    compile=False
)
print("Model loaded ✅")

# -------------------- FUNCTIONS --------------------

def normalize(vv, vh):
    vv = np.clip((vv + 30) / 30, 0, 1)
    vh = np.clip((vh + 35) / 30, 0, 1)
    return vv, vh

def crop(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    with rasterio.open(tif_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)

        r1, c1 = src.index(min_lon, max_lat)
        r2, c2 = src.index(max_lon, min_lat)

        r1, r2 = sorted([r1, r2])
        c1, c2 = sorted([c1, c2])

        return vv[r1:r2, c1:c2], vh[r1:r2, c1:c2]

# ⭐ NEW: crop permanent water
def crop_permanent_water(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    with rasterio.open(tif_path) as src:
        water = src.read(1).astype(np.uint8)

        r1, c1 = src.index(min_lon, max_lat)
        r2, c2 = src.index(max_lon, min_lat)

        r1, r2 = sorted([r1, r2])
        c1, c2 = sorted([c1, c2])

        return water[r1:r2, c1:c2]

def predict_mask(vv, vh):
    vv, vh = normalize(vv, vh)
    H, W = vv.shape

    vv_r = resize(vv, (512, 512))
    vh_r = resize(vh, (512, 512))

    img = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]
    pred = model.predict(img, verbose=0)[0, ..., 0]

    mask = (pred > 0.5).astype(np.uint8)
    return resize(mask, (H, W), order=0).astype(np.uint8)

def to_km2(pixels):
    return (pixels * 100) / 1_000_000

# ============================================================
#  MAIN LOOP (WITH PERMANENT WATER REMOVAL)
# ============================================================

all_results = {}

for period, tif in TIF_PATHS.items():
    print(f"\n===== {period} =====")
    all_results[period] = {}

    for loc, bbox in LOCALITIES.items():

        # --- U-Net prediction ---
        vv, vh = crop(tif, bbox)
        mask = predict_mask(vv, vh)

        # --- Permanent water ---
        perm_water = crop_permanent_water(PERMANENT_WATER_PATH, bbox)

        # Resize permanent water if needed
        perm_water = resize(
            perm_water,
            mask.shape,
            order=0,
            preserve_range=True
        ).astype(np.uint8)

        # ⭐ REMOVE permanent water
        flood_only = mask.copy()
        flood_only[perm_water == 1] = 0

        # --- Compute ---
        pixels = int(np.sum(flood_only == 1))
        area   = float(to_km2(pixels))

        print(f"{loc}: Pixels={pixels}, Area={area:.4f} km²")

        all_results[period][loc] = {
            "pixels": pixels,
            "flood_area": area
        }

# ============================================================
#  SAVE RESULTS
# ============================================================

save_path = f"{OUTPUT_DIR}/flood_results_clean.json"

with open(save_path, "w") as f:
    json.dump(all_results, f, indent=2)

print("\nSaved clean results →", save_path)

In [ ]:
# ============================================================
#  FINAL FLOOD RISK (WITH CORRECT THRESHOLDS + DISPLAY)
# ============================================================

import pandas as pd

# -------------------- UPDATED THRESHOLDS --------------------
RAIN_LOW    = 10
RAIN_MED    = 35
RAIN_HIGH   = 65

# 🔥 UPDATED for your cropped areas
FLOOD_LOW   = 0.1
FLOOD_MED   = 0.4
FLOOD_HIGH  = 0.8

# -------------------- RISK FUNCTION --------------------
def classify_risk(rainfall_mm, flood_area_km2):

    # Rain classification
    if rainfall_mm >= RAIN_HIGH:
        rain_level = "High"
    elif rainfall_mm >= RAIN_MED:
        rain_level = "Moderate"
    else:
        rain_level = "Low"

    # Flood classification
    if flood_area_km2 >= FLOOD_HIGH:
        flood_level = "High"
    elif flood_area_km2 >= FLOOD_MED:
        flood_level = "Moderate"
    else:
        flood_level = "Low"

    # Risk rules
    if rain_level == "High" and flood_level == "High":
        risk = "High Risk"
    elif rain_level == "High" and flood_level == "Moderate":
        risk = "High Risk"
    elif rain_level == "Moderate" and flood_level == "High":
        risk = "High Risk"
    elif rain_level == "Moderate" or flood_level == "Moderate":
        risk = "Medium Risk"
    else:
        risk = "Low Risk"

    return risk, rain_level, flood_level


# -------------------- YOUR FLOOD DATA --------------------
FLOOD_AREAS = {
    "Budameru Floodplain": {
        "Pre_Flood": 0.0895,
        "During_Flood": 0.8753,
        "Post_Flood": 0.1280,
    },
    "Vijayawada City Core": {
        "Pre_Flood": 0.1949,
        "During_Flood": 0.6752,
        "Post_Flood": 0.2896,
    },
    "Nunna": {
        "Pre_Flood": 0.0301,
        "During_Flood": 0.3191,
        "Post_Flood": 0.0640,
    },
}

# -------------------- RAIN DATA --------------------
PERIOD_MAX_RAIN = {
    "Pre_Flood": 175.2,
    "During_Flood": 81.9,
    "Post_Flood": 29.6,
}

# -------------------- CALCULATION --------------------
results = []

for loc, periods in FLOOD_AREAS.items():
    for period, area in periods.items():

        rain = PERIOD_MAX_RAIN[period]

        risk, rain_lvl, flood_lvl = classify_risk(rain, area)

        results.append({
            "Locality": loc,
            "Period": period.replace("_", " "),
            "Rain (mm)": rain,
            "Flood Area (km²)": round(area, 3),
            "Rain Level": rain_lvl,
            "Flood Level": flood_lvl,
            "Risk": risk
        })

df = pd.DataFrame(results)

# -------------------- DISPLAY --------------------
print("\n" + "="*85)
print("  🌊 FLOOD RISK ASSESSMENT — FINAL OUTPUT")
print("="*85)

print(df.to_string(index=False))

print("="*85)


# -------------------- OPTIONAL: % INCREASE --------------------
print("\n📈 FLOOD INCREASE (Pre → During)\n")

for loc in FLOOD_AREAS:
    pre = FLOOD_AREAS[loc]["Pre_Flood"]
    dur = FLOOD_AREAS[loc]["During_Flood"]

    change = ((dur - pre) / pre) * 100 if pre > 0 else 0

    print(f"{loc}: {change:.1f}% increase 🔥")

In [ ]:
# ============================================================
#  FINAL MAP VISUALIZATION (ALL LOCATIONS + ALL PERIODS)
# ============================================================

import numpy as np
import rasterio
import tensorflow as tf
from skimage.transform import resize
import folium
from PIL import Image
import os
import warnings
warnings.filterwarnings("ignore")

# -------------------- PATHS --------------------
DRIVE_DIR = "/content/drive/MyDrive/FloodMonitoring_Project"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TIF_PATHS = {
    "Pre Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Pre_Flood_S1.tif",
    "During Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_During_Flood_S1.tif",
    "Post Flood": f"{DRIVE_DIR}/vijayawada_sar/vijayawada_Post_Flood_S1.tif",
}

PERMANENT_WATER_PATH = f"{DRIVE_DIR}/vijayawada_sar/permanent_water.tif"

LOCALITIES = {
    "Budameru Floodplain": [80.58, 16.52, 80.64, 16.57],
    "Vijayawada City Core": [80.60, 16.50, 80.68, 16.56],
    "Nunna": [80.67, 16.52, 80.72, 16.56],
}

# -------------------- LOAD MODEL --------------------
print("Loading model...")
model = tf.keras.models.load_model(
    f"{DRIVE_DIR}/model/best_unet_flood.keras",
    compile=False
)
print("Model loaded ✅")

# -------------------- FUNCTIONS --------------------

def normalize(vv, vh):
    vv = np.clip((vv + 30) / 30, 0, 1)
    vh = np.clip((vh + 35) / 30, 0, 1)
    return vv, vh

def crop(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    with rasterio.open(tif_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)

        r1, c1 = src.index(min_lon, max_lat)
        r2, c2 = src.index(max_lon, min_lat)

        r1, r2 = sorted([r1, r2])
        c1, c2 = sorted([c1, c2])

        return vv[r1:r2, c1:c2], vh[r1:r2, c1:c2]

def crop_perm(tif_path, bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    with rasterio.open(tif_path) as src:
        water = src.read(1).astype(np.uint8)

        r1, c1 = src.index(min_lon, max_lat)
        r2, c2 = src.index(max_lon, min_lat)

        r1, r2 = sorted([r1, r2])
        c1, c2 = sorted([c1, c2])

        return water[r1:r2, c1:c2]

def predict_mask(vv, vh):
    vv, vh = normalize(vv, vh)
    H, W = vv.shape

    vv_r = resize(vv, (512, 512))
    vh_r = resize(vh, (512, 512))

    img = np.stack([vv_r, vh_r], axis=-1)[np.newaxis]
    pred = model.predict(img, verbose=0)[0, ..., 0]

    mask = (pred > 0.5).astype(np.uint8)
    return resize(mask, (H, W), order=0).astype(np.uint8)

# ============================================================
#  CREATE BASE MAP (NO 403 ERROR)
# ============================================================

base_map = folium.Map(
    location=[16.52, 80.63],
    zoom_start=12,
    tiles="CartoDB positron"
)

# ============================================================
#  LOOP THROUGH ALL (LOCATIONS × PERIODS)
# ============================================================

for period, tif in TIF_PATHS.items():

    for loc, bbox in LOCALITIES.items():

        print(f"Processing: {loc} | {period}")

        # ---- Get data ----
        vv, vh = crop(tif, bbox)
        mask = predict_mask(vv, vh)

        perm = crop_perm(PERMANENT_WATER_PATH, bbox)
        perm = resize(perm, mask.shape, order=0, preserve_range=True).astype(np.uint8)

        # ---- Remove permanent water ----
        flood_only = mask.copy()
        flood_only[perm == 1] = 0

        # ---- Convert to RGBA ----
        colored = np.zeros((mask.shape[0], mask.shape[1], 4))

        # Different colors per period
        if period == "Pre Flood":
            colored[flood_only == 1] = [100, 149, 237, 150]   # light blue
        elif period == "During Flood":
            colored[flood_only == 1] = [255, 0, 0, 180]       # red
        else:
            colored[flood_only == 1] = [0, 0, 255, 150]       # blue

        # ---- Save temp image ----
        img_path = f"{OUTPUT_DIR}/{loc}_{period}.png"
        Image.fromarray(colored.astype(np.uint8)).save(img_path)

        # ---- Overlay on map ----
        min_lon, min_lat, max_lon, max_lat = bbox

        folium.raster_layers.ImageOverlay(
            image=img_path,
            bounds=[[min_lat, min_lon], [max_lat, max_lon]],
            opacity=0.6,
            name=f"{loc} - {period}"
        ).add_to(base_map)

# ============================================================
#  ADD LAYER CONTROL
# ============================================================

folium.LayerControl().add_to(base_map)

# ============================================================
#  SAVE & DISPLAY
# ============================================================

map_path = f"{OUTPUT_DIR}/final_flood_map.html"
base_map.save(map_path)

print("\nSaved →", map_path)

base_map